# 原始研究 Notebook（封存版）

這是 2026 年春季在 AI QC Lab 進行研究時的原始探索性 notebook，對應 `docs/slides/` 的期末簡報。

**此檔僅供追溯研究脈絡，已不再維護**；可重現的正式版本請見 `src/qaportfolio/`、`scripts/` 與 `notebooks/`。

封存時所做的處理：清除所有輸出、移除空白 cell 與數個事後補寫的說明 cell、資料路徑改為相對路徑。程式邏輯未修改。

已知問題（已在正式版本修正，詳見 `docs/original-results.md`）：Section 4 標題寫 D-Wave Quantum Annealing，實際使用的是 `neal` 古典模擬退火；三種方法的「MSRP」目標函數不一致；只回測單一季度；LSTM 未固定亂數種子；Section 5（Qiskit）未完成。

# 0. Packages & Paths

In [ ]:
import os
import pandas as pd
import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from datetime import datetime, timedelta

import joblib
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error

import matplotlib.pyplot as plt
from pypfopt import risk_models
import seaborn as sns
from pypfopt import EfficientFrontier
import shutil
import time

In [ ]:
data_folder = os.path.join('..', 'data')
other_data_folder = os.path.join(data_folder, 'Other Data')
all_adjusted_daily_stock_prices_folder = os.path.join(data_folder, 'Adjusted Daily Stock Data')
trading_date_path = os.path.join(other_data_folder, 'trading_dates.feather')
holding_data_0050_path = os.path.join(other_data_folder, 'Holding_0050.xlsx')

In [ ]:
trading_date_df = pd.read_feather(trading_date_path)
trading_date_df

In [ ]:
# 2. 主程式：計算原始公告日
def get_raw_announcement_date(year, month):
    """計算指定年月的第一個星期五 (原始公告日)"""
    first_day = datetime(year, month, 1)
    days_to_first_friday = (4 - first_day.weekday()) % 7
    return first_day + timedelta(days=days_to_first_friday)


# 指定目標年月
target_months = [
    (2025, 3), (2025, 6), (2025, 9), (2025, 12),
    (2026, 3), (2026,6)
]

data = []
# 將交易日轉成 set 集合，提升 lookup 查詢效能
valid_dates_set = set(trading_date_df['date'])

for year, month in target_months:
    quarter_str = f"{year}年 {month:>2}月"
    current_date = get_raw_announcement_date(year, month)
    
    # 3. 檢查機制：如果不在交易日裡，就順延到最近的下一個交易日
    while current_date.strftime('%Y-%m-%d') not in valid_dates_set:
        current_date += timedelta(days=1)
        
    data.append([quarter_str, current_date])

# 4. 建立 strategy_date_df
strategy_date_df = pd.DataFrame(data, columns=['Quarter', 'Announcement_Date'])

# 將 Announcement_Date 轉換為 datetime64 型態
strategy_date_df['Announcement_Date'] = pd.to_datetime(strategy_date_df['Announcement_Date'])

In [ ]:
# 將 trading_date_df 的 date 轉換為 datetime64 格式並排序，確保前後交易日搜尋正確
trading_dates_series = pd.to_datetime(trading_date_df['date']).sort_values().reset_index(drop=True)

start_dates = []
end_dates = []

for i, row in strategy_date_df.iterrows():
    current_announce = row['Announcement_Date']
    
    # 1. 計算 Strategy_Start_Date (下一個交易日)
    # 找出所有大於公告日的交易日，取第一個
    next_trading_days = trading_dates_series[trading_dates_series > current_announce]
    if not next_trading_days.empty:
        start_dates.append(next_trading_days.iloc[0])
    else:
        start_dates.append(pd.NaT)
        
    # 2. 計算 Strategy_End_Date (下一個 quarter 公告日 的 上一個交易日)
    # 如果還有下一個季度
    if i < len(strategy_date_df) - 1:
        next_quarter_announce = strategy_date_df.loc[i + 1, 'Announcement_Date']
        # 找出所有小於下個季度公告日的交易日，取最後一個
        prev_trading_days = trading_dates_series[trading_dates_series < next_quarter_announce]
        if not prev_trading_days.empty:
            end_dates.append(prev_trading_days.iloc[-1])
        else:
            end_dates.append(pd.NaT)
    else:
        # 最後一筆資料沒有下一個季度，故填入空值 (NaT)
        end_dates.append(pd.NaT)

# 將計算結果新增回 strategy_date_df
strategy_date_df['Strategy_Start_Date'] = start_dates
strategy_date_df['Strategy_End_Date'] = end_dates

# 刪除最後一個 row 並重新整理索引
strategy_date_df = strategy_date_df.iloc[:-1].reset_index(drop=True)

# --- 驗證更新後的結果 ---
print("\n=== 更新後的 strategy_date_df ===")
print(strategy_date_df)
print("\n--- 新增欄位資料型態檢查 ---")
print(strategy_date_df[['Strategy_Start_Date', 'Strategy_End_Date']].dtypes)

In [ ]:
# 確保 trading_date_df 的 date 欄位是 datetime 型態
trading_date_df['date'] = pd.to_datetime(trading_date_df['date'])

# 1. 根據「年-月」進行群組 (Groupby)，並計算每個月有幾天交易日
monthly_trading_days = trading_date_df.groupby(trading_date_df['date'].dt.to_period('M')).size()

# 2. 計算平均每個月的交易日天數
average_days = monthly_trading_days.mean()

# --- 輸出結果 ---
print("=== 各月份交易日天數統計 ===")
print(monthly_trading_days)
print("\n" + "="*30)
print(f"平均每個月有 {average_days:.2f} 個交易日")
print("="*30)

In [ ]:
# =====================================================================
# 新增功能：計算 ER_Test_Start_Date
# =====================================================================

# 確保 trading_dates_series 是時間型態、排序正確且無重複值
trading_dates_series = pd.to_datetime(trading_date_df['date']).sort_values().drop_duplicates().reset_index(drop=True)

er_start_dates = []
cov_start_dates = []

for i, row in strategy_date_df.iterrows():
    end_date = row['Announcement_Date']
    
    # 如果 End_Date 是 NaT (最後一筆資料)，則 Start_Date 也設為 NaT
    if pd.isna(end_date):
        er_start_dates.append(pd.NaT)
        cov_start_dates.append(pd.NaT)
        continue
        
    # 找出 Announcement_Date 在交易日行事曆中的位置 (Index)
    try:
        end_idx = trading_dates_series[trading_dates_series == end_date].index[0]
        
        # 1. 計算 ER_Test_Start_Date (上 63 個交易日)
        er_idx = end_idx - 62
        if er_idx >= 0:
            er_start_dates.append(trading_dates_series.iloc[er_idx])
        else:
            er_start_dates.append(pd.NaT) # 若歷史交易日不夠長，填入 NaT
            
        # 2. 計算 Cov_Matrix_Start_Date (上 250 個交易日)
        cov_idx = end_idx - 250
        if cov_idx >= 0:
            cov_start_dates.append(trading_dates_series.iloc[cov_idx])
        else:
            cov_start_dates.append(pd.NaT) # 若歷史交易日不夠長，填入 NaT
            
    except IndexError:
        # 萬一 End_Date 不在交易日行事曆中，防呆機制
        er_start_dates.append(pd.NaT)
        cov_start_dates.append(pd.NaT)

# 將結果新增至 strategy_date_df
strategy_date_df['ER_Test_Start_Date'] = er_start_dates

# 將新欄位轉換為 datetime64 型態
strategy_date_df['ER_Test_Start_Date'] = pd.to_datetime(strategy_date_df['ER_Test_Start_Date'])

strategy_date_df

In [ ]:
# =====================================================================
# 1. 指定特定的 Quarter
# =====================================================================
target_quarter = '2025_03'  # 你可以在這裡自由切換，例如 '2025_06', '2025_09' 等

# 原始基礎路徑
fmqc_folder = os.path.join(data_folder, 'Financial Modeling Using Quantum Computing')
ch6_portfolio_optimization_folder = os.path.join(fmqc_folder, 'ch6 Portfolio Optimization')

# =====================================================================
# 2. 在 ch6 底下創建該 quarter 的資料夾路徑
# =====================================================================
quarter_folder = os.path.join(ch6_portfolio_optimization_folder, target_quarter)


# =====================================================================
# 3. 將所有子資料夾與檔案路徑，全部改指到 quarter_folder 底下
# =====================================================================
adjusted_daily_stock_data_folder = os.path.join(quarter_folder, 'Adjusted Stock Prices')
adj_price_data_0050_path = os.path.join(adjusted_daily_stock_data_folder, '0050.feather')

expected_return_prediction_folder = os.path.join(quarter_folder, 'Expected Return Prediction')
prediction_results_folder = os.path.join(expected_return_prediction_folder, 'Prediction Results')
recovered_results_folder = os.path.join(expected_return_prediction_folder, 'Recovered Results')
training_process_folder = os.path.join(expected_return_prediction_folder, 'Training Process')
hparameter_tuning_folder = os.path.join(expected_return_prediction_folder, 'Hyperparameter Tuning')
analysis_folder = os.path.join(expected_return_prediction_folder, 'Analysis')
std_scalars_folder = os.path.join(expected_return_prediction_folder, 'Standard Scalers')
plots_folder = os.path.join(prediction_results_folder, 'Stock_Time_Series_Plots')
expected_return_path = os.path.join(expected_return_prediction_folder, 'expected_return_predictions.csv')


cov_matrix_folder = os.path.join(quarter_folder, 'Covariance Matrices')
portfolio_weights_folder = os.path.join(quarter_folder, 'Optimized Portfolio Weights')
unconstrained_and_answer_weights_path = os.path.join(portfolio_weights_folder, 'Unconstrained and Answer Portfolio Weights.csv')


# =====================================================================
# 4. 自動檢查並建立所有不存在的資料夾 (包含 quarter_folder 本身)
# =====================================================================
folders_to_create = [
    quarter_folder,  # 確保季度根資料夾優先建立
    adjusted_daily_stock_data_folder, # 如果調整後股價也是依季度分開，要把這行留著
    expected_return_prediction_folder,
    prediction_results_folder,
    recovered_results_folder,
    training_process_folder,
    hparameter_tuning_folder,
    analysis_folder,
    std_scalars_folder,
    plots_folder,
    cov_matrix_folder,
    portfolio_weights_folder
]

for folder in folders_to_create:
    os.makedirs(folder, exist_ok=True)

In [ ]:
for col in strategy_date_df:
    print(col)

In [ ]:
# =====================================================================
# 根據 target_quarter 提取對應日期並存入字典
# =====================================================================

# 2. 將 '2025_03' 轉換成 strategy_date_df 裡的字串格式 (用來做條件篩選)
year, month = target_quarter.split('_')
# 修正後：在引號前面加上 fr，代表同時支援 f-string 變數與 r 原生字串
quarter_filter_str = fr"{year}年\s*{int(month)}月"  # 使用正則表達式相容空格

# 3. 篩選出該季度對應的 row
quarter_row = strategy_date_df[strategy_date_df['Quarter'].str.contains(quarter_filter_str)]

if not quarter_row.empty:
    # 提取第一筆匹配的資料 (Series)
    row_data = quarter_row.iloc[0]
    
    # 4. 建立日期字典，並將 datetime64 轉換為字串格式 (eg. '2025-03-07')
    # 如果欄位值是 NaT，則保持 None 或 'NaT'
    date_dict = {
        'Announcement_Date': row_data['Announcement_Date'].strftime('%Y-%m-%d') if pd.notna(row_data['Announcement_Date']) else None,
        'Strategy_Start_Date': row_data['Strategy_Start_Date'].strftime('%Y-%m-%d') if pd.notna(row_data['Strategy_Start_Date']) else None,
        'Strategy_End_Date': row_data['Strategy_End_Date'].strftime('%Y-%m-%d') if pd.notna(row_data['Strategy_End_Date']) else None,
        'ER_Test_Start_Date': row_data['ER_Test_Start_Date'].strftime('%Y-%m-%d') if pd.notna(row_data['ER_Test_Start_Date']) else None
    }
    
    print(f"成功建立季度 [{target_quarter}] 的日期字典：")
    import pprint
    pprint.pprint(date_dict)

else:
    print(f"⚠️ 找不到與 {target_quarter} 相符的季度資料，請檢查 strategy_date_df。")
    date_dict = {}

# 1. Downloading Data & Preprocessing

In [ ]:
holding_data_0050_df = pd.read_excel(holding_data_0050_path)

Note: 
1. 0050 每年換股四次 3月, 6月, 9月, 12月
2. 公布名單日: 當月的第一個星期五
3. 實際換股日: 名單公布後的第三個星期五的下一個營業日（通常為星期一）正式以新成分股進行權重計算與調整
4. 對於25年3月: 公布名單日為: 2025/3/7, 換股日為: 2025/3/24



對於此次回測，假設我們在2025/3/07 公布名單後開始決定我們下一季(未來3個月至下一個換股日(6/6))的最佳投資組合

我們需要兩個輸入:
1. 每支股票預期回報率
2. 每支股票covariance matrix

如何計算: 
1. 每支股票於2025/3/07時對未來一季的預期回報率: LSTM預測每日的下一季回報率
2. 每支股票covariance matrix: 以從2025/3/07回推一年的每日回報率計算


最後組出在3/7能找到的最佳化權重後把它拿到2025/03/07~2025/06/06資料來回測

In [ ]:
holding_data_0050_df['年月'].unique()

In [ ]:
# 將 '2025_03' 轉換為 '2025/03' 以符合 DataFrame 的年月格式
formatted_quarter = target_quarter.replace('_', '/')
holding_data_0050_df[holding_data_0050_df['年月'] == formatted_quarter]

In [ ]:
# 1. 根據動態指定的季度篩選資料並取得唯一標的碼
unique_codes = holding_data_0050_df[holding_data_0050_df['年月'] == formatted_quarter]['標的碼'].unique()

# 2. 使用 list comprehension 搭配 .isdigit() 篩選出純數字的值
this_quarter_holdings_0050 = [code for code in unique_codes if code.isdigit()]

# --- 驗證結果 ---
print(f"[{target_quarter}] 篩選出的純數字股票代碼數量：", len(this_quarter_holdings_0050))
print("股票代碼範例：", this_quarter_holdings_0050[:5])

In [ ]:
print(len(this_quarter_holdings_0050), "unique stock codes in 0050 for this quarter:")
for code in this_quarter_holdings_0050:
    print(code)

In [ ]:
# 1. 先篩選出 2025/03 的資料
filtered_df = holding_data_0050_df[holding_data_0050_df["年月"] == formatted_quarter]

# 2. 再從中篩選出「標的碼」不是純數字的資料
#    (~ 代表 NOT，把 str.isdigit() 是 True 的過濾掉，只留下 False 的)
non_numeric_df = filtered_df[~filtered_df["標的碼"].str.isdigit()]
non_numeric_df

In [ ]:
# 1. 為了讓圖表更好看，先將投資比率由大到小排序
non_numeric_df = non_numeric_df.sort_values(by="投資比率％", ascending=False)

# 2. 繪製長條圖
fig = px.bar(
    non_numeric_df,
    x="標的名稱",
    y="投資比率％",
    title=f"Investment Ratio of Industries ({target_quarter})",
    labels={"標的名稱": "Security Name", "投資比率％": "Investment Ratio (%)"},
    text="投資比率％",  # 在長條圖上方直接顯示數值
)

# 3. 優化圖表樣式 (全英文標籤與排版)
fig.update_traces(
    texttemplate="%{text:.2f}%",  # 數值顯示到小數點後兩位並加上 %
    textposition="outside",  # 數值標籤放在柱狀圖外面
    marker_color="#ef553b",  # 換個顏色（例如暖橘紅色）與之前的 0050 做出區隔
)

fig.update_layout(
    title={
        "y": 0.95,
        "x": 0.5,
        "xanchor": "center",
        "yanchor": "top",
    },
    template="plotly_white",
    xaxis_tickangle=-45,  # 如果名稱太長，讓 X 軸文字傾斜 45 度避免重疊
    yaxis=dict(ticksuffix="%"),  # Y 軸刻度加上 % 後綴
)

# 顯示圖表
fig.show()

In [ ]:
numeric_df = filtered_df[filtered_df["標的碼"].str.isdigit()]
numeric_df

In [ ]:
# 1. 為了讓圖表更好看，先將投資比率由大到小排序
numeric_df = numeric_df.sort_values(by="投資比率％", ascending=False)

# 2. 繪製長條圖
fig = px.bar(
    numeric_df,
    x="標的名稱",
    y="投資比率％",
    title="Investment Ratio of Securities",
    labels={"標的名稱": "Security Name", "投資比率％": "Investment Ratio (%)"},
    text="投資比率％",  # 在長條圖上方直接顯示數值
)

# 3. 優化圖表樣式 (全英文標籤與排版)
fig.update_traces(
    texttemplate="%{text:.2f}%",  # 數值顯示到小數點後兩位並加上 %
    textposition="outside",  # 數值標籤放在柱狀圖外面
    marker_color="#ef553b",  # 換個顏色（例如暖橘紅色）與之前的 0050 做出區隔
)

fig.update_layout(
    title={
        "y": 0.95,
        "x": 0.5,
        "xanchor": "center",
        "yanchor": "top",
    },
    template="plotly_white",
    xaxis_tickangle=-45,  # 如果名稱太長，讓 X 軸文字傾斜 45 度避免重疊
    yaxis=dict(ticksuffix="%"),  # Y 軸刻度加上 % 後綴
)

# 顯示圖表
fig.show()

In [ ]:
# =====================================================================
# 1. 準備需要複製的標的名單
# =====================================================================
# 蒐集 numeric_df 中不重複的標的碼 (例如: ['2330', '2454', ...])
stock_codes = list(numeric_df['標的碼'].unique())

# 將 '0050' 也加入名單中，這樣可以一起用迴圈處理
if '0050' not in stock_codes:
    stock_codes.append('0050')

print(f"準備從來源資料夾複製共 {len(stock_codes)} 個檔案 (含 0050)...")


# =====================================================================
# 2. 執行檔案複製與覆蓋邏輯
# =====================================================================
missing_files = []
copied_count = 0

for code in stock_codes:
    filename = f"{code}.feather"
    
    # 定義來源與目的地的完整路徑
    src_path = os.path.join(all_adjusted_daily_stock_prices_folder, filename)
    dst_path = os.path.join(adjusted_daily_stock_data_folder, filename)
    
    # 檢查來源檔案是否存在
    if os.path.exists(src_path):
        # shutil.copy2 會直接覆蓋目的地已存在的同名檔案，並保留元數據 (如修改時間)
        shutil.copy2(src_path, dst_path)
        copied_count += 1
    else:
        # 如果找不到檔案，記錄下來
        missing_files.append(filename)

# =====================================================================
# 3. 顯示執行結果報告
# =====================================================================
print("-" * 50)
print(f"🎉 複製完成！共成功複製/覆蓋 {copied_count} 個檔案至目的地。")

if missing_files:
    print(f"\n⚠️ 警告：有 {len(missing_files)} 個檔案在來源資料夾中找不到：")
    for missing_file in missing_files:
        print(f"   - {missing_file}")

In [ ]:
# 2. 收集每檔股票的起始與結束日期
summary_data = []

# 遍歷資料夾中的所有 feather 檔案
for filename in os.listdir(adjusted_daily_stock_data_folder):
    if filename.endswith('.feather'):
        # 從檔名取得 stock_id
        stock_id = filename.replace('.feather', '')
        file_path = os.path.join(adjusted_daily_stock_data_folder, filename)
        
        # 效能優化：只讀取 'date' 欄位，節省記憶體與時間
        df = pd.read_feather(file_path, columns=['date'])
        
        # 確保 date 是 datetime 格式以正確比較大小
        df['date'] = pd.to_datetime(df['date'])
        
        # 取得最早與最晚日期
        start_date = df['date'].min()
        end_date = df['date'].max()
        
        summary_data.append({
            'Stock_ID': stock_id,
            'Start_Date': start_date,
            'End_Date': end_date
        })

# 轉換成 DataFrame 並根據起始日期排序，讓圖表更美觀
summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Start_Date', ascending=False)

# 3. 使用 Plotly 繪製 Timeline 圖表 (全英文標籤)
fig = px.timeline(
    summary_df, 
    x_start="Start_Date", 
    x_end="End_Date", 
    y="Stock_ID",
    title="Data Availability Range per Stock",
    labels={
        "Stock_ID": "Stock ID", 
        "Start_Date": "Start Date", 
        "End_Date": "End Date"
    },
    color="Stock_ID", # 加上顏色區分
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# 隱藏右側的圖例 (因為 y 軸已經有 Stock ID 了)
fig.update_layout(showlegend=False)

# 讓 y 軸的股票代碼按照資料順序排列，不會被 Plotly 自動按字母打亂
fig.update_yaxes(categoryorder="array", categoryarray=summary_df['Stock_ID'])

# 顯示圖表
fig.show()

In [ ]:
# 2. 讀取並合併所有 feather 檔案中的 date 與 close
all_data = []

for filename in os.listdir(adjusted_daily_stock_data_folder):
    if filename.endswith('.feather'):
        stock_id = filename.replace('.feather', '')
        file_path = os.path.join(adjusted_daily_stock_data_folder, filename)
        
        # 只讀取需要的欄位：date 和 close，節省記憶體
        df = pd.read_feather(file_path, columns=['date', 'close'])
        
        # 新增一欄 stock_id 用來區分不同股票
        df['stock_id'] = stock_id
        
        all_data.append(df)

# 將 50 個 DataFrame 垂直合併成一個大的 DataFrame
combined_df = pd.concat(all_data, ignore_index=True)

# 確保 date 是 datetime 格式，這樣 X 軸才會以時間序列正確排序
combined_df['date'] = pd.to_datetime(combined_df['date'])
# 為了繪圖效能與正確性，確保資料按照時間排序
combined_df = combined_df.sort_values(['stock_id', 'date'])

# 3. 使用 Plotly 繪製折線圖
# 注意：如果資料量非常大（例如 50檔 * 20年），我們加上 render_mode='webgl' 可以大幅提升瀏覽器的渲染流暢度
fig = px.line(
    combined_df,
    x='date',
    y='close',
    color='stock_id',
    title='Adjusted Close Price Over Time (50 Stocks)',
    labels={
        'date': 'Date',
        'close': 'Adjusted Close Price',
        'stock_id': 'Stock ID'
    },
    render_mode='webgl' 
)

# 讓圖表的背景乾淨一點，滑鼠游標顯示所有交錯的數值
fig.update_layout(
    hovermode='x unified',
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgrey'),
    yaxis=dict(showgrid=True, gridcolor='lightgrey')
)

fig.show()

In [ ]:
# 2. 讀取並合併資料
all_data = []

for filename in os.listdir(adjusted_daily_stock_data_folder):
    if filename.endswith('.feather'):
        stock_id = filename.replace('.feather', '')
        file_path = os.path.join(adjusted_daily_stock_data_folder, filename)
        
        df = pd.read_feather(file_path, columns=['date', 'close'])
        df['stock_id'] = stock_id
        all_data.append(df)

combined_df = pd.concat(all_data, ignore_index=True)
combined_df['date'] = pd.to_datetime(combined_df['date'])

# ---------------- 新增的裁切邏輯 ----------------

# 找出每檔股票的最早資料日期
start_dates = combined_df.groupby('stock_id')['date'].min()

# 「所有股票都有資料的那天」＝ 所有最早日期中，最晚的那一天
common_start_date = start_dates.max()

print(f"✅ 所有股票的共同起始日為: {common_start_date.strftime('%Y-%m-%d')}")

# 裁切資料：只保留大於等於共同起始日的資料
aligned_df = combined_df[combined_df['date'] >= common_start_date]

# 確保資料排序，以利繪圖效能
aligned_df = aligned_df.sort_values(['stock_id', 'date'])

# ----------------------------------------------

# 3. 繪製圖表 (使用裁切後的 aligned_df)
fig = px.line(
    aligned_df,
    x='date',
    y='close',
    color='stock_id',
    title=f"Adjusted Close Price Over Time (Aligned from {common_start_date.strftime('%Y-%m-%d')})",
    labels={
        'date': 'Date',
        'close': 'Adjusted Close Price',
        'stock_id': 'Stock ID'
    },
    render_mode='webgl' 
)

fig.update_layout(
    hovermode='x unified',
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgrey'),
    yaxis=dict(showgrid=True, gridcolor='lightgrey')
)

fig.show()

In [ ]:
date_dict.keys()

In [ ]:
date_dict['Announcement_Date']

In [ ]:
# 1. 初始化兩個儲存容器
all_data_dic_backtest = {}
all_data_dic_test = {}

# 定義回測切分日期的條件 (含該日期)
split_date = pd.to_datetime(date_dict['Announcement_Date'])

# 2. 讀取所有資料並依照需求切分與儲存
for filename in os.listdir(adjusted_daily_stock_data_folder):
    if filename.endswith('.feather'):
        stock_id = filename.replace('.feather', '')
        
        # 💡 判斷：如果是 '0050' 則直接跳過，不進行後續的讀取與拆分
        if stock_id == '0050':
            continue
            
        file_path = os.path.join(adjusted_daily_stock_data_folder, filename)
        
        # 讀取完整的 DataFrame
        df = pd.read_feather(file_path)
        
        # 確保 date 欄位是 datetime 格式
        df['date'] = pd.to_datetime(df['date'])
        
        # 依照日期排序，確保資料具有時間順序性
        df = df.sort_values('date').reset_index(drop=True)
        
        # 3. 篩選 backtest 資料：2025-03-07 前的所有資料 (含 03-07)
        mask_backtest = df['date'] <= split_date
        backtest_df = df[mask_backtest].copy()
        
        # 4. 指派 test 資料：為「所有資料」
        # 直接複製完整的 df，不進行任何日期的條件篩選
        test_df = df.copy()
        
        # 將處理好的 DataFrame 重置 index 並存入對應的字典中
        all_data_dic_backtest[stock_id] = backtest_df.reset_index(drop=True)
        all_data_dic_test[stock_id] = test_df.reset_index(drop=True)

print("✅ 所有 DataFrame 已成功讀取，並已排除 '0050'！")
print(f"📊 Backtest 股票總數: {len(all_data_dic_backtest)}")
print(f"📊 Test 股票總數: {len(all_data_dic_test)}")

In [ ]:
all_data_dic_backtest.keys()

In [ ]:
all_data_dic_backtest['2330']

In [ ]:
adj_price_0050_df = pd.read_feather(adj_price_data_0050_path)
adj_price_0050_df

In [ ]:
# 確保 date 欄位是 datetime 格式，這樣 Plotly 的時間軸才會排序正確
adj_price_0050_df["date"] = pd.to_datetime(adj_price_0050_df["date"])

# 繪製折線圖
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=adj_price_0050_df["date"],
        y=adj_price_0050_df["close"],
        mode="lines",
        name="0050 Close Price",
        line=dict(color="#1f77b4", width=2),
        hovertemplate="<b>Date</b>: %{x|%Y-%m-%d}<br><b>Price</b>: $%{y:.2f}<extra></extra>",
    )
)

# 設定圖表樣式 (全英文標籤)
fig.update_layout(
    title={
        "text": "0050 Daily Adjusted Closing Price Development",
        "y": 0.95,
        "x": 0.5,
        "xanchor": "center",
        "yanchor": "top",
    },
    xaxis_title="Date",
    yaxis_title="Adjusted Closing Price (TWD)",
    template="plotly_white",  # 使用乾淨的白色背景主題
    hovermode="x unified",  # 讓滑鼠移過去時更方便對齊時間點
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        rangeslider=dict(visible=True),  # 下方加入時間範圍滑桿，方便縮放
    ),
    yaxis=dict(showgrid=True, gridcolor="lightgray", tickprefix="$"),
)

# 顯示圖表
fig.show()

# 2. Unconstrained Portfolio Optimzation

## 2.1. Building Optimized Portfolio

### 2.1.1. Calculate Expected Return

#### 2.1.1.1. Data Preprocessing

In [ ]:
# 假設你的收盤價欄位名稱為 'close' (請根據你的資料集修改為 'Adj Close' 或其他名稱)
price_col = 'close'

for stock_id, df in all_data_dic_backtest.items():
    
    # 1. daily_return: 今日收盤價相較於昨日收盤價的報酬率 (過去特徵)
    df['daily_return'] = df[price_col].pct_change(1)
    
    # -----------------------------------------------------------------
    # 以下為 Forward Returns (未來報酬率)，通常作為預測模型的 Label (Y)
    # 公式：(未來某日的價格 - 今日價格) / 今日價格
    # 命名慣例：使用 'fwd' (forward) 來與過去的歷史報酬率做區隔
    # -----------------------------------------------------------------
    
    # 2. 今日至明天的 return (向未來偏移 1 個 row)
    df['return_fwd_1d'] = (df[price_col].shift(-1) - df[price_col]) / df[price_col]
    
    # 3. 今日至下 5 row 的 return (向未來偏移 5 個交易日，約 1 週)
    df['return_fwd_5d'] = (df[price_col].shift(-5) - df[price_col]) / df[price_col]
    
    # 4. 今日至下月的 return (實務慣例：21 個交易日)
    df['return_fwd_1m'] = (df[price_col].shift(-21) - df[price_col]) / df[price_col]
    
    # 5. 今日至下季的 return (實務慣例：63 個交易日)
    df['return_fwd_1q'] = (df[price_col].shift(-63) - df[price_col]) / df[price_col]
    
    # 6. 今日至下年的 return (實務慣例：252 個交易日)
    df['return_fwd_1y'] = (df[price_col].shift(-252) - df[price_col]) / df[price_col]

    # 將計算完的 df 存回字典
    all_data_dic_backtest[stock_id] = df

print("✅ 所有報酬率特徵與未來預測標籤 (Labels) 已計算完畢！")

In [ ]:
# 假設你的收盤價欄位名稱為 'close' (請根據你的資料集修改為 'Adj Close' 或其他名稱)
price_col = 'close'

for stock_id, df in all_data_dic_test.items():
    
    # 1. daily_return: 今日收盤價相較於昨日收盤價的報酬率 (過去特徵)
    df['daily_return'] = df[price_col].pct_change(1)
    
    # -----------------------------------------------------------------
    # 以下為 Forward Returns (未來報酬率)，通常作為預測模型的 Label (Y)
    # 公式：(未來某日的價格 - 今日價格) / 今日價格
    # 命名慣例：使用 'fwd' (forward) 來與過去的歷史報酬率做區隔
    # -----------------------------------------------------------------
    
    # 2. 今日至明天的 return (向未來偏移 1 個 row)
    df['return_fwd_1d'] = (df[price_col].shift(-1) - df[price_col]) / df[price_col]
    
    # 3. 今日至下 5 row 的 return (向未來偏移 5 個交易日，約 1 週)
    df['return_fwd_5d'] = (df[price_col].shift(-5) - df[price_col]) / df[price_col]
    
    # 4. 今日至下月的 return (實務慣例：21 個交易日)
    df['return_fwd_1m'] = (df[price_col].shift(-21) - df[price_col]) / df[price_col]
    
    # 5. 今日至下季的 return (實務慣例：63 個交易日)
    df['return_fwd_1q'] = (df[price_col].shift(-63) - df[price_col]) / df[price_col]
    
    # 6. 今日至下年的 return (實務慣例：252 個交易日)
    df['return_fwd_1y'] = (df[price_col].shift(-252) - df[price_col]) / df[price_col]

    # 將計算完的 df 存回字典
    all_data_dic_test[stock_id] = df

print("✅ 所有報酬率特徵與未來預測標籤 (Labels) 已計算完畢！")

In [ ]:
all_data_dic_backtest['2330']

In [ ]:
# 1. 將 all_data_dic_backtest 中的所有 DataFrame 合併成一個，方便進行跨股票的統計視覺化
all_dfs = []
for stock_id, df in all_data_dic_backtest.items():
    temp_df = df.copy()
    temp_df['ticker'] = stock_id  # 標記股票代號
    all_dfs.append(temp_df)

# 將所有股票資料垂直合併
combined_df = pd.concat(all_dfs, ignore_index=True)

# 2. 定義要畫的 6 個欄位與對應的英文標題
return_cols = [
    'daily_return', 'return_fwd_1d', 'return_fwd_5d',
    'return_fwd_1m', 'return_fwd_1q', 'return_fwd_1y'
]

titles = [
    'Historical 1-Day Return', 
    'Forward 1-Day Return', 
    'Forward 5-Day Return',
    'Forward 1-Month Return', 
    'Forward 1-Quarter Return', 
    'Forward 1-Year Return'
]

# 3. 建立 2x3 的子圖 (Subplots) 畫布
fig = make_subplots(
    rows=2, cols=3, 
    subplot_titles=titles,
    vertical_spacing=0.15,
    horizontal_spacing=0.08
)

# 4. 利用迴圈將 6 個欄位的直方圖 (Histogram) 填入子圖中
row_idx = 1
col_idx = 1

for i, col in enumerate(return_cols):
    # 移除 NaN 值 (因為 shift 會產生空值，Plotly 畫圖時需先排除)
    plot_data = combined_df[col].dropna()
    
    # 加入直方圖軌跡
    fig.add_trace(
        go.Histogram(
            x=plot_data, 
            nbinsx=100,  # 設定切分 100 個區塊，讓分佈更細緻
            name=col, 
            marker_color='rgb(55, 83, 109)',
            opacity=0.75
        ),
        row=row_idx, col=col_idx
    )
    
    # 更新 X 軸與 Y 軸的英文標籤
    fig.update_xaxes(title_text="Return", row=row_idx, col=col_idx)
    fig.update_yaxes(title_text="Frequency", row=row_idx, col=col_idx)
    
    # 計算下一個子圖的位置
    col_idx += 1
    if col_idx > 3:
        col_idx = 1
        row_idx += 1

# 5. 更新整體排版與外觀
fig.update_layout(
    title_text="Distribution of Returns Across the 50-Stock Universe",
    height=800,       # 設定畫布高度
    width=1400,       # 設定畫布寬度
    showlegend=False, # 隱藏圖例以節省空間
    template="plotly_white", # 使用乾淨的白底主題
    bargap=0.1        # 直方圖柱子之間的間距
)

# 顯示圖表
fig.show()

In [ ]:
date_dict.keys()

In [ ]:
date_dict['Strategy_Start_Date']

In [ ]:
# 1. 將 all_data_dic_test 中的特定日期區間資料合併成一個
all_dfs = []
start_date = date_dict['Strategy_Start_Date']
end_date = date_dict['Strategy_End_Date']

for stock_id, df in all_data_dic_test.items():
    temp_df = df.copy()
    
    # 確保 date 欄位是 datetime 格式以利篩選
    temp_df['date'] = pd.to_datetime(temp_df['date'])
    
    # 💡 核心篩選：2025-03-10 之後 (含) ~ 2025-06-06 之前 (含)
    mask = (temp_df['date'] >= start_date) & (temp_df['date'] <= end_date)
    temp_df = temp_df[mask]
    
    # 如果該股票在這個區間有資料，才標記並加入 list
    if not temp_df.empty:
        temp_df['ticker'] = stock_id  # 標記股票代號
        all_dfs.append(temp_df)

# 將所有篩選後的股票資料垂直合併
combined_df = pd.concat(all_dfs, ignore_index=True)

# 2. 定義要畫的 6 個欄位與對應的英文標題
return_cols = [
    'daily_return', 'return_fwd_1d', 'return_fwd_5d',
    'return_fwd_1m', 'return_fwd_1q', 'return_fwd_1y'
]

titles = [
    'Historical 1-Day Return', 
    'Forward 1-Day Return', 
    'Forward 5-Day Return',
    'Forward 1-Month Return', 
    'Forward 1-Quarter Return', 
    'Forward 1-Year Return'
]

# 3. 建立 2x3 的子圖 (Subplots) 畫布
fig = make_subplots(
    rows=2, cols=3, 
    subplot_titles=titles,
    vertical_spacing=0.15,
    horizontal_spacing=0.08
)

# 4. 利用迴圈將 6 個欄位的直方圖 (Histogram) 填入子圖中
row_idx = 1
col_idx = 1

for i, col in enumerate(return_cols):
    # 移除 NaN 值 (因為 shift 會產生空值，Plotly 畫圖時需先排除)
    plot_data = combined_df[col].dropna()
    
    # 加入直方圖軌跡
    fig.add_trace(
        go.Histogram(
            x=plot_data, 
            nbinsx=100,  # 設定切分 100 個區塊，讓分佈更細緻
            name=col, 
            marker_color='rgb(55, 83, 109)',
            opacity=0.75
        ),
        row=row_idx, col=col_idx
    )
    
    # 更新 X 軸與 Y 軸的英文標籤
    fig.update_xaxes(title_text="Return", row=row_idx, col=col_idx)
    fig.update_yaxes(title_text="Frequency", row=row_idx, col=col_idx)
    
    # 計算下一個子圖的位置
    col_idx += 1
    if col_idx > 3:
        col_idx = 1
        row_idx += 1

# 5. 更新整體排版與外觀
fig.update_layout(
    # 💡 修正點：直接帶入字串變數，拿掉 .strftime('%Y-%m-%d')
    title_text=f"Distribution of Test Returns ({start_date} to {end_date})",
    height=800,               # 設定畫布高度
    width=1400,               # 設定畫布寬度
    showlegend=False,         # 隱藏圖例以節省空間
    template="plotly_white",  # 使用乾淨的白底主題
    bargap=0.1                # 直方圖柱子之間的間距
)

# 顯示圖表
fig.show()

In [ ]:
strategy_date_df

In [ ]:
date_dict['ER_Test_Start_Date']

In [ ]:
import pandas as pd

# 用來存放每支股票「最後一個有值」的日期
fwd_1q_null_starts = {}

for stock_id, df in all_data_dic_backtest.items():
    # 確保資料有按照日期排序，以免 index 錯亂
    df = df.sort_values('date').reset_index(drop=True)
    
    # 取得 'return_fwd_1q' 欄位最後一個非空值 (有效值) 的 index
    last_valid_idx = df['return_fwd_1q'].last_valid_index()
    
    # 判斷是否整條都是空的 (如果資料總筆數小於或等於 63 筆，就會全空)
    if last_valid_idx is not None:
        # 取得最後一個有值的日期
        last_valid_date = df.loc[last_valid_idx, 'date']
        
        # 我們將日期轉成 YYYY-MM-DD 的字串格式方便閱讀
        fwd_1q_null_starts[stock_id] = last_valid_date.strftime('%Y-%m-%d')
    else:
        # 如果這檔股票資料不足 63 天，那這欄就會從頭到尾都是空的
        fwd_1q_null_starts[stock_id] = "全為空值 (資料不足 63 天)"

# 印出結果
print("📊 各股票在以下日期「之後」，return_fwd_1q 皆為空值：")
print("-" * 50)
for stock_id, date_str in fwd_1q_null_starts.items():
    print(f"股票代號 {stock_id: <10} | {date_str} 之後皆為空")

# 可以把結果轉換成 Series 來觀察
res_series = pd.Series(fwd_1q_null_starts)
print("\n總結統計：")
print(res_series.value_counts())


# =====================================================================
# 新增與修改功能：檢查日期一致性，並尋找「下一個交易日」更新至 date_dict
# =====================================================================
print("-" * 50)

# 1. 排除掉「資料不足 63 天」的無效日期
valid_dates = [d for d in fwd_1q_null_starts.values() if d != "全為空值 (資料不足 63 天)"]

if len(valid_dates) == 0:
    raise ValueError("❌ 報錯並暫停：所有股票的 return_fwd_1q 皆為空值，無法進行後續計算！")

# 2. 檢查所有股票最後一個有效日期是否完全相同
unique_dates = set(valid_dates)

if len(unique_dates) == 1:
    last_valid_date_str = list(unique_dates)[0]
    print(f"✅ 驗證成功！所有股票最後一個有值的日期皆相同為: {last_valid_date_str}")
    
    # 3. 尋找該日期的「下一個交易日」
    # 確保 trading_date_df 的 date 欄位是 datetime64 格式並排序
    trading_dates_series = pd.to_datetime(trading_date_df['date']).sort_values().drop_duplicates().reset_index(drop=True)
    target_date_dt = pd.to_datetime(last_valid_date_str)
    
    # 找出所有大於最後有效日期的交易日，並取第一個
    next_trading_days = trading_dates_series[trading_dates_series > target_date_dt]
    
    if not next_trading_days.empty:
        next_trading_date = next_trading_days.iloc[0].strftime('%Y-%m-%d')
        
        # 4. 成功更新 date_dict
        date_dict['ER_Test_Start_Date'] = next_trading_date
        print(f"🚀 已成功將 date_dict['ER_Test_Start_Date'] 更新為下一個交易日: '{next_trading_date}'")
    else:
        raise ValueError(f"❌ 報錯並暫停：在 trading_date_df 中找不到 '{last_valid_date_str}' 之後的下一個交易日！")
        
else:
    error_msg = (
        f"❌ 報錯並暫停：並非所有股票最後一個有值的日期都相同！\n"
        f"目前偵測到不同的日期有：{sorted(list(unique_dates))}\n"
        f"請檢查資料源或處理邏輯。"
    )
    raise ValueError(error_msg)

The data before 2024-11-27 are the one can be used to train ER Prdiction ML models

In [ ]:
def add_quarterly_prediction_features(df):
    """
    為單一股票的 DataFrame 計算適用於預測中長線 (下季) 回報率的技術指標。
    """
    # 確保資料按時間排序，避免 rolling 計算錯亂
    df = df.sort_values('date').reset_index(drop=True)
    
    # 1. 趨勢與乖離率 (Bias)
    df['MA_20'] = df['close'].rolling(window=20).mean()
    df['MA_60'] = df['close'].rolling(window=60).mean() # 季線
    df['Bias_20'] = (df['close'] - df['MA_20']) / df['MA_20']
    df['Bias_60'] = (df['close'] - df['MA_60']) / df['MA_60']
    
    # 2. 中長線動能 (Rate of Change, ROC)
    # 過去 20 天與 60 天的漲跌幅
    df['ROC_20'] = df['close'].pct_change(periods=20)
    df['ROC_60'] = df['close'].pct_change(periods=60)
    
    # 3. 波動度 (Historical Volatility)
    # 利用 daily_return 計算過去 60 天的標準差，並乘上 sqrt(252) 進行年化
    df['Vol_60'] = df['daily_return'].rolling(window=60).std() * np.sqrt(252)
    
    # 4. 量能指標 (Volume Ratio)
    # 當日成交量相對於過去 20 天平均成交量的倍數
    df['Vol_MA_20'] = df['Trading_Volume'].rolling(window=20).mean()
    # 加上 1e-8 避免分母為 0 的狀況
    df['Volume_Ratio_20'] = df['Trading_Volume'] / (df['Vol_MA_20'] + 1e-8) 
    
    # 5. 價格區間相對位置 (Price Position in 60 days)
    # 計算公式：(收盤價 - 區間最低) / (區間最高 - 區間最低)
    rolling_max_60 = df['max'].rolling(window=60).max()
    rolling_min_60 = df['min'].rolling(window=60).min()
    df['Price_Position_60'] = (df['close'] - rolling_min_60) / (rolling_max_60 - rolling_min_60 + 1e-8)
    
    # 移除運算過程中產生的過渡欄位 (如果不需要保留 MA 絕對數值的話)
    # df.drop(columns=['MA_20', 'MA_60', 'Vol_MA_20'], inplace=True)
    
    return df

def process_all_stocks(data_dic):
    """
    走訪整個 dictionary，對每檔股票應用特徵工程
    """
    processed_dic = {}
    for stock_id, df in data_dic.items():
        # 使用 copy() 避免改動到原始資料
        processed_dic[stock_id] = add_quarterly_prediction_features(df.copy())
    return processed_dic

# ==========================================
# 執行運算
# ==========================================

print("⏳ 正在計算 Backtest 資料集的技術特徵...")
all_data_dic_backtest = process_all_stocks(all_data_dic_backtest)

print("⏳ 正在計算 Test 資料集的技術特徵...")
all_data_dic_test = process_all_stocks(all_data_dic_test)

print("✅ 所有特徵計算完畢！")

In [ ]:
all_data_dic_backtest['2330']

In [ ]:
date_dict['ER_Test_Start_Date']

In [ ]:
# 定義截斷日期
cutoff_date = pd.to_datetime(date_dict['ER_Test_Start_Date'])

# 儲存每檔股票的 NaN 比例
nan_percentages = {}

print("⏳ 開始處理資料：截斷日期與 ffill 填充...")

for stock_id, df in all_data_dic_backtest.items():
    # 確保 date 格式並排序 (良好的習慣，避免 ffill 補錯方向)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    # 1. 丟掉 2024-11-28 以後 (含當日) 的資料 (只保留小於該日期的)
    df_filtered = df[df['date'] < cutoff_date].copy()
    
    # 2. 進行前向填充 (ffill)
    # 使用 ffill() 填補中間的缺失值 (例如停牌、資料遺失等)
    df_filtered = df_filtered.ffill()
    
    # 將處理完的 DataFrame 存回原字典
    all_data_dic_backtest[stock_id] = df_filtered
    
    # 3. 計算包含 NaN 的 row 的比例
    total_rows = len(df_filtered)
    if total_rows > 0:
        # isna().any(axis=1) 會找出「只要有任何一個欄位是 NaN」的 row
        rows_with_nan = df_filtered.isna().any(axis=1).sum()
        pct_nan = (rows_with_nan / total_rows) * 100
    else:
        # 防呆機制：如果該檔股票在截斷後沒有資料
        pct_nan = 0.0 
        
    nan_percentages[stock_id] = pct_nan

# 4. 印出每檔股票的結果
print("\n📊 各股票含有 NaN 的 Row 比例：")
print("-" * 40)
for stock_id, pct in nan_percentages.items():
    print(f"股票 {stock_id: <8} | 含有 NaN 的比例: {pct:>6.2f}%")

# 5. 計算並印出平均比例
if len(nan_percentages) > 0:
    avg_nan_pct = sum(nan_percentages.values()) / len(nan_percentages)
    print("-" * 40)
    print(f"🎯 總結：平均每個 DataFrame 有 **{avg_nan_pct:.2f}%** 的 row 在任意欄位包含 NaN。")
else:
    print("⚠️ 沒有可用的資料進行統計。")

In [ ]:
all_data_dic_backtest['2330']

In [ ]:
print("⏳ 開始刪除 all_data_dic_backtest 中每個 DataFrame 的 NaN 值...")

# 紀錄刪除前後的資料筆數，方便你確認刪除了多少
total_before = 0
total_after = 0

for stock_id, df in all_data_dic_backtest.items():
    rows_before = len(df)
    total_before += rows_before
    
    # 💡 核心操作：直接將含有任何 NaN 的 row 丟棄，並重置 index
    df_cleaned = df.dropna().reset_index(drop=True)
    
    rows_after = len(df_cleaned)
    total_after += rows_after
    
    # 更新回原本的字典中
    all_data_dic_backtest[stock_id] = df_cleaned
    
    # (選用) 如果你想看個別股票刪除了多少筆，可以取消註解下一行
    # print(f"股票 {stock_id: <6} | 原始筆數: {rows_before: >4} -> 刪除後筆數: {rows_after: >4} (刪除了 {rows_before - rows_after} 筆)")

print("\n✅ 所有 DataFrame 的 NaN 值已成功刪除！")
print(f"📊 總資料筆數變化：{total_before} 筆 -> {total_after} 筆 (共過濾掉 {total_before - total_after} 筆包含空值的資料)")

In [ ]:
# 定義時間區間
start_date = pd.to_datetime(date_dict['ER_Test_Start_Date'])
end_date = pd.to_datetime(date_dict['Strategy_End_Date'])

# 儲存每檔股票的 NaN 比例
test_nan_percentages = {}

print("⏳ 開始處理 Test 資料集：先 ffill 填補，再進行時間區間篩選...")

for stock_id, df in all_data_dic_test.items():
    # 確保 date 格式並排序
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    # 1. 先做前向填充 (ffill)
    df_filled = df.ffill()
    
    # 2. 只保留 2024-11-28 (含) 至 2025-06-05 (含) 的資料
    mask = (df_filled['date'] >= start_date) & (df_filled['date'] <= end_date)
    df_filtered = df_filled[mask].copy()
    
    # 將處理完的 DataFrame 存回原字典
    all_data_dic_test[stock_id] = df_filtered
    
    # 3. 計算包含 NaN 的 row 的比例
    total_rows = len(df_filtered)
    if total_rows > 0:
        # 找出只要有任何一個欄位是 NaN 的 row 數量
        rows_with_nan = df_filtered.isna().any(axis=1).sum()
        pct_nan = (rows_with_nan / total_rows) * 100
    else:
        pct_nan = 0.0
        
    test_nan_percentages[stock_id] = pct_nan

# 4. 印出每檔股票的結果
print("\n📊 Test 資料集各股票含有 NaN 的 Row 比例：")
print("-" * 40)
for stock_id, pct in test_nan_percentages.items():
    print(f"股票 {stock_id: <8} | 含有 NaN 的比例: {pct:>6.2f}%")

# 5. 計算並印出平均比例
if len(test_nan_percentages) > 0:
    avg_test_nan_pct = sum(test_nan_percentages.values()) / len(test_nan_percentages)
    print("-" * 40)
    print(f"🎯 總結：Test 每個 DataFrame 平均有 **{avg_test_nan_pct:.2f}%** 的 row 在任意欄位包含 NaN。")
else:
    print("⚠️ 沒有可用的資料進行統計。")

In [ ]:
# 1. 定義要刪除的欄位清單
cols_to_drop = ['return_fwd_1d', 'return_fwd_5d', 'return_fwd_1m', 'return_fwd_1y']

print("⏳ 開始從兩個字典中移除不需要的未來報酬欄位...")

# 2. 處理 all_data_dic_backtest
for stock_id, df in all_data_dic_backtest.items():
    # 使用 errors='ignore' 可以確保就算某些 df 本來就沒有該欄位，程式也不會報錯中斷
    all_data_dic_backtest[stock_id] = df.drop(columns=cols_to_drop, errors='ignore')

# 3. 處理 all_data_dic_test
for stock_id, df in all_data_dic_test.items():
    all_data_dic_test[stock_id] = df.drop(columns=cols_to_drop, errors='ignore')

print("✅ 欄位刪除完成！")

# 4. 驗證欄位是否成功刪除 (以其中一檔股票為例檢查)
sample_stock = list(all_data_dic_backtest.keys())[0]
print(f"\n📋 檢查股票 {sample_stock} 剩餘的欄位：")
print(list(all_data_dic_backtest[sample_stock].columns))

In [ ]:
# 1. 建立新的字典，並把每個 df 的最後 63 列放進去
all_data_dic_leakage_protection = {
    code: df.iloc[-63:] 
    for code, df in all_data_dic_backtest.items()
}

# 2. 將原本字典裡的每個 df 更新，只保留「除了最後 63 列」以外的前面所有資料
all_data_dic_backtest = {
    code: df.iloc[:-63] 
    for code, df in all_data_dic_backtest.items()
}

In [ ]:
all_data_dic_backtest['2330']

In [ ]:
all_data_dic_test['2330']

In [ ]:
# 1. 計算所有 df 的長度總和
total_length = sum(len(df) for df in all_data_dic_backtest.values())

# 2. 計算總共有幾個 df (股票數量)
num_of_dfs = len(all_data_dic_backtest)

# 3. 計算平均長度（加上條件判斷防止字典為空時除以零）
average_length = total_length / num_of_dfs if num_of_dfs > 0 else 0

# 顯示結果
print(f"所有 DataFrame 的總長度 (總列數)：{total_length}")
print(f"共有幾筆 DataFrame (股票數量)：{num_of_dfs}")
print(f"平均每個 DataFrame 的長度：{average_length:.2f}")

In [ ]:
# 1. 計算所有 df 的長度總和
total_length = sum(len(df) for df in all_data_dic_test.values())

# 2. 計算總共有幾個 df (股票數量)
num_of_dfs = len(all_data_dic_test)

# 3. 計算平均長度（加上條件判斷防止字典為空時除以零）
average_length = total_length / num_of_dfs if num_of_dfs > 0 else 0

# 顯示結果
print(f"所有 DataFrame 的總長度 (總列數)：{total_length}")
print(f"共有幾筆 DataFrame (股票數量)：{num_of_dfs}")
print(f"平均每個 DataFrame 的長度：{average_length:.2f}")

In [ ]:
# 印出 2330 這檔股票所有欄位的名稱與型態
print(all_data_dic_backtest['2330'].dtypes)

In [ ]:
# 印出 2330 這檔股票所有欄位的名稱與型態
print(all_data_dic_test['2330'].dtypes)

#### 2.1.1.2. Training LSTM

Note: I didn't do HP tuning to save time

In [ ]:
import pandas as pd

# 建立一個 list 來儲存每支股票的所有日期區間資訊
date_summary_extended = []

for stock_id, df_backtest in all_data_dic_backtest.items():
    # 確保回測資料依日期排序
    df_backtest_sorted = df_backtest.sort_values('date').reset_index(drop=True)
    total_len = len(df_backtest_sorted)
    
    # 計算切分點 (85% 處)
    split_idx = int(total_len * 0.85)
    
    # 1. 訓練集與驗證集範圍 (維持你設定的 63 天緩衝)
    df_train_part = df_backtest_sorted.iloc[:split_idx]
    df_val_part = df_backtest_sorted.iloc[split_idx + 63:]
    
    # 2. 尋找並處理測試集日期 (從 all_data_dic_test 字典中提取)
    test_start = 'N/A'
    test_end = 'N/A'
    test_days = 0
    
    if stock_id in all_data_dic_test:
        df_test = all_data_dic_test[stock_id]
        if len(df_test) > 0:
            df_test_sorted = df_test.sort_values('date').reset_index(drop=True)
            test_start = df_test_sorted['date'].iloc[0]
            test_end = df_test_sorted['date'].iloc[-1]
            test_days = len(df_test_sorted)
    else:
        print(f"⚠️ 警告：測試集字典中找不到股票 {stock_id} 的資料")

    # 3. 計算驗證集的結束日
    val_end = df_val_part['date'].iloc[-1] if len(df_val_part) > 0 else 'N/A'

    # 4. 【新功能】計算第二條隔離帶 Purge2 的起始與結束日期
    purge2_start = 'N/A'
    purge2_end = 'N/A'
    
    # 唯有當 test_start 與 val_end 都有明確日期時，才計算日曆天的上一天與下一天
    if test_start != 'N/A' and val_end != 'N/A':
        # 轉成 Timestamp 以便進行日期加減
        test_start_dt = pd.to_datetime(test_start)
        val_end_dt = pd.to_datetime(val_end)
        
        # Purge2_Start: Test_Start 的上一天
        purge2_start = (test_start_dt - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        # Purge2_End: Val_End 的下一天
        purge2_end = (val_end_dt + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

    # 紀錄該股票的所有日期區間
    info = {
        'Stock_ID': stock_id,
        'Backtest_Days': total_len,
        'Test_Days': test_days,
        # 訓練集區間
        'Train_Start': df_train_part['date'].iloc[0] if len(df_train_part) > 0 else 'N/A',
        'Train_End': df_train_part['date'].iloc[-1] if len(df_train_part) > 0 else 'N/A',
        # 【修改點 1】改名為 Purge1_Start 與 Purge1_End (被丟棄不用的 63 天)
        'Purge1_Start': df_backtest_sorted['date'].iloc[split_idx] if split_idx < total_len else 'N/A',
        'Purge1_End': df_backtest_sorted['date'].iloc[split_idx + 62] if (split_idx + 62) < total_len else 'N/A',
        # 驗證集區間
        'Val_Start': df_val_part['date'].iloc[0] if len(df_val_part) > 0 else 'N/A',
        'Val_End': val_end,
        # 【修改點 2】新增新版隔離帶 Purge2_Start 與 Purge2_End
        'Purge2_Start': purge2_start,
        'Purge2_End': purge2_end,
        # 測試集區間
        'Test_Start': test_start,
        'Test_End': test_end
    }
    date_summary_extended.append(info)

# 轉換為新版的 DataFrame
df_date_summary = pd.DataFrame(date_summary_extended)

# 顯示結果
df_date_summary

In [ ]:
import pandas as pd
import plotly.express as px

plot_data_extended = []

for _, row in df_date_summary.iterrows():
    # 確保基礎日期完整才進行計算，避免 N/A 造成錯誤
    if 'N/A' in [row['Train_Start'], row['Test_End']]:
        continue
        
    t_start = pd.to_datetime(row['Train_Start'])
    te_end = pd.to_datetime(row['Test_End'])
    
    # 計算該股票從頭到尾的總日曆天數，作為百分比的分母
    total_duration = (te_end - t_start).total_seconds()
    
    # 建立一個輔助函數來計算各區間佔比
    def calc_pct(start_dt, end_dt):
        if 'N/A' in [start_dt, end_dt] or pd.isna(start_dt) or pd.isna(end_dt):
            return 0
        duration = (pd.to_datetime(end_dt) - pd.to_datetime(start_dt)).total_seconds()
        return round((duration / total_duration) * 100, 1) if total_duration > 0 else 0

    # 1. 訓練集
    if row['Train_Start'] != 'N/A':
        plot_data_extended.append({
            'Stock_ID': str(row['Stock_ID']),
            'Start': row['Train_Start'], 'End': row['Train_End'],
            'Phase': '1. Training Set (訓練集)',
            'Percentage': f"{calc_pct(row['Train_Start'], row['Train_End'])}%"
        })
        
    # 2. 隔離區一 (訓練 vs 驗證)
    if row['Purge1_Start'] != 'N/A':
        plot_data_extended.append({
            'Stock_ID': str(row['Stock_ID']),
            'Start': row['Purge1_Start'], 'End': row['Purge1_End'],
            'Phase': '2. Purge Buffer 1 (隔離區一)',
            'Percentage': f"{calc_pct(row['Purge1_Start'], row['Purge1_End'])}%"
        })
        
    # 3. 驗證集
    if row['Val_Start'] != 'N/A':
        plot_data_extended.append({
            'Stock_ID': str(row['Stock_ID']),
            'Start': row['Val_Start'], 'End': row['Val_End'],
            'Phase': '3. Validation Set (驗證集)',
            'Percentage': f"{calc_pct(row['Val_Start'], row['Val_End'])}%"
        })
        
    # 4. 隔離區二 (驗證 vs 測試)
    if row['Purge2_Start'] != 'N/A':
        plot_data_extended.append({
            'Stock_ID': str(row['Stock_ID']),
            'Start': row['Purge2_Start'], 'End': row['Purge2_End'],
            'Phase': '4. Purge Buffer 2 (隔離區二)',
            'Percentage': f"{calc_pct(row['Purge2_Start'], row['Purge2_End'])}%"
        })
        
    # 5. 測試集
    if row['Test_Start'] != 'N/A':
        plot_data_extended.append({
            'Stock_ID': str(row['Stock_ID']),
            'Start': row['Test_Start'], 'End': row['Test_End'],
            'Phase': '5. Testing Set (測試集)',
            'Percentage': f"{calc_pct(row['Test_Start'], row['Test_End'])}%"
        })

# 建立 Plotly 繪圖用的 DataFrame
df_plot = pd.DataFrame(plot_data_extended)
df_plot['Start'] = pd.to_datetime(df_plot['Start'])
df_plot['End'] = pd.to_datetime(df_plot['End'])

# 設定專屬顏色地圖 (主要資料集用鮮豔色，隔離緩衝區用深淺不同的灰色)
color_map = {
    '1. Training Set (訓練集)': '#3182bd',    # 藍色
    '2. Purge Buffer 1 (隔離區一)': '#969696', # 深灰
    '3. Validation Set (驗證集)': '#e6550d',  # 橘色
    '4. Purge Buffer 2 (隔離區二)': '#cccccc', # 淺灰
    '5. Testing Set (測試集)': '#2ca02c'      # 綠色
}

# 繪製甘特圖
fig = px.timeline(
    df_plot, 
    x_start="Start", 
    x_end="End", 
    y="Stock_ID", 
    color="Phase",
    color_discrete_map=color_map,
    title="每支股票完整量化流水線時間軸分佈圖 (含雙重防走光隔離帶與時間佔比)",
    labels={'Stock_ID': '股票代號', 'Phase': '資料集階段'},
    custom_data=['Percentage'] # 將百分比傳遞給 hovertemplate
)

# 優化圖表佈局
fig.update_layout(
    xaxis_title="年份 / 日期",
    yaxis_title="股票代號",
    yaxis={'categoryorder': 'category ascending'}, # 股票代碼依數字由小到大排列
    height=1100, # 給予 50 支股票足夠的垂直空間
    barmode='stack',
    hovermode='x',
    template='plotly_white'
)

# 定義滑鼠懸停時顯示的中文提示框，包含計算好的 %
fig.update_traces(
    hovertemplate="<b>股票代號:</b> %{y}<br>" +
                  "<b>資料階段:</b> %{legendgroup}<br>" +
                  "<b>時間佔比:</b> %{customdata[0]}<br>" +
                  "<b>區間開始:</b> %{start|%Y-%m-%d}<br>" +
                  "<b>區間結束:</b> %{end|%Y-%m-%d}<br><extra></extra>"
)

# 顯示圖表
fig.show()

In [ ]:
import pandas as pd
import plotly.express as px

# 初始化 5 個階段的總天數計數器
total_days_stats = {
    '1. Training Set (訓練集)': 0,
    '2. Purge Buffer 1 (隔離區一)': 0,
    '3. Validation Set (驗證集)': 0,
    '4. Purge Buffer 2 (隔離區二)': 0,
    '5. Testing Set (測試集)': 0
}

# 遍歷所有股票，累加各階段的實際日曆天數
for _, row in df_date_summary.iterrows():
    # 確保基礎日期完整才進行計算
    if 'N/A' in [row['Train_Start'], row['Train_End'], row['Test_Start'], row['Test_End']]:
        continue
        
    # 將日期字串轉換為 Timestamp 物件以計算天數差
    t_start  = pd.to_datetime(row['Train_Start'])
    t_end    = pd.to_datetime(row['Train_End'])
    p1_start = pd.to_datetime(row['Purge1_Start'])
    p1_end   = pd.to_datetime(row['Purge1_End'])
    v_start  = pd.to_datetime(row['Val_Start'])
    v_end    = pd.to_datetime(row['Val_End'])
    p2_start = pd.to_datetime(row['Purge2_Start'])
    p2_end   = pd.to_datetime(row['Purge2_End'])
    te_start = pd.to_datetime(row['Test_Start'])
    te_end   = pd.to_datetime(row['Test_End'])
    
    # 計算各區間天數並累加 (包含結束當天，故 +1)
    total_days_stats['1. Training Set (訓練集)']      += (t_end - t_start).days + 1
    total_days_stats['2. Purge Buffer 1 (隔離區一)'] += (p1_end - p1_start).days + 1
    total_days_stats['3. Validation Set (驗證集)']     += (v_end - v_start).days + 1
    
    # 針對 Purge2 進行時間倒置的防呆檢查 (正常情況下 p2_end 應小於等於 p2_start)
    # 因為在你的定義中：Purge2_Start 是 Test_Start 的前一天，Purge2_End 是 Val_End 的後一天
    # 在時間軸上，它是由右往左倒扣的區間
    if p2_start >= p2_end:
        total_days_stats['4. Purge Buffer 2 (隔離區二)'] += (p2_start - p2_end).days + 1
    else:
        # 若發生異常時間交錯，則不計入天數
        pass
        
    total_days_stats['5. Testing Set (測試集)']        += (te_end - te_start).days + 1

# 轉換為 DataFrame 方便處理
df_total_share = pd.DataFrame(list(total_days_stats.items()), columns=['Phase', 'Total_Days'])

# 計算總計天數與總體百分比
grand_total_days = df_total_share['Total_Days'].sum()
df_total_share['Percentage (%)'] = ((df_total_share['Total_Days'] / grand_total_days) * 100).round(2)

# ==========================================
# 1. 輸出文字統計結果
# ==========================================
print(f"🌍 全市場 50 支股票總累積資料天數: {grand_total_days:,} 天\n")
print(df_total_share.to_string(index=False))

# ==========================================
# 2. 繪製總體佔比甜甜圈圖
# ==========================================
# 顏色地圖與前面的甘特圖完全保持一致
color_map_extended = {
    '1. Training Set (訓練集)': '#3182bd',
    '2. Purge Buffer 1 (隔離區一)': '#969696',
    '3. Validation Set (驗證集)': '#e6550d',
    '4. Purge Buffer 2 (隔離區二)': '#cccccc',
    '5. Testing Set (測試集)': '#2ca02c'
}

fig = px.pie(
    df_total_share, 
    values='Total_Days', 
    names='Phase',
    title=f'綜合全市場股票各階段資料量總體佔比 (總計: {grand_total_days:,} 天)',
    color='Phase',
    color_discrete_map=color_map_extended,
    hole=0.4 # 設為 0.4 形成甜甜圈圖
)

# 調整文字與 Hover 資訊
fig.update_traces(
    textposition='inside', 
    textinfo='percent+label',
    hovertemplate="<b>%{label}</b><br>總計天數: %{value:,} 天<br>總體佔比: %{percent}<extra></extra>"
)

fig.update_layout(
    template='plotly_white',
    legend_title_text='資料集階段'
)

fig.show()

In [ ]:

hparameter_tuning_folder = os.path.join(expected_return_prediction_folder, 'Hyperparameter Tuning')



In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib
import matplotlib.pyplot as plt  # 用於繪製訓練趨勢圖
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Input, LeakyReLU # 💡 新增：引入 LeakyReLU
from tensorflow.keras.callbacks import EarlyStopping

# ==========================================
# 0. 參數與目錄設定
# ==========================================

WINDOW_SIZE = 63       # LSTM 過去回看的天數
FORWARD_DAYS = 63      # 未來一季對應的交易日天數 (標籤重疊長度)
BATCH_SIZE = 512       # 加大批次量以平滑重疊標籤的梯度
EPOCHS = 60            # 給予足夠的回合數讓單層模型收斂

# 設定訓練過程儲存目錄，並自動建立資料夾
training_process_folder = os.path.join(expected_return_prediction_folder, 'Training Process')
os.makedirs(training_process_folder, exist_ok=True)

# 定義特徵與標籤
target_col = 'return_fwd_1q'
feature_cols = [
    'stock_id', 'Trading_Volume', 'Trading_money', 'open', 'max', 'min', 'close',
    'spread', 'Trading_turnover', 'daily_return', 'MA_20', 'MA_60', 'Bias_20', 
    'Bias_60', 'ROC_20', 'ROC_60', 'Vol_60', 'Vol_MA_20', 'Volume_Ratio_20', 'Price_Position_60'
]

# 建立序列的輔助函數
def create_sequences(features_arr, target_arr, window_size):
    """
    將 2D 的表格資料轉換為 3D 張量 [samples, time_steps, features] 供 LSTM 使用
    """
    X, y = [], []
    for i in range(len(features_arr) - window_size + 1):
        X.append(features_arr[i : i + window_size])
        y.append(target_arr[i + window_size - 1])
    return np.array(X), np.array(y)

# ==========================================
# 1. 處理 Training Set 與 Validation Set (依時間嚴格切割)
# ==========================================
print("⏳ 開始依時間軸處理 Training 與 Validation Set，並儲存 Scalers...")
X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []

for stock_id, df in all_data_dic_backtest.items():
    df = df.sort_values('date').reset_index(drop=True)
    
    # 複製資料以避免更動原字典
    temp_df = df.copy()
    temp_df['stock_id'] = temp_df['stock_id'].astype(int)
    
    # 1.1 依日期切割（假設最後 15% 的時間作為驗證期）
    split_idx = int(len(temp_df) * 0.85)
    
    df_train_part = temp_df.iloc[:split_idx].copy()
    # 為了防止標籤重疊造成的資訊洩漏，驗證集起點必須往後挪 FORWARD_DAYS 天（緩衝禁區）
    df_val_part = temp_df.iloc[split_idx + FORWARD_DAYS:].copy()
    
    # 1.2 僅用「訓練集部分」來 fit 標準化器，再套用至兩者
    if len(df_train_part) < WINDOW_SIZE:
        continue
        
    scaler = StandardScaler()
    df_train_part[feature_cols] = scaler.fit_transform(df_train_part[feature_cols])
    
    # 儲存 Scaler 供 Testing Set 使用
    scaler_path = os.path.join(std_scalars_folder, f'{stock_id}_scaler.pkl')
    joblib.dump(scaler, scaler_path)
    
    # 轉為序列：訓練集
    X_seq_train, y_seq_train = create_sequences(
        df_train_part[feature_cols].values, df_train_part[target_col].values, WINDOW_SIZE
    )
    X_train_list.append(X_seq_train)
    y_train_list.append(y_seq_train)
    
    # 轉為序列：驗證集 (若驗證集長度足夠)
    if len(df_val_part) >= WINDOW_SIZE:
        df_val_part[feature_cols] = scaler.transform(df_val_part[feature_cols]) 
        X_seq_val, y_seq_val = create_sequences(
            df_val_part[feature_cols].values, df_val_part[target_col].values, WINDOW_SIZE
        )
        X_val_list.append(X_seq_val)
        y_val_list.append(y_seq_val)

# 合併所有股票的訓練與驗證序列
X_train_all = np.concatenate(X_train_list, axis=0)
y_train_all = np.concatenate(y_train_list, axis=0)
X_val_all = np.concatenate(X_val_list, axis=0)
y_val_all = np.concatenate(y_val_list, axis=0)

print(f"✅ 資料準備完成！")
print(f"   - 訓練集樣本數: {X_train_all.shape[0]}, 特徵數: {X_train_all.shape[2]}")
print(f"   - 驗證集樣本數: {X_val_all.shape[0]}")

# ==========================================
# 2. 建立並訓練 LSTM 模型 (💡 已更換為 LeakyReLU)
# ==========================================
print("\n⏳ 開始訓練極簡版單層 LSTM 模型...")
model = Sequential([
    Input(shape=(WINDOW_SIZE, len(feature_cols))),
    
    # LSTM 層的內部激活函數維持常規的 'tanh'
    LSTM(32, activation='tanh', return_sequences=False), 
    Dropout(0.35),                     
    
    # 【關鍵修改點】隱藏層 Dense 先不給 activation，並在下一行單獨疊加 LeakyReLU
    Dense(16),
    LeakyReLU(negative_slope=0.1),    # 💡 允許負數區間有 10% 的微弱洩漏，防止神經元壞死
    
    # 輸出層保持不變 (Linear)，回歸問題核心
    Dense(1) 
])

# 優化器與損失函數維持不變
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 設定 Early Stopping 煞車機制
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# 執行訓練
history = model.fit(
    X_train_all, y_train_all,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_all, y_val_all), 
    callbacks=[early_stop],
    verbose=1
)
print("✅ 模型訓練完成！")

# ==========================================
# 💾 儲存訓練過程相關資料
# ==========================================
print(f"\n💾 開始儲存訓練歷史與模型至: {training_process_folder} ...")

model_save_path = os.path.join(training_process_folder, 'lstm_stock_model.keras')
model.save(model_save_path)

history_df = pd.DataFrame(history.history)
history_csv_path = os.path.join(training_process_folder, 'training_history.csv')
history_df.to_csv(history_csv_path, index_label='epoch')

plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Train Loss (MSE)', color='#3182bd', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)', color='#e6550d', linewidth=2)
plt.title('Model Loss Convergence Trend (Single-Layer Leaky ReLU)', fontsize=14, fontweight='bold') # 💡 修正標題
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

plot_save_path = os.path.join(training_process_folder, 'loss_convergence_plot.png')
plt.savefig(plot_save_path, bbox_inches='tight', dpi=150)
plt.close()
print("✅ 歷史紀錄與趨勢圖儲存完畢。")


# ==========================================
# 3. 處理 Testing Set 並進行滾動預測
# ==========================================
print("\n⏳ 開始進行 Testing Set 預測 (防走光滾動預測)...")

for stock_id, df_test in all_data_dic_test.items():
    if stock_id not in all_data_dic_backtest:
        print(f"⚠️ 找不到股票 {stock_id} 的歷史資料，跳過預測。")
        continue
        
    scaler_path = os.path.join(std_scalars_folder, f'{stock_id}_scaler.pkl')
    if not os.path.exists(scaler_path):
        continue
        
    scaler = joblib.load(scaler_path)
    
    df_backtest = all_data_dic_backtest[stock_id].sort_values('date').reset_index(drop=True)
    history_tail = df_backtest.tail(WINDOW_SIZE - 1).copy()
    
    df_combined = pd.concat([history_tail, df_test], ignore_index=True)
    df_combined['stock_id'] = df_combined['stock_id'].astype(int)
    
    df_combined[feature_cols] = scaler.transform(df_combined[feature_cols])
    
    features_arr = df_combined[feature_cols].values
    target_arr = df_combined[target_col].values 
    
    X_test_seq, y_true_seq = create_sequences(features_arr, target_arr, WINDOW_SIZE)
    
    if len(X_test_seq) > 0:
        predictions = model.predict(X_test_seq, verbose=0).flatten()
        
        result_df = pd.DataFrame({
            'date': df_test['date'].values,
            'stock_id': [stock_id] * len(predictions),
            'true_return_fwd_1q': df_test[target_col].values,
            'predicted_return_fwd_1q': predictions
        })
        
        save_path = os.path.join(prediction_results_folder, f'{stock_id}_predictions.csv')
        result_df.to_csv(save_path, index=False)

print(f"\n✅ 所有測試集預測完畢！結果已儲存至：{prediction_results_folder}")

In [ ]:
# 直接在終端機或 Jupyter Notebook 印出模型摘要
model.summary()

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score 

# ==========================================
# 0. 參數與目錄設定
# ==========================================

print("⏳ Loading prediction results...")

# 讀取所有預測結果 CSV
all_preds = []
for file in os.listdir(prediction_results_folder):
    if file.endswith('.csv'):
        file_path = os.path.join(prediction_results_folder, file)
        df_temp = pd.read_csv(file_path)
        all_preds.append(df_temp)

# 合併為單一 DataFrame
df_results = pd.concat(all_preds, ignore_index=True)
df_results['date'] = pd.to_datetime(df_results['date'])

# 為了評估模型，我們只能拿「真實報酬率不為空 (Not NaN)」的資料來對答案
df_eval = df_results.dropna(subset=['true_return_fwd_1q', 'predicted_return_fwd_1q']).copy()

print(f"✅ Data loaded successfully! Total evaluable rows: {len(df_eval)}")

# ==========================================
# 1. 績效指標計算 (Performance Metrics)
# ==========================================
# 1. MAE & RMSE
mae = mean_absolute_error(df_eval['true_return_fwd_1q'], df_eval['predicted_return_fwd_1q'])
rmse = np.sqrt(mean_squared_error(df_eval['true_return_fwd_1q'], df_eval['predicted_return_fwd_1q']))

# 💡 業界標準：計算每日 Rank IC，再求 average 值
def calc_daily_rank_ic(group):
    # method='spearman' 代表計算排名相關係數 (Rank IC)
    return group['true_return_fwd_1q'].corr(group['predicted_return_fwd_1q'], method='spearman')

# 💡 【關鍵修正點】：在 groupby 後方直接切片出運算所需的欄位，徹底絕後患
daily_ics = (
    df_eval.groupby('date')[['true_return_fwd_1q', 'predicted_return_fwd_1q']]
    .apply(calc_daily_rank_ic, include_groups=False) # 新版 pandas 強烈建議加上 include_groups=False
)

# 排除因為當天樣本太少算不出來的 NaN，並取平均
correlation = daily_ics.dropna().mean()

# 3. 計算 R-squared (決定係數)
r_squared = r2_score(df_eval['true_return_fwd_1q'], df_eval['predicted_return_fwd_1q'])

# 4. 方向準確率 (Directional Accuracy) - 預測漲跌方向的勝率
true_direction = np.where(df_eval['true_return_fwd_1q'] > 0, 1, 0)
pred_direction = np.where(df_eval['predicted_return_fwd_1q'] > 0, 1, 0)
directional_accuracy = np.mean(true_direction == pred_direction) * 100

# 輸出結果並存成 TXT 
metrics_text = (
    "=========================================\n"
    "    LSTM Model Performance Summary       \n"
    "=========================================\n"
    f"Total Evaluated Samples : {len(df_eval)}\n"
    f"Mean Absolute Error (MAE): {mae:.4f}\n"
    f"Root Mean Squared (RMSE) : {rmse:.4f}\n"
    f"Information Coeff (IC)   : {correlation:.4f}\n"
    f"R-squared (R2)           : {r_squared:.4f}\n" 
    f"Directional Accuracy     : {directional_accuracy:.2f}%\n"
    "=========================================\n"
)

print("\n" + metrics_text)
with open(os.path.join(analysis_folder, 'model_performance_metrics.txt'), 'w') as f:
    f.write(metrics_text)

# ==========================================
# 2. 繪圖與視覺化 (Plotly)
# ==========================================
print("⏳ Generating Plotly interactive charts...")

# --- Plot 1: Scatter Plot (Actual vs Predicted) ---
fig_scatter = px.scatter(
    df_eval, 
    x='true_return_fwd_1q', 
    y='predicted_return_fwd_1q',
    opacity=0.4,
    title='Actual vs. Predicted 1-Quarter Returns',
    labels={
        'true_return_fwd_1q': 'Actual Return (Y)', 
        'predicted_return_fwd_1q': 'Predicted Return (Y_hat)'
    },
    template="plotly_white"
)
min_val = min(df_eval['true_return_fwd_1q'].min(), df_eval['predicted_return_fwd_1q'].min())
max_val = max(df_eval['true_return_fwd_1q'].max(), df_eval['predicted_return_fwd_1q'].max())
fig_scatter.add_shape(
    type="line", line=dict(color="red", dash="dash"),
    x0=min_val, y0=min_val, x1=max_val, y1=max_val
)
fig_scatter.write_html(os.path.join(analysis_folder, 'actual_vs_predicted_scatter.html'))
fig_scatter.show()


# --- Plot 2: Residual Distribution (Error Distribution) ---
df_eval['residual'] = df_eval['predicted_return_fwd_1q'] - df_eval['true_return_fwd_1q']
fig_hist = px.histogram(
    df_eval, 
    x='residual', 
    nbins=100, 
    title='Distribution of Prediction Errors (Predicted - Actual)',
    labels={'residual': 'Error Value'},
    color_discrete_sequence=['indianred'],
    template="plotly_white"
)
fig_hist.write_html(os.path.join(analysis_folder, 'prediction_error_distribution.html'))
fig_hist.show()


# --- Plot 3: Time Series Trend (Average Daily Returns) ---
df_daily = df_eval.groupby('date')[['true_return_fwd_1q', 'predicted_return_fwd_1q']].mean().reset_index()

fig_time = go.Figure()
fig_time.add_trace(go.Scatter(
    x=df_daily['date'], y=df_daily['true_return_fwd_1q'], 
    mode='lines', name='Actual Return (Market Avg)', line=dict(color='royalblue')
))
fig_time.add_trace(go.Scatter(
    x=df_daily['date'], y=df_daily['predicted_return_fwd_1q'], 
    mode='lines', name='Predicted Return (Market Avg)', line=dict(color='darkorange', dash='dot')
))
fig_time.update_layout(
    title='Average Predicted vs Actual Returns Over Time',
    xaxis_title='Date',
    yaxis_title='Average Return',
    template="plotly_white",
    hovermode="x unified"
)
fig_time.write_html(os.path.join(analysis_folder, 'average_returns_over_time.html'))
fig_time.show()

print(f"✅ Analysis complete! All charts and metrics are saved in: {analysis_folder} and displayed on screen.")

In [ ]:
# ==========================================
# 0. 參數與目錄設定
# ==========================================


print(f"⏳ Start processing and plotting for each stock...\nTarget folder: {plots_folder}")

# 計算成功與失敗的數量
success_count = 0
skip_count = 0

# ==========================================
# 1. 讀取並個別繪製 50 支股票的圖表
# ==========================================
for file in os.listdir(prediction_results_folder):
    if file.endswith('.csv'):
        # 取得股票代號 (假設檔名是 '2330_predictions.csv')
        stock_id = file.split('_')[0]
        file_path = os.path.join(prediction_results_folder, file)
        
        # 讀取資料
        df = pd.read_csv(file_path)
        df['date'] = pd.to_datetime(df['date'])
        
        # 過濾掉包含 NaN 的資料列，確保有真實答案可以對
        df_clean = df.dropna(subset=['true_return_fwd_1q', 'predicted_return_fwd_1q']).copy()
        
        if len(df_clean) == 0:
            print(f"⚠️ Stock {stock_id} has no valid evaluation data. Skipped.")
            skip_count += 1
            continue
            
        # 按照時間排序以繪製正確的折線圖
        df_clean = df_clean.sort_values('date')
        
        # ==========================================
        # 2. 計算個別股票的績效指標
        # ==========================================
        mae = mean_absolute_error(df_clean['true_return_fwd_1q'], df_clean['predicted_return_fwd_1q'])
        rmse = np.sqrt(mean_squared_error(df_clean['true_return_fwd_1q'], df_clean['predicted_return_fwd_1q']))
        
        # ==========================================
        # 3. 使用 Matplotlib 繪製折線圖
        # ==========================================
        # 設定畫布大小與解析度 (DPI=150 確保圖片清晰)
        plt.figure(figsize=(12, 6), dpi=150)
        
        # 繪製真實值與預測值的折線
        plt.plot(df_clean['date'], df_clean['true_return_fwd_1q'], 
                 label='Actual 1Q Return', color='#1f77b4', linewidth=2)
        plt.plot(df_clean['date'], df_clean['predicted_return_fwd_1q'], 
                 label='Predicted 1Q Return', color='#ff7f0e', linestyle='--', linewidth=2)
        
        # 設定標題 (包含 MAE 與 RMSE)
        title_text = f"Stock {stock_id} - Actual vs Predicted 1-Quarter Returns\nMAE: {mae:.4f} | RMSE: {rmse:.4f}"
        plt.title(title_text, fontsize=14, pad=15, fontweight='bold')
        
        # 設定 X 軸與 Y 軸標籤
        plt.xlabel('Date', fontsize=12)
        plt.ylabel('Return', fontsize=12)
        
        # 加上格線與圖例
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.legend(loc='best', fontsize=10)
        
        # 自動旋轉 X 軸的日期標籤，避免重疊
        plt.gcf().autofmt_xdate()
        
        # ==========================================
        # 4. 儲存圖表並關閉畫布
        # ==========================================
        save_path = os.path.join(plots_folder, f'{stock_id}_prediction_plot.png')
        
        # bbox_inches='tight' 確保存檔時周圍的字不會被裁切掉
        plt.savefig(save_path, bbox_inches='tight')
        
        # 💡 重要：一定要 close，否則迴圈畫 50 張圖會吃光記憶體
        plt.close()
        
        success_count += 1

print("-" * 50)
print(f"✅ Plotting complete! Successfully generated {success_count} plots.")
if skip_count > 0:
    print(f"⚠️ Skipped {skip_count} stocks due to missing data.")
print(f"📂 All plots are saved in: {plots_folder}")

In [ ]:
df_results

In [ ]:
# 1. 設定目標日期 (確保格式為 datetime)
target_date = pd.to_datetime(date_dict['Announcement_Date'])

# 2. 從先前的 df_results 中篩選出 2025-03-07 當天的資料
df_ER_prediction_AD = df_results[df_results['date'] == target_date].copy()

# 3. 計算預測值與實際值的「絕對值差」
df_ER_prediction_AD['abs_error'] = (df_ER_prediction_AD['predicted_return_fwd_1q'] - df_ER_prediction_AD['true_return_fwd_1q']).abs()

# 4. 整理 DataFrame，只保留我們關心的欄位，並重新命名讓它更易讀
df_ER_prediction_AD = df_ER_prediction_AD[['stock_id', 'true_return_fwd_1q', 'predicted_return_fwd_1q', 'abs_error']]
df_ER_prediction_AD.columns = ['Stock_ID', 'Actual_Return', 'Predicted_Return', 'Absolute_Error']

# 5. 依照「絕對誤差 (Absolute_Error)」由小到大排序，這樣最準確的股票會排在最前面
df_ER_prediction_AD = df_ER_prediction_AD.sort_values(by='Absolute_Error', ascending=True).reset_index(drop=True)

# 印出結果預覽 (前 10 名最準的股票)
print(f"📊 2025-03-07 預測結果總表 (共 {len(df_ER_prediction_AD)} 檔股票):")
print("-" * 60)
print(df_ER_prediction_AD.head(10))



In [ ]:
# (選用) 如果你想把這個表格存成 CSV 獨立觀察

df_ER_prediction_AD.to_csv(expected_return_path, index=False)
print(f"\n✅ 資料已儲存至：{expected_return_path}")

In [ ]:
import os
import plotly.express as px

# 💡 安全機制：如果你的原始 df 還沒改名，先幫它改名（若已改名這行不會有影響）
rename_dict = {
    'stock_id': 'Stock_ID', 
    'true_return_fwd_1q': 'Actual_Return', 
    'predicted_return_fwd_1q': 'Predicted_Return'
}
df_ER_prediction_AD = df_ER_prediction_AD.rename(columns=rename_dict)

# 2. 建立 expected_return_AD DataFrame，並計算絕對值差（統一使用新欄位名）
expected_return_AD = df_ER_prediction_AD[['Stock_ID', 'Actual_Return', 'Predicted_Return']].copy()

# 計算絕對誤差: | 預測值 - 真實值 |
expected_return_AD['absolute_error'] = (expected_return_AD['Predicted_Return'] - expected_return_AD['Actual_Return']).abs()

# 計算所有股票的平均絕對誤差
mean_absolute_error_AD = expected_return_AD['absolute_error'].mean()

# 依照絕對誤差由大到小排序
expected_return_AD = expected_return_AD.sort_values(by='absolute_error', ascending=False).reset_index(drop=True)
expected_return_AD['Stock_ID'] = expected_return_AD['Stock_ID'].astype(str)

print(f"✅ 成功擷取資料！2025-03-07 當日 50 支股票的平均絕對誤差為: {mean_absolute_error_AD:.4f}")

# 3. 使用 Plotly 視覺化絕對值差 (Bar Chart)
fig = px.bar(
    expected_return_AD,
    x='Stock_ID',              # 💡 統一為大寫
    y='absolute_error',
    title='Absolute Prediction Error by Stock on 2025-03-07 (with Market Average)',
    labels={
        'Stock_ID': 'Stock ID', # 💡 統一為大寫
        'absolute_error': 'Absolute Error (|Predicted - Actual|)'
    },
    hover_data={
        'Actual_Return': ':.4f',    # 💡 統一為新欄位名
        'Predicted_Return': ':.4f', # 💡 統一為新欄位名
        'absolute_error': ':.4f'
    },
    template='plotly_white',
    color='absolute_error',                 
    color_continuous_scale='Reds'           
)

# 💡 加入代表「平均絕對誤差」的水平參考線
fig.add_hline(
    y=mean_absolute_error_AD,
    line_dash="dash",               # 設定為虛線
    line_color="rgba(50, 50, 50, 0.8)", # 暗灰色，避免干擾視覺
    line_width=2,
    annotation_text=f"Market Average: {mean_absolute_error_AD:.4f}", # 線上的提示文字
    annotation_position="top right", # 文字標籤置於右上方
    annotation_font_color="black",
    annotation_font_size=12
)

# 4. 更新版面設定
fig.update_layout(
    xaxis_type='category', 
    xaxis_tickangle=-45,
    height=600,
    width=1400
)

# 5. 儲存與顯示圖表
error_plot_path = os.path.join(analysis_folder, 'absolute_error_announcement_date_with_avg.html')
fig.write_html(error_plot_path)
fig.show()

print(f"✅ 圖表（含平均值線）已儲存至: {error_plot_path} 并顯示於畫面上。")

### 2.1.2. Construct Covariance Matrix

In [ ]:
# 1. 初始化單一儲存容器，放入除 0050 以外的所有資料
all_data_dic = {}

# 2. 讀取所有資料並儲存
for filename in os.listdir(adjusted_daily_stock_data_folder):
    if filename.endswith('.feather'):
        stock_id = filename.replace('.feather', '')
        
        # 💡 判斷：如果是 '0050' 則直接跳過
        if stock_id == '0050':
            continue
            
        file_path = os.path.join(adjusted_daily_stock_data_folder, filename)
        
        # 讀取完整的 DataFrame
        df = pd.read_feather(file_path)
        
        # 確保 date 欄位是 datetime 格式
        df['date'] = pd.to_datetime(df['date'])
        
        # 依照日期排序，確保資料具有時間順序性
        df = df.sort_values('date').reset_index(drop=True)
        
        # 將處理好、排序過且重置 index 的完整 DataFrame 存入字典中
        all_data_dic[stock_id] = df

print("✅ 所有 DataFrame 已成功讀取，並已排除 '0050'！")
print(f"📊 總共讀取的股票數量: {len(all_data_dic)}")

In [ ]:
all_data_dic['2330']

In [ ]:
all_data_dic['2330']

In [ ]:
strategy_date_df

In [ ]:
# ==========================================
# 0. 參數設定
# ==========================================
target_date = pd.to_datetime(date_dict['Announcement_Date'])
lookback_date = target_date - pd.Timedelta(days=365) 
frequency = 63 

historical_returns_list = []

print("⏳ 開始從 all_data_dic 擷取歷史日報酬率資料...")

# ==========================================
# 1. 提取 2025-03-07 之前的歷史日報酬率
# ==========================================
for stock_id, df in all_data_dic.items():
    temp_df = df.copy()
    temp_df = temp_df.sort_values('date').reset_index(drop=True)
    
    # 💡 強制確保 close 是數字型態 (若混雜字串會被轉為 NaN)
    if 'close' in temp_df.columns:
        temp_df['close'] = pd.to_numeric(temp_df['close'], errors='coerce')
    
    if 'daily_return' not in temp_df.columns:
        temp_df['daily_return'] = temp_df['close'].pct_change(1)
        
    mask = (temp_df['date'] >= lookback_date) & (temp_df['date'] <= target_date)
    temp_df = temp_df[mask].copy()
    
    if not temp_df.empty:
        temp_df = temp_df[['date', 'stock_id', 'daily_return']]
        historical_returns_list.append(temp_df)

# ==========================================
# 2. 轉換與終極清洗機制 (確保絕對沒有 NaN/Inf)
# ==========================================
df_historical_returns = pd.concat(historical_returns_list, ignore_index=True)

# 強制將 daily_return 轉為純粹的浮點數
df_historical_returns['daily_return'] = pd.to_numeric(df_historical_returns['daily_return'], errors='coerce')

# 轉換成二維矩陣
returns_matrix = df_historical_returns.pivot(index='date', columns='stock_id', values='daily_return')

# 🧹 步驟 1: 將所有的無限大 (Inf, -Inf) 轉為 NaN
returns_matrix = returns_matrix.replace([np.inf, -np.inf], np.nan)

# 🧹 步驟 2: 刪除「所有股票都是 NaN」的日期 (通常是沒有開盤的假日)
returns_matrix = returns_matrix.dropna(how='all')

# 🧹 步驟 3: 將剩餘的個別 NaN 強制補為 0.0 (代表無漲跌)
returns_matrix = returns_matrix.fillna(0.0)

# 🧹 步驟 4: 強制將整個矩陣轉為 numpy 的 float64 型態，滿足 sklearn 的嚴格要求
returns_matrix = returns_matrix.astype(np.float64)

# 🔎 最終防呆檢查
nan_count = returns_matrix.isna().sum().sum()
print(f"🧹 清洗完畢！目前的 NaN 總數量：{nan_count} (必須為 0)")

if nan_count > 0:
    raise ValueError("⚠️ 資料中仍然包含無法清理的 NaN，請檢查原始資料是否損壞！")

print(f"✅ 成功建立歷史報酬率矩陣！矩陣維度：{returns_matrix.shape}")

# ==========================================
# 3. 透過 PyPortfolioOpt 計算共變異數矩陣
# ==========================================
print("\n⏳ 正在計算共變異數矩陣 (Covariance Matrix)...")

# 💡 關鍵修復：加入 returns_data=True 告訴套件這是報酬率，不是原始價格！
cov_matrix_lw = risk_models.CovarianceShrinkage(
    returns_matrix, 
    returns_data=True,  # <--- 就是這個救命參數
    frequency=frequency
).ledoit_wolf()

# 如果你也有跑其他兩種矩陣，也都要加上這個參數：
cov_matrix_ewma = risk_models.exp_cov(returns_matrix, span=180, returns_data=True, frequency=frequency)
cov_matrix_sample = risk_models.sample_cov(returns_matrix, returns_data=True, frequency=frequency)

print("✅ 共變異數矩陣計算成功！")
print("\n📋 Ledoit-Wolf 壓縮共變異數矩陣預覽 (前 5x5) :")
print(cov_matrix_lw.iloc[:5, :5])

In [ ]:
# 1. 常數相關矩陣 (Constant Correlation) 作為目標
# 雖然每一檔股票的波動度（變異數）不同，但如果我們把時間拉長、去掉市場雜訊，所有股票兩兩之間的「相關係數 (Correlation)」應該都差不多。
cov_matrix_lw_cc = risk_models.risk_matrix(
    returns_matrix,
    method="ledoit_wolf_constant_correlation", # 正確的參數字串
    returns_data=True,
    frequency=frequency
)

# 2. 單因子模型矩陣 (Single Factor) 作為目標
# 股票之間之所以會齊漲齊跌（產生共變異數），完全是因為它們都受到同一個「總體市場因子」（例如大盤指數）的驅動。除了大盤的影響之外，股票自身的特質風險 (Idiosyncratic Risk) 彼此互不相關。
cov_matrix_lw_sf = risk_models.risk_matrix(
    returns_matrix,
    method="ledoit_wolf_single_factor",        # 正確的參數字串
    returns_data=True,
    frequency=frequency
)

In [ ]:
# Calculate cov_matrix_answer

# ==========================================
# 0. 參數設定 (指定新的時間區間)
# ==========================================
start_date = pd.to_datetime(date_dict['Strategy_Start_Date'])
end_date = pd.to_datetime(date_dict['Strategy_End_Date'])

annualization_factor = 63

historical_returns_list = []
print(f"⏳ 開始擷取 {start_date.date()} 至 {end_date.date()} 的歷史日報酬率資料...")

# ==========================================
# 1. 提取指定區間的歷史日報酬率
# ==========================================
for stock_id, df in all_data_dic.items():
    temp_df = df.copy()
    
    # 確保日期是 datetime 格式以便精準篩選
    temp_df['date'] = pd.to_datetime(temp_df['date'])
    temp_df = temp_df.sort_values('date').reset_index(drop=True)
    
    if 'close' in temp_df.columns:
        temp_df['close'] = pd.to_numeric(temp_df['close'], errors='coerce')
    
    if 'daily_return' not in temp_df.columns:
        temp_df['daily_return'] = temp_df['close'].pct_change(1)
        
    # 💡 篩選新的時間區間
    mask = (temp_df['date'] >= start_date) & (temp_df['date'] <= end_date)
    temp_df = temp_df[mask].copy()
    
    if not temp_df.empty:
        # 確保 DataFrame 中有 stock_id 欄位
        if 'stock_id' not in temp_df.columns:
            temp_df['stock_id'] = stock_id
            
        temp_df = temp_df[['date', 'stock_id', 'daily_return']]
        historical_returns_list.append(temp_df)

# ==========================================
# 2. 轉換與清洗機制
# ==========================================
df_historical_returns = pd.concat(historical_returns_list, ignore_index=True)
df_historical_returns['daily_return'] = pd.to_numeric(df_historical_returns['daily_return'], errors='coerce')

# 轉換成二維矩陣 (日期為列，股票為欄)
returns_matrix = df_historical_returns.pivot(index='date', columns='stock_id', values='daily_return')

# 清洗 NaN 與 Inf
returns_matrix = returns_matrix.replace([np.inf, -np.inf], np.nan)
returns_matrix = returns_matrix.dropna(how='all')
returns_matrix = returns_matrix.fillna(0.0)
returns_matrix = returns_matrix.astype(np.float64)

# ==========================================
# 3. 用最基本的方法計算共變異數矩陣
# ==========================================
print("\n⏳ 正在使用 Pandas 基本方法計算樣本共變異數矩陣...")

# 💡 最基本的方法：直接呼叫 Pandas 內建的 .cov()，並乘上年化常數
cov_matrix_answer = risk_models.sample_cov(returns_matrix, returns_data=True, frequency=annualization_factor)

In [ ]:
# 3. 定義一個字典，將變數名稱與實際物件對應起來
cov_matrices = {
    'cov_matrix_answer.csv': cov_matrix_answer,
    'cov_matrix_sample.csv': cov_matrix_sample,
    'cov_matrix_ewma.csv': cov_matrix_ewma,
    'cov_matrix_lw.csv': cov_matrix_lw,
    'cov_matrix_lw_cc.csv': cov_matrix_lw_cc,
    'cov_matrix_lw_sf.csv': cov_matrix_lw_sf
}

# 4. 迴圈執行存檔操作
print("💾 開始儲存共變異數矩陣...")
for filename, df in cov_matrices.items():
    file_path = os.path.join(cov_matrix_folder, filename)
    # 使用 utf-8-sig 確保 Excel 兼容性
    df.to_csv(file_path, index=True, encoding='utf-8-sig')
    print(f"✅ 已成功儲存: {filename}")

print("\n🎉 所有共變異數矩陣皆已存檔完畢。")

In [ ]:
cov_matrix_answer

In [ ]:
cov_matrix_sample

In [ ]:
cov_matrix_ewma

In [ ]:
cov_matrix_lw

In [ ]:
cov_matrix_lw_cc 

In [ ]:
cov_matrix_lw_sf

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("⏳ 正在生成 Plotly 互動式共變異數矩陣熱力圖 (6-Grid Heatmaps)...")

# 1. 統一顏色刻度 (Color Scale) - 加入 cov_matrix_answer 一起計算
all_values = np.concatenate([
    cov_matrix_answer.values.flatten(),  # <--- 新增
    cov_matrix_sample.values.flatten(),
    cov_matrix_ewma.values.flatten(),
    cov_matrix_lw.values.flatten(),
    cov_matrix_lw_cc.values.flatten(),  
    cov_matrix_lw_sf.values.flatten()   
])
zmin = np.percentile(all_values, 1)
zmax = np.percentile(all_values, 99)

# 2. 建立 1x6 的並排子圖畫布 (原本是 cols=5 改為 cols=6)
fig = make_subplots(
    rows=1, cols=6,
    subplot_titles=(
        "1. True Answer<br>(Future Realized)",      # <--- 新增
        "2. Sample Covariance<br>(Noisy & Unstable)",
        "3. EWMA Covariance<br>(Recent Vol Focus)",
        "4. LW: Constant Var<br>(Scaled Identity)",
        "5. LW: Constant Corr<br>(Market Avg Corr)",
        "6. LW: Single Factor<br>(Market Beta)"
    ),
    horizontal_spacing=0.015  # 稍微縮小間距讓 6 張圖排列更緊湊
)

colorscale = 'RdBu_r'

# --- 圖 1：未來的真實答案 (Answer) ---
fig.add_trace(go.Heatmap(
    z=cov_matrix_answer.values,
    zmin=zmin, zmax=zmax,
    colorscale=colorscale,
    showscale=False, 
    hoverinfo="z"
), row=1, col=1)

# --- 圖 2：傳統歷史樣本 (Sample) ---
fig.add_trace(go.Heatmap(
    z=cov_matrix_sample.values,
    zmin=zmin, zmax=zmax,
    colorscale=colorscale,
    showscale=False, 
    hoverinfo="z"
), row=1, col=2)

# --- 圖 3：指數加權移動平均 (EWMA) ---
fig.add_trace(go.Heatmap(
    z=cov_matrix_ewma.values,
    zmin=zmin, zmax=zmax,
    colorscale=colorscale,
    showscale=False,
    hoverinfo="z"
), row=1, col=3)

# --- 圖 4：Ledoit-Wolf 壓縮 (縮放單位矩陣目標) ---
fig.add_trace(go.Heatmap(
    z=cov_matrix_lw.values,
    zmin=zmin, zmax=zmax,
    colorscale=colorscale,
    showscale=False,
    hoverinfo="z"
), row=1, col=4)

# --- 圖 5：Ledoit-Wolf 壓縮 (常數相關目標) ---
fig.add_trace(go.Heatmap(
    z=cov_matrix_lw_cc.values,
    zmin=zmin, zmax=zmax,
    colorscale=colorscale,
    showscale=False,
    hoverinfo="z"
), row=1, col=5)

# --- 圖 6：Ledoit-Wolf 壓縮 (單因子模型目標) ---
fig.add_trace(go.Heatmap(
    z=cov_matrix_lw_sf.values,
    zmin=zmin, zmax=zmax,
    colorscale=colorscale,
    # 💡 顏色條移到最後一張圖顯示
    colorbar=dict(
        title=dict(text="Covariance", side="right"), 
        thickness=15,
        len=0.8
    ), 
    hoverinfo="z"
), row=1, col=6)

# 3. 更新版面設定
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False, autorange="reversed")

fig.update_layout(
    title_text="Comparison of Covariance Matrices: Historic Estimators vs. Future Answer",
    title_font=dict(size=24),
    title_x=0.5, 
    height=500,
    width=2000,  # <--- 寬度調整為 2000 讓 6 張圖更好看
    template="plotly_white",
    margin=dict(t=120, b=40, l=40, r=40)
)

fig.show()

print("✅ 6 格矩陣視覺化完成！已成功加入真實答案對比。")

In [ ]:
# 確保所有矩陣的 index 和 columns 順序完全一致，避免因順序錯亂導致計算錯誤
matrix_list = {
    "Sample Covariance": cov_matrix_sample,
    "EWMA Covariance": cov_matrix_ewma,
    "LW: Constant Var": cov_matrix_lw,
    "LW: Constant Corr": cov_matrix_lw_cc,
    "LW: Single Factor": cov_matrix_lw_sf
}

# 用 cov_matrix_answer 的標籤進行統一對齊
target_cols = cov_matrix_answer.columns
target_idx = cov_matrix_answer.index

results = []

print("⏳ 開始計算各矩陣與未來真實答案 (cov_matrix_answer) 的距離...")

for name, mat in matrix_list.items():
    # 💡 防呆：對齊股票代碼與順序
    aligned_mat = mat.reindex(index=target_idx, columns=target_cols).fillna(0.0)
    
    # 1. 計算 Frobenius Norm (數值越小，代表與答案越接近)
    # 公式：sqrt(sum((Matrix_A - Matrix_B) ^ 2))
    matrix_diff = aligned_mat.values - cov_matrix_answer.values
    frobenius_dist = np.linalg.norm(matrix_diff, ord='fro')
    
    # 2. 計算相關係數 (數值越接近 1，代表矩陣結構與答案越相似)
    # 將矩陣拉平為一維向量後計算皮爾森相關係數
    corr_coef = np.corrcoef(aligned_mat.values.flatten(), cov_matrix_answer.values.flatten())[0, 1]
    
    results.append({
        "Model": name,
        "Frobenius Distance (Lower is Better)": frobenius_dist,
        "Correlation with Answer (Higher is Better)": corr_coef
    })

# 3. 轉成 DataFrame 並排序
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="Frobenius Distance (Lower is Better)").reset_index(drop=True)

# --- 輸出評比結果 ---
print("\n🏆 【共變異數矩陣預測準確度評比】 🏆")
print("=" * 75)
print(df_results.to_string(index=False, formatters={
    "Frobenius Distance (Lower is Better)": "{:,.6f}".format,
    "Correlation with Answer (Higher is Better)": "{:.2%}".format
}))
print("=" * 75)

best_model = df_results.iloc[0]['Model']
print(f"📌 結論：根據 Frobenius 距離指標，與未來真實答案最接近的是：【{best_model}】")

In [ ]:
# 假設你已經有了這些矩陣
matrices = {
    "Sample Covariance": cov_matrix_sample,
    "EWMA Covariance": cov_matrix_ewma,
    "LW: Constant Var": cov_matrix_lw,
    "LW: Constant Corr": cov_matrix_lw_cc,
    "LW: Single Factor": cov_matrix_lw_sf
}

eigen_stats = []

for name, mat in matrices.items():
    # 💡 計算特徵值 (使用 eigh 是因為共變異數矩陣是對稱矩陣)
    eigenvalues, _ = np.linalg.eigh(mat.values)
    
    max_eig = np.max(eigenvalues)
    min_eig = np.min(eigenvalues)
    
    # 💡 條件數 (Condition Number) = 最大特徵值 / 最小特徵值
    # 數值越小，代表矩陣結構越穩健，求逆矩陣時越不會放大雜訊
    condition_number = max_eig / min_eig if min_eig > 0 else np.inf
    
    eigen_stats.append({
        "Model": name,
        "Max Eigenvalue (Scale)": max_eig,
        "Min Eigenvalue (Stability)": min_eig,
        "Condition Number (Lower is Better)": condition_number
    })

df_eigen = pd.DataFrame(eigen_stats).sort_values(by="Condition Number (Lower is Better)").reset_index(drop=True)

print("📋 【共變異數矩陣之特徵值穩定度報告】")
print("=" * 80)
print(df_eigen.to_string(index=False, formatters={
    "Max Eigenvalue (Scale)": "{:.6f}".format,
    "Min Eigenvalue (Stability)": "{:.6f}".format,
    "Condition Number (Lower is Better)": "{:,.2f}".format
}))
print("=" * 80)

### 2.1.3. Efficient Frontier


In [ ]:
import time  # ⏱️ 確保匯入時間模組
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pypfopt import EfficientFrontier

# ==========================================
# 0. 輔助函數設定
# ==========================================
def clean_ticker(x):
    s = str(x).strip()
    if s.endswith('.0'): 
        s = s[:-2]
    return s

df_mu_clean = df_ER_prediction_AD.copy()
df_mu_clean['cleaned_id'] = df_mu_clean['Stock_ID'].apply(clean_ticker)
df_mu_clean = df_mu_clean.set_index('cleaned_id')

# ==========================================
# 1. 核心優化與繪圖函數 (已修改為高精度計時)
# ==========================================
def optimize_and_plot(return_col, cov_matrix, model_name, is_actual=False):
    print(f"\n{'='*60}")
    print(f"🚀 正在執行模型: {model_name}")
    print(f"{'='*60}")
    
    cov_df = cov_matrix.copy()
    cov_df.columns = [clean_ticker(c) for c in cov_df.columns]
    cov_df.index = [clean_ticker(i) for i in cov_df.index]
    
    mu_series = df_mu_clean[return_col]
    
    common_stocks = list(set(cov_df.columns).intersection(set(mu_series.index)))
    mu = mu_series.loc[common_stocks]
    cov_clean = cov_df.loc[common_stocks, common_stocks]
    
    stock_volatilities = np.sqrt(np.diag(cov_clean.values))
    
    ret_label = "Actual Return" if is_actual else "Expected Return"
    std_label = "Actual Std" if is_actual else "Expected Std"
    sr_label = "Actual Sharpe Ratio" if is_actual else "Expected Sharpe Ratio"

    # --- ⏱️ 計算 GMVP 耗時 (改用 perf_counter) ---
    start_gmvp = time.perf_counter() # ⚡ 記錄高精度開始時間
    ef_min = EfficientFrontier(mu, cov_clean, solver='SCS')
    ef_min.min_volatility()
    ret_min, vol_min, sharpe_min = ef_min.portfolio_performance(risk_free_rate=0.0)
    weights_min = ef_min.clean_weights()
    end_gmvp = time.perf_counter()   # ⚡ 記錄高精度結束時間
    gmvp_time = end_gmvp - start_gmvp # 精確計算到微秒等級的秒數

    # --- ⏱️ 計算 MSRP 耗時 (改用 perf_counter) ---
    has_max_sharpe = False
    weights_max = {}
    msrp_time = np.nan # 預設為 NaN，若未收斂則保持
    
    try:
        start_msrp = time.perf_counter() # ⚡ 記錄高精度開始時間
        ef_max = EfficientFrontier(mu, cov_clean, solver='SCS')
        ef_max.max_sharpe(risk_free_rate=0.0)
        ret_max, vol_max, sharpe_max = ef_max.portfolio_performance(risk_free_rate=0.0)
        weights_max = ef_max.clean_weights()
        end_msrp = time.perf_counter()   # ⚡ 記錄高精度結束時間
        msrp_time = end_msrp - start_msrp
        has_max_sharpe = True
    except Exception as e:
        print(f"⚠️ {model_name} 的最大夏普組合未收斂。")

    # --- 模擬效率前緣曲線 (此處不納入核心計算耗時) ---
    target_returns = np.linspace(ret_min, mu.max(), 50)
    frontier_vols, frontier_rets = [], []
    for target_r in target_returns:
        try:
            ef_curve = EfficientFrontier(mu, cov_clean, solver='SCS')
            ef_curve.efficient_return(target_return=target_r)
            r, v, _ = ef_curve.portfolio_performance()
            frontier_vols.append(v)
            frontier_rets.append(r)
        except:
            pass

    # --- Plotly 視覺化 ---
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=stock_volatilities, y=mu.values, mode='markers', name='Individual Stocks', text=mu.index,
        hovertemplate=f"<b>Stock: %{{text}}</b><br>{ret_label}: %{{y:.4f}}<br>{std_label}: %{{x:.4f}}<extra></extra>",
        marker=dict(size=8, color='rgba(70, 130, 180, 0.6)', line=dict(width=1, color='DimGray'))
    ))
    fig.add_trace(go.Scatter(
        x=frontier_vols, y=frontier_rets, mode='lines', name='Efficient Frontier',
        line=dict(color='firebrick', width=3),
        hovertemplate=f"<b>Frontier</b><br>{ret_label}: %{{y:.4f}}<br>{std_label}: %{{x:.4f}}<extra></extra>"
    ))
    fig.add_trace(go.Scatter(
        x=[vol_min], y=[ret_min], mode='markers', name='GMVP',
        marker=dict(color='LimeGreen', size=14, symbol='star', line=dict(width=1, color='black')),
        hovertemplate=(f"<b>GMVP</b><br>{ret_label}: %{{y:.4f}}<br>{std_label}: %{{x:.4f}}<br>"
                       f"{sr_label}: {sharpe_min:.4f}<extra></extra>")
    ))
    if has_max_sharpe:
        fig.add_trace(go.Scatter(
            x=[vol_max], y=[ret_max], mode='markers', name='MSRP (Rf=0)',
            marker=dict(color='Gold', size=14, symbol='diamond', line=dict(width=1, color='black')),
            hovertemplate=(f"<b>MSRP</b><br>{ret_label}: %{{y:.4f}}<br>{std_label}: %{{x:.4f}}<br>"
                           f"{sr_label}: {sharpe_max:.4f}<extra></extra>")
        ))
    fig.update_layout(
        title=dict(text=f"<b>Efficient Frontier: {model_name}</b>", font=dict(size=22), x=0.5),
        xaxis_title=f"Portfolio Risk ({std_label})",
        yaxis_title=f"Return ({ret_label})",
        template="plotly_white", width=1000, height=600, hovermode="closest"
    )
    fig.show()

    # --- 整理權重與時間資料 ---
    df_gmvp = pd.Series(weights_min, name=f'{model_name}_GMVP')
    df_msrp = pd.Series(weights_max, name=f'{model_name}_MSRP') if has_max_sharpe else pd.Series(np.nan, index=df_gmvp.index, name=f'{model_name}_MSRP')
    
    weights_df = pd.concat([df_gmvp, df_msrp], axis=1)
    
    # 🧼 建立該模型的耗時字典
    time_dict = {
        "Model_Name": model_name,
        "GMVP_Time_Sec": gmvp_time,
        "MSRP_Time_Sec": msrp_time
    }
    
    return weights_df, time_dict

# ==========================================
# 2. 定義 6 種測試組合並執行
# ==========================================
configurations = [
    {"name": "01_Answer_Actual", "return_col": "Actual_Return", "cov": cov_matrix_answer, "is_actual": True},
    {"name": "02_Sample",        "return_col": "Predicted_Return", "cov": cov_matrix_sample, "is_actual": False},
    {"name": "03_EWMA",          "return_col": "Predicted_Return", "cov": cov_matrix_ewma,   "is_actual": False},
    {"name": "04_LedoitWolf_Identity", "return_col": "Predicted_Return", "cov": cov_matrix_lw,     "is_actual": False},
    {"name": "05_LedoitWolf_ConstCorr", "return_col": "Predicted_Return", "cov": cov_matrix_lw_cc, "is_actual": False},
    {"name": "06_LedoitWolf_Factor", "return_col": "Predicted_Return", "cov": cov_matrix_lw_sf, "is_actual": False}
]

all_weights = []
time_records = [] # 🧼 用來收集時間紀錄的清單

for config in configurations:
    w_df, t_dict = optimize_and_plot(
        return_col=config["return_col"],
        cov_matrix=config["cov"],
        model_name=config["name"],
        is_actual=config["is_actual"]
    )
    all_weights.append(w_df)
    time_records.append(t_dict) # 儲存時間

# ==========================================
# 3. 結果統整與展示
# ==========================================
# 3a. 權重總表
df_all_portfolios = pd.concat(all_weights, axis=1)
df_all_portfolios.index.name = 'Stock_ID'
active_mask = (df_all_portfolios > 0).any(axis=1)
df_active_all = df_all_portfolios[active_mask].fillna(0)

print("\n" + "="*80)
print(" 🏆 全模型投資組合權重總表 (Active Assets Only) ")
print("="*80)
display(df_active_all.style.format("{:.2%}").background_gradient(cmap='Blues', axis=1))

# 3b. ⏱️ 創建耗時統計 DataFrame
df_time_report = pd.DataFrame(time_records)
df_time_report = df_time_report.set_index("Model_Name")

# 算一下每個模型的總優化時間
df_time_report['Total_Optimization_Time_Sec'] = df_time_report.sum(axis=1, skipna=True)

print("\n" + "="*80)
print(" ⏱️ 各模型最佳化求解耗時總表 (秒) ")
print("="*80)
# 使用樣式美化，高精度下維持顯示到小數點後 4 位即可清楚看出差異
display(df_time_report.style.format("{:.4f}秒", na_rep="未收斂").background_gradient(cmap='Reds', axis=0))
print("="*80)
print("✅ 耗時比較結果已儲存至 DataFrame: `df_time_report`")

In [ ]:
df_time_report

In [ ]:
df_all_portfolios

In [ ]:
print("⏳ 正在計算並提取所有模型的投資組合績效指標...")

performance_metrics = []

# 重新遍歷一次 configuration 來單純提取績效指標 (不畫個別前緣)
for config in configurations:
    model_name = config["name"]
    is_actual = config["is_actual"]
    
    # 資料準備
    cov_df = config["cov"].copy()
    cov_df.columns = [clean_ticker(c) for c in cov_df.columns]
    cov_df.index = [clean_ticker(i) for i in cov_df.index]
    mu_series = df_mu_clean[config["return_col"]]
    common_stocks = list(set(cov_df.columns).intersection(set(mu_series.index)))
    mu = mu_series.loc[common_stocks]
    cov_clean = cov_df.loc[common_stocks, common_stocks]

    # 設定標籤文字
    ret_label = "Actual Return" if is_actual else "Expected Return"
    std_label = "Actual Std" if is_actual else "Expected Std"
    sr_label = "Actual Sharpe Ratio" if is_actual else "Expected Sharpe Ratio"

    # --- 1. 計算 GMVP ---
    try:
        ef_min = EfficientFrontier(mu, cov_clean)
        ef_min.min_volatility()
        ret_min, vol_min, sharpe_min = ef_min.portfolio_performance(risk_free_rate=0.0)
        performance_metrics.append({
            "Model": model_name,
            "Portfolio Type": "GMVP (Min Volatility)",
            "Return": ret_min,
            "Risk_Std": vol_min,
            "Sharpe_Ratio": sharpe_min,
            "Return_Label": ret_label,
            "Risk_Label": std_label,
            "Sharpe_Label": sr_label
        })
    except Exception as e:
        print(f"⚠️ {model_name} 的 GMVP 計算失敗。")

    # --- 2. 計算 MSRP ---
    try:
        ef_max = EfficientFrontier(mu, cov_clean)
        ef_max.max_sharpe(risk_free_rate=0.0)
        ret_max, vol_max, sharpe_max = ef_max.portfolio_performance(risk_free_rate=0.0)
        performance_metrics.append({
            "Model": model_name,
            "Portfolio Type": "MSRP (Max Sharpe)",
            "Return": ret_max,
            "Risk_Std": vol_max,
            "Sharpe_Ratio": sharpe_max,
            "Return_Label": ret_label,
            "Risk_Label": std_label,
            "Sharpe_Label": sr_label
        })
    except Exception as e:
        print(f"⚠️ {model_name} 的 MSRP 計算未收斂。")

# 建立 DataFrame 方便 Plotly 讀取
df_metrics_MVO = pd.DataFrame(performance_metrics)

print("⏳ 正在生成多模型綜合比較圖表...")

# ==========================================
# 視覺化：使用 Plotly 建立綜合散佈圖
# ==========================================
fig = go.Figure()

# 定義不同模型的專屬顏色，方便對比
color_map = {
    "01_Answer_Actual": "black",       # 正解用黑色強調
    "02_Sample": "red",                # 傳統樣本用紅色 (通常風險最高/雜訊最多)
    "03_EWMA": "orange",               # 移動平均用橘色
    "04_LedoitWolf_Identity": "green", # 單位矩陣壓縮用綠色
    "05_LedoitWolf_ConstCorr": "blue", # 常數相關壓縮用藍色
    "06_LedoitWolf_Factor": "purple"   # 單因子模型壓縮用紫色
}

# 根據 Portfolio Type 設定形狀 (GMVP 用星星，MSRP 用鑽石)
symbol_map = {
    "GMVP (Min Volatility)": "star",
    "MSRP (Max Sharpe)": "diamond"
}

# 逐一將每個模型的點加入圖表中
for i, row in df_metrics_MVO.iterrows():
    fig.add_trace(go.Scatter(
        x=[row["Risk_Std"]], 
        y=[row["Return"]],
        mode='markers',
        name=f"{row['Model']} - {row['Portfolio Type'].split(' ')[0]}", # 圖例名稱
        legendgroup=row['Model'], # 將同一個模型的 GMVP 和 MSRP 綁在同一個圖例群組
        marker=dict(
            color=color_map.get(row["Model"], "gray"),
            symbol=symbol_map.get(row["Portfolio Type"], "circle"),
            size=18,  # 點放大一點比較清楚
            line=dict(width=1.5, color='white')
        ),
        # 客製化 Hover 面板，動態顯示 Actual 或 Expected
        hovertemplate=(
            f"<b>{row['Model']}</b><br>"
            f"Type: {row['Portfolio Type']}<br>"
            f"---<br>"
            f"{row['Return_Label']}: %{{y:.4f}}<br>"
            f"{row['Risk_Label']}: %{{x:.4f}}<br>"
            f"{row['Sharpe_Label']}: {row['Sharpe_Ratio']:.4f}<extra></extra>"
        )
    ))

# 版面美化與設定
fig.update_layout(
    title=dict(
        text="<b>Cross-Model Portfolio Comparison: Risk vs Return</b><br><sup>Evaluating the Impact of Covariance Shrinkage Targets</sup>", 
        font=dict(size=24), 
        x=0.5
    ),
    xaxis_title="Portfolio Risk / Volatility (Std Dev)",
    yaxis_title="Return (1-Quarter)",
    template="plotly_white",
    width=1100,
    height=750,
    hovermode="closest",
    legend=dict(
        title="<b>Models & Portfolios</b><br>(★: GMVP | ♦: MSRP)",
        groupclick="toggleitem", # 允許點擊圖例開關單一模型
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="lightgray",
        borderwidth=1
    )
)

# 畫一條 Y=0 的輔助線，方便看哪些組合預期報酬為負
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

fig.show()

# 同步印出數據表供查閱
print("\n" + "="*80)
print(" 📊 綜合績效指標數據表 (Performance Metrics) ")
print("="*80)
display(df_metrics_MVO[['Model', 'Portfolio Type', 'Return', 'Risk_Std', 'Sharpe_Ratio']].round(4))

In [ ]:
print("⏳ 正在計算各模型與「真實最佳解」的權重偏離度...")

# ==========================================
# 1. 定義基準 (Ground Truth) 與待測模型
# ==========================================
# 將 NaN 補 0 (代表該組合沒有買入該檔股票)
actual_gmvp = df_all_portfolios['01_Answer_Actual_GMVP'].fillna(0)
actual_msrp = df_all_portfolios['01_Answer_Actual_MSRP'].fillna(0)

models_to_evaluate = [
    ("02_Sample", "Sample (傳統樣本)"),
    ("03_EWMA", "EWMA (指數移動平均)"),
    ("04_LedoitWolf_Identity", "LW-Identity (縮放單位)"),
    ("05_LedoitWolf_ConstCorr", "LW-ConstCorr (常數相關)"),
    ("06_LedoitWolf_Factor", "LW-Factor (單因子)")
]

# ==========================================
# 2. 計算權重偏離度與排名
# ==========================================
evaluation_results = []

for prefix, display_name in models_to_evaluate:
    pred_gmvp = df_all_portfolios[f'{prefix}_GMVP'].fillna(0)
    pred_msrp = df_all_portfolios[f'{prefix}_MSRP'].fillna(0)
    
    # 計算權重偏離度 (Total Misallocation Percentage)
    # 公式： sum(abs(預測 - 真實)) / 2
    gmvp_deviation = np.sum(np.abs(pred_gmvp - actual_gmvp)) / 2
    msrp_deviation = np.sum(np.abs(pred_msrp - actual_msrp)) / 2
    
    evaluation_results.append({
        "Model": display_name,
        "GMVP_Deviation": gmvp_deviation,
        "MSRP_Deviation": msrp_deviation,
        "Total_Deviation": gmvp_deviation + msrp_deviation # 用於計算總排名
    })

# 轉換為 DataFrame 並計算排名
df_eval = pd.DataFrame(evaluation_results)
df_eval['GMVP Rank'] = df_eval['GMVP_Deviation'].rank(method='min').astype(int)
df_eval['MSRP Rank'] = df_eval['MSRP_Deviation'].rank(method='min').astype(int)
df_eval['Overall Rank'] = df_eval['Total_Deviation'].rank(method='min').astype(int)

# 依據整體表現排序
df_eval = df_eval.sort_values('Overall Rank')

# ==========================================
# 3. Plotly 視覺化：模型偏離度長條圖
# ==========================================
fig = go.Figure()

# 加入 GMVP 誤差長條
fig.add_trace(go.Bar(
    x=df_eval['Model'],
    y=df_eval['GMVP_Deviation'],
    name='GMVP (最小風險) 偏離度',
    marker_color='LimeGreen',
    text=df_eval['GMVP_Deviation'].apply(lambda x: f"{x:.1%}"),
    textposition='auto'
))

# 加入 MSRP 誤差長條
fig.add_trace(go.Bar(
    x=df_eval['Model'],
    y=df_eval['MSRP_Deviation'],
    name='MSRP (最大夏普) 偏離度',
    marker_color='Gold',
    text=df_eval['MSRP_Deviation'].apply(lambda x: f"{x:.1%}"),
    textposition='auto'
))

# 版面設定
fig.update_layout(
    title=dict(
        text="<b>模型權重配置誤差比較 (Weight Deviation from Actual)</b><br><sup>數值越低代表越接近真實最佳解（單位：% 資金錯置率）</sup>", 
        font=dict(size=20),
        x=0.5
    ),
    xaxis_title="共變異數矩陣模型",
    yaxis_title="資金錯置率 (%)",
    barmode='group',
    template='plotly_white',
    height=550,
    width=1000,
    yaxis=dict(tickformat=".0%") # Y 軸顯示百分比
)

fig.show()

# ==========================================
# 4. 印出統整報表
# ==========================================
print("\n" + "="*80)
print(" 🏆 共變異數矩陣模型表現排名報表 (相對於 Actual 最佳解) ")
print("="*80)
# 格式化輸出
df_styled = df_eval[['Overall Rank', 'Model', 'GMVP Rank', 'GMVP_Deviation', 'MSRP Rank', 'MSRP_Deviation']].copy()
display(df_styled.style.format({
    'GMVP_Deviation': "{:.2%}",
    'MSRP_Deviation': "{:.2%}"
}).hide(axis="index"))
print("="*80)

In [ ]:
df_all_portfolios

In [ ]:
# 1. 確保 df_all_portfolios 的 Index (Stock_ID) 轉成乾淨的字串型態
df_weights_copy = df_all_portfolios.copy().reset_index()
df_weights_copy['Stock_ID'] = df_weights_copy['Stock_ID'].astype(str).str.strip()

# 2. 建立一個乾淨的 numeric_df 副本，並將對齊欄位與名稱欄位轉為字串
df_info = numeric_df[['標的碼', '標的名稱']].copy()
df_info['標的碼'] = df_info['標的碼'].astype(str).str.strip()
df_info['標的名稱'] = df_info['標的名稱'].astype(str).str.strip()

# 💡 為了避免 numeric_df 裡有重複的股票代碼導致合併後資料列膨脹，先做 drop_duplicates
df_info = df_info.drop_duplicates(subset=['標的碼'])

# 3. 執行 Left Merge (以權重表為主體，串接標的名稱)
df_all_portfolios_merged = pd.merge(
    df_weights_copy,
    df_info,
    left_on='Stock_ID',
    right_on='標的碼',
    how='left'
)

# 4. 調整欄位順序，把「標的名稱」拉到「Stock_ID」旁邊，並將 Stock_ID 設回 Index
# 先移除多餘的 '標的碼' 欄位
df_all_portfolios_merged = df_all_portfolios_merged.drop(columns=['標的碼'])

# 取得除了 'Stock_ID' 和 '標的名稱' 以外的所有其他欄位名稱
other_cols = [col for col in df_all_portfolios_merged.columns if col not in ['Stock_ID', '標的名稱']]

# 重新排列欄位順序：讓 Stock_ID 和 標的名稱 排在最前面
df_all_portfolios_merged = df_all_portfolios_merged[['Stock_ID', '標的名稱'] + other_cols]

# 最後將 Stock_ID 設為 Index（設定後 Stock_ID 會變成 index，而標的名稱就會緊鄰在 index 旁邊的第一欄）
df_all_portfolios_merged = df_all_portfolios_merged.set_index('Stock_ID')

In [ ]:
df_all_portfolios_merged

In [ ]:
df_all_portfolios_merged.to_csv(unconstrained_and_answer_weights_path, encoding='utf-8-sig', index=True)

In [ ]:
import plotly.graph_objects as go
import numpy as np

# ==========================================
# 0. 前置資料準備與過濾 (只保留有分配權重的股票)
# ==========================================
# 複製一份資料避免動到原始 DataFrame
df_plot = df_all_portfolios_merged.copy().fillna(0)

# 找出所有的權重欄位 (排除 '標的名稱')
weight_cols = [col for col in df_plot.columns if col != '標的名稱']

# 只要有任何一個模型對該股票的分配大於 0.01% (0.0001)，就保留
active_mask = (df_plot[weight_cols] > 0.0001).any(axis=1)
df_active = df_plot[active_mask]

# 準備 X 軸的顯示文字： 股票代號 + 名稱 (例如 "6669_緯穎")
x_labels = df_active.index + "_" + df_active['標的名稱']

# 定義統一的模型顏色 (讓 Answer 擁有最高對比度)
color_map = {
    "Answer": "rgb(20, 20, 20)",      # 正解：深黑色（最顯眼）
    "Sample": "rgb(230, 75, 75)",      # 傳統樣本：紅色
    "EWMA": "rgb(245, 150, 60)",       # EWMA：橘色
    "Identity": "rgb(60, 180, 115)",   # LW-Identity：綠色
    "ConstCorr": "rgb(65, 115, 215)",  # LW-ConstCorr：藍色
    "Factor": "rgb(155, 90, 205)"      # LW-Factor：紫色
}

# 輔助函數：用來自動識別欄位屬於哪種模型並對應顏色
def get_trace_color(col_name):
    for key, color in color_map.items():
        if key in col_name:
            return color
    return "gray"

# ==========================================
# 圖 1：僅顯示 MSRP (最大夏普組合)
# ==========================================
fig_msrp = go.Figure()
msrp_cols = [c for c in weight_cols if "MSRP" in c]

# 調整排序，讓 Answer 排在第一個顯示
msrp_cols = sorted(msrp_cols, key=lambda x: 0 if "Answer" in x else 1)

for col in msrp_cols:
    fig_msrp.add_trace(go.Bar(
        x=x_labels,
        y=df_active[col],
        name=col.replace("01_", "").replace("02_", "").replace("03_", "").replace("04_", "").replace("05_", "").replace("06_", ""),
        marker_color=get_trace_color(col),
        hovertemplate="<b>%{x}</b><br>MSRP 權重: %{y:.2%}<extra></extra>"
    ))

fig_msrp.update_layout(
    title=dict(text="<b>Portfolio Weights Comparison: MSRP Only (Max Sharpe)</b>", font=dict(size=20), x=0.5),
    xaxis_title="標的 (代碼_名稱)", yaxis_title="配置權重 (%)", yaxis=dict(tickformat=".0%"),
    barmode='group', template='plotly_white', width=1300, height=600
)
fig_msrp.show()


# ==========================================
# 圖 2：僅顯示 GMVP (全球最小變異數組合)
# ==========================================
fig_gmvp = go.Figure()
gmvp_cols = [c for c in weight_cols if "GMVP" in c]

# 調整排序，讓 Answer 排在第一個
gmvp_cols = sorted(gmvp_cols, key=lambda x: 0 if "Answer" in x else 1)

for col in gmvp_cols:
    fig_gmvp.add_trace(go.Bar(
        x=x_labels,
        y=df_active[col],
        name=col.replace("01_", "").replace("02_", "").replace("03_", "").replace("04_", "").replace("05_", "").replace("06_", ""),
        marker_color=get_trace_color(col),
        hovertemplate="<b>%{x}</b><br>GMVP 權重: %{y:.2%}<extra></extra>"
    ))

fig_gmvp.update_layout(
    title=dict(text="<b>Portfolio Weights Comparison: GMVP Only (Min Volatility)</b>", font=dict(size=20), x=0.5),
    xaxis_title="標的 (代碼_名稱)", yaxis_title="配置權重 (%)", yaxis=dict(tickformat=".0%"),
    barmode='group', template='plotly_white', width=1300, height=600
)
fig_gmvp.show()


# ==========================================
# 圖 3：全部顯示 (MSRP + GMVP 12個欄位全開) - 修正版
# ==========================================
fig_all = go.Figure()

# 調整排序：Answer 放最前，並讓 MSRP 與 GMVP 分組或交錯顯示
sorted_all_cols = sorted(weight_cols, key=lambda x: (0 if "Answer" in x else 1, "MSRP" in x))

for col in sorted_all_cols:
    # 為了在 12 根柱子中區分 MSRP 和 GMVP，我們讓 GMVP 帶有網格線紋理 (pattern)
    is_gmvp = "GMVP" in col
    
    fig_all.add_trace(go.Bar(
        x=x_labels,
        y=df_active[col],
        name=col.replace("01_", "").replace("02_", "").replace("03_", "").replace("04_", "").replace("05_", "").replace("06_", ""),
        marker_color=get_trace_color(col),
        marker_pattern_shape="+" if is_gmvp else "", # GMVP 加上十字紋路做視覺區隔
        hovertemplate=f"<b>%{{x}}</b><br>{'GMVP' if is_gmvp else 'MSRP'} 權重: %{{y:.2%}}<extra></extra>"
    ))

# 💡 這裡修正了 legend 參數，移除不支援的 ncol，並透過 x/y 坐標優化排版
fig_all.update_layout(
    title=dict(
        text="<b>Master Portfolio Weights Comparison (All 12 Portfolios)</b><br><sup>註：帶有斜線/網格紋路的長條代表 GMVP，實心代表 MSRP</sup>", 
        font=dict(size=20), 
        x=0.5
    ),
    xaxis_title="標的 (代碼_名稱)", 
    yaxis_title="配置權重 (%)", 
    yaxis=dict(tickformat=".0%"),
    barmode='group', 
    template='plotly_white', 
    width=1500, 
    height=750, # 稍微拉高畫布，留空間給底部的圖例
    
    # 修正後的圖例設定
    legend=dict(
        orientation="h",        # 水平排列
        x=0.5,                  # 居中對齊
        xanchor="center",
        y=-0.25,                # 往下擺放，避免蓋到 X 軸股票標籤
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="lightgray",
        borderwidth=1
    )
)

fig_all.show()

print("🎉 修正後的 Master 投資組合權重視覺化圖表已成功生成！圖例報錯已解決。")

In [ ]:
all_data_dic_test['2330']

In [ ]:
all_data_dic['2330']

In [ ]:
print(all_data_dic['2330'].dtypes)

In [ ]:
print(all_data_dic_test['2330'].dtypes)

In [ ]:
print(adj_price_0050_df.dtypes)

In [ ]:
adj_price_0050_df

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

print("⏳ 步驟 1：正在提取測試期間的收盤價並計算每日報酬率...")

start_date = pd.to_datetime(date_dict['Strategy_Start_Date'])
end_date = pd.to_datetime(date_dict['Strategy_End_Date'])

# =====================================================================
# 1. 提取所有權重表中的個股，並計算日報酬率
# =====================================================================
# 取出需要用到的股票代號 (排除標的名稱)
weights_df = df_all_portfolios_merged.drop(columns=['標的名稱'], errors='ignore').fillna(0)
stock_ids = weights_df.index.astype(str).str.strip().tolist()

returns_list = []
for sid in stock_ids:
    if sid in all_data_dic:
        df = all_data_dic[sid].copy()
        df['date'] = pd.to_datetime(df['date'])
        # 篩選時間區間
        mask = (df['date'] >= start_date) & (df['date'] <= end_date)
        df_sub = df[mask].sort_values('date').set_index('date')
        
        # 計算日報酬率 (若無交易日則填 0)
        ret = df_sub['close'].pct_change().fillna(0)
        ret.name = sid
        returns_list.append(ret)

# 合併所有個股的日報酬率 (若有某天某檔股票沒數據，補 0)
df_stock_returns = pd.concat(returns_list, axis=1).fillna(0)

# =====================================================================
# 2. 提取 0050 大盤的日報酬率
# =====================================================================
df_0050 = adj_price_0050_df.copy()
df_0050['date'] = pd.to_datetime(df_0050['date'])
mask_0050 = (df_0050['date'] >= start_date) & (df_0050['date'] <= end_date)
df_0050_sub = df_0050[mask_0050].sort_values('date').set_index('date')

ret_0050 = df_0050_sub['close'].pct_change().fillna(0)
ret_0050.name = 'Benchmark (0050)'

# =====================================================================
# 3. 計算投資組合 (Portfolios) 的日報酬率與累積報酬率
# =====================================================================
print("⏳ 步驟 2：正在計算所有投資組合的績效指標...")

# 確保權重與報酬率矩陣的股票順序完全對齊
common_stocks = list(set(df_stock_returns.columns).intersection(set(weights_df.index)))
df_stock_returns = df_stock_returns[common_stocks]
weights_df = weights_df.loc[common_stocks]

# (A) 模型投組日報酬率 (矩陣內積: 每日報酬率 x 個股權重)
df_port_returns = df_stock_returns.dot(weights_df)

# (B) 等權重投組日報酬率 (Equal Weight)
ew_weight = np.ones(len(common_stocks)) / len(common_stocks)
df_port_returns['Equal Weight'] = df_stock_returns.dot(ew_weight)

# (C) 將 0050 併入比較表
df_all_daily_returns = df_port_returns.join(ret_0050, how='left').fillna(0)

# 計算所有投組的「累積報酬率 (Cumulative Return)」
df_cum_returns = (1 + df_all_daily_returns).cumprod() - 1

# =====================================================================
# 4. 計算核心績效指標 (Metrics)
# =====================================================================
def calc_metrics(returns):
    cum_ret = (1 + returns).prod() - 1
    daily_std = returns.std()
    ann_std = daily_std * np.sqrt(252) # 年化標準差
    
    # Sharpe Ratio (假設無風險利率 = 0)
    sharpe = (returns.mean() / daily_std) * np.sqrt(252) if daily_std != 0 else 0
    
    # Max Drawdown
    cum_roll_max = (1 + returns).cumprod().cummax()
    drawdown = ((1 + returns).cumprod() / cum_roll_max) - 1
    mdd = drawdown.min()
    
    # Calmar Ratio = Cumulative Return / abs(Max Drawdown)
    calmar = cum_ret / abs(mdd) if mdd != 0 else np.nan
    
    return pd.Series({
        'Cumulative Return': cum_ret,
        'Daily Std': daily_std,
        'Annualized Std': ann_std,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': mdd,
        'Calmar Ratio': calmar
    })

# 套用計算並轉置，方便閱讀
df_metrics_MVO = df_all_daily_returns.apply(calc_metrics).T

print("\n" + "="*90)
print(f" 📊 投資組合回測績效總表 ({start_date.date()} - {end_date.date()})")
print("="*90)
# 將報表美化，轉為百分比及小數格式
format_dict = {
    'Cumulative Return': '{:.2%}', 'Daily Std': '{:.2%}', 'Annualized Std': '{:.2%}', 
    'Sharpe Ratio': '{:.2f}', 'Max Drawdown': '{:.2%}', 'Calmar Ratio': '{:.2f}'
}
display(df_metrics_MVO.style.format(format_dict).background_gradient(subset=['Cumulative Return', 'Sharpe Ratio', 'Calmar Ratio'], cmap='Blues'))
print("="*90)

# =====================================================================
# 5. Plotly 視覺化 (3 張走勢圖)
# =====================================================================
print("⏳ 步驟 3：正在繪製 Plotly 累積報酬走勢圖...")

# 定義群組
gmvp_cols = [c for c in weights_df.columns if 'GMVP' in c]
msrp_cols = [c for c in weights_df.columns if 'MSRP' in c]
baselines = ['Equal Weight', 'Benchmark (0050)']

def plot_cumulative_returns(cols_to_plot, title):
    fig = go.Figure()
    
    for col in cols_to_plot:
        # 特別強調 Answer 正解與 Baseline，其他模型用較細的線條
        if col == 'Benchmark (0050)':
            line_style = dict(color='crimson', width=3, dash='dashdot')
        elif col == 'Equal Weight':
            line_style = dict(color='gray', width=2, dash='dot')
        elif 'Answer_Actual' in col:
            line_style = dict(color='black', width=3)
        else:
            line_style = dict(width=1.5) # 其他預測模型
            
        fig.add_trace(go.Scatter(
            x=df_cum_returns.index, 
            y=df_cum_returns[col],
            mode='lines', 
            name=col,
            line=line_style,
            hovertemplate="<b>%{data.name}</b><br>Date: %{x}<br>Cum Return: %{y:.2%}<extra></extra>"
        ))

    fig.update_layout(
        title=dict(text=f"<b>{title}</b>", font=dict(size=20), x=0.5),
        xaxis_title="Date",
        yaxis_title="Cumulative Return",
        yaxis=dict(tickformat=".1%"),
        template="plotly_white",
        width=1300, 
        height=650,
        hovermode="x unified",
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.8)")
    )
    fig.show()

# 繪製三張圖
plot_cumulative_returns(gmvp_cols + baselines, "1. Out-of-Sample Performance: GMVP Models vs Benchmarks")
plot_cumulative_returns(msrp_cols + baselines, "2. Out-of-Sample Performance: MSRP Models vs Benchmarks")
plot_cumulative_returns(df_cum_returns.columns, "3. Out-of-Sample Performance: ALL Portfolios vs Benchmarks")

print("🎉 所有績效計算與繪圖已完成！")

In [ ]:
print("⏳ 正在生成 df_metrics_MVO 績效視覺化圖表...")

# 確保 df_metrics_MVO 的 index 是乾淨的字串
df_plot = df_metrics_MVO.copy()
df_plot.index = df_plot.index.astype(str)

# 為了方便上色，幫每個投資組合打上「類別標籤」
def categorize_portfolio(name):
    if '0050' in name or 'Equal Weight' in name:
        return 'Baseline'
    elif 'GMVP' in name:
        return 'GMVP (Min Volatility)'
    elif 'MSRP' in name:
        return 'MSRP (Max Sharpe)'
    return 'Other'

df_plot['Category'] = df_plot.index.map(categorize_portfolio)

# 定義顏色與形狀
color_map = {
    'Baseline': 'crimson',
    'GMVP (Min Volatility)': '#2ca02c',  # 綠色系
    'MSRP (Max Sharpe)': '#ff7f0e'       # 橘色系
}
symbol_map = {
    'Baseline': 'square',
    'GMVP (Min Volatility)': 'star',
    'MSRP (Max Sharpe)': 'diamond'
}

# =====================================================================
# 圖 1：Out-of-Sample 風險 vs. 報酬 散佈圖 (Risk-Return Tradeoff)
# =====================================================================
fig_scatter = go.Figure()

for category in ['Baseline', 'GMVP (Min Volatility)', 'MSRP (Max Sharpe)']:
    df_sub = df_plot[df_plot['Category'] == category]
    
    fig_scatter.add_trace(go.Scatter(
        # X 軸放 Max Drawdown (轉為正數，越往右代表回撤越大/風險越高)
        x=df_sub['Max Drawdown'].abs(), 
        y=df_sub['Cumulative Return'],
        mode='markers+text',
        name=category,
        text=df_sub.index.str.replace("01_|02_|03_|04_|05_|06_", "", regex=True), # 簡化文字標籤
        textposition="top center",
        marker=dict(
            size=14,
            color=color_map[category],
            symbol=symbol_map[category],
            line=dict(width=1, color='DarkSlateGrey')
        ),
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Cumulative Return: %{y:.2%}<br>"
            "Max Drawdown: -%{x:.2%}<br>"
            "Sharpe Ratio: %{customdata[0]:.2f}<br>"
            "Calmar Ratio: %{customdata[1]:.2f}<extra></extra>"
        ),
        customdata=df_sub[['Sharpe Ratio', 'Calmar Ratio']]
    ))

fig_scatter.update_layout(
    title=dict(text="<b>Out-of-Sample Risk vs. Return (Realized Performance)</b><br><sup>越往左上角越好：高報酬、低回撤</sup>", font=dict(size=20), x=0.5),
    xaxis_title="Realized Maximum Drawdown (Risk)",
    yaxis_title="Realized Cumulative Return",
    xaxis=dict(tickformat=".1%", autorange="reversed"), # 反轉 X 軸，讓「風險低(左)」到「風險高(右)」符合視覺直覺
    yaxis=dict(tickformat=".1%"),
    template="plotly_white", width=1200, height=700,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.8)")
)

# 畫一條 Y=0 輔助線
fig_scatter.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

fig_scatter.show()

# =====================================================================
# 圖 2：四大核心績效指標排行榜 (Subplots Bar Charts)
# =====================================================================
# 設定 2x2 子圖畫布
fig_bars = make_subplots(
    rows=2, cols=2, 
    subplot_titles=(
        "1. Cumulative Return (Higher is Better)", 
        "2. Max Drawdown (Closer to 0 is Better)", 
        "3. Sharpe Ratio (Higher is Better)", 
        "4. Calmar Ratio (Higher is Better)"
    ),
    horizontal_spacing=0.15,
    vertical_spacing=0.25
)

metrics_to_plot = [
    ('Cumulative Return', 1, 1, '.1%'),
    ('Max Drawdown', 1, 2, '.1%'),
    ('Sharpe Ratio', 2, 1, '.2f'),
    ('Calmar Ratio', 2, 2, '.2f')
]

# 為了讓長條圖顏色對應模型，建立一個顏色陣列
bar_colors = df_plot['Category'].map(color_map).tolist()

for metric, row, col, tickfmt in metrics_to_plot:
    # 針對該指標排序 (全部由大到小排)
    # 注意：Max Drawdown 是負數，由大到小排剛好會把最接近 0 的（最好）排在前面
    df_sorted = df_plot.sort_values(by=metric, ascending=False)
    sorted_colors = df_sorted['Category'].map(color_map).tolist()
    
    # 清理標籤名稱讓圖表乾淨一點
    clean_labels = df_sorted.index.str.replace("01_|02_|03_|04_|05_|06_", "", regex=True)
    
    fig_bars.add_trace(go.Bar(
        x=df_sorted[metric],
        y=clean_labels,
        orientation='h', # 橫向長條圖比較好讀字
        marker_color=sorted_colors,
        hovertemplate="<b>%{y}</b><br>Score: %{x}<extra></extra>"
    ), row=row, col=col)
    
    fig_bars.update_xaxes(tickformat=tickfmt, row=row, col=col)
    # 讓 Y 軸反轉，這樣第一名會在最上面
    fig_bars.update_yaxes(autorange="reversed", row=row, col=col)

fig_bars.update_layout(
    title=dict(text="<b>Performance Metrics Leaderboards</b><br><sup>綠色: GMVP | 橘色: MSRP | 紅色: Baselines</sup>", font=dict(size=20), x=0.5),
    showlegend=False, # 隱藏圖例，因為顏色已經說明一切
    template="plotly_white", width=1400, height=900,
    margin=dict(l=200) # 增加左側留白，避免投資組合名稱被切掉
)

fig_bars.show()
print("🎉 兩張深度績效分析圖表已成功生成！")

## 2.2. Stress Testing with Monte Carlo

In [ ]:
print("⏳ 步驟 1：正在從 all_data_dic 擷取 " + date_dict['Announcement_Date'] + " 以前的歷史基期資料...")

# =====================================================================
# 1. 重新建構歷史報酬率矩陣 (確保無未來函數，純看歷史參數)
# =====================================================================
target_date = pd.to_datetime(date_dict['Announcement_Date'])
hist_start_date = target_date - pd.Timedelta(days=365)

def clean_ticker(x):
    s = str(x).strip()
    if s.endswith('.0'): 
        s = s[:-2]
    return s

weights_df = df_all_portfolios_merged.drop(columns=['標的名稱'], errors='ignore').fillna(0)
weights_df.index = [clean_ticker(i) for i in weights_df.index]

hist_returns_list = []
for stock_id, df in all_data_dic.items():
    s_id = clean_ticker(stock_id)
    if s_id in weights_df.index:
        temp_df = df.copy()
        temp_df['date'] = pd.to_datetime(temp_df['date'])
        temp_df = temp_df.sort_values('date').reset_index(drop=True)
        
        if 'daily_return' not in temp_df.columns:
            temp_df['close'] = pd.to_numeric(temp_df['close'], errors='coerce')
            temp_df['daily_return'] = temp_df['close'].pct_change(fill_method=None)
            
        mask = (temp_df['date'] >= hist_start_date) & (temp_df['date'] < target_date)
        subset = temp_df[mask][['date', 'daily_return']].copy()
        subset['stock_id'] = s_id
        
        if not subset.empty:
            hist_returns_list.append(subset)

df_hist_all = pd.concat(hist_returns_list, ignore_index=True)
hist_returns_matrix = df_hist_all.pivot(index='date', columns='stock_id', values='daily_return')
hist_returns_matrix = hist_returns_matrix.replace([np.inf, -np.inf], np.nan).fillna(0.0)

# 處理 0050 歷史資料
df_0050_all = adj_price_0050_df.copy()
df_0050_all['date'] = pd.to_datetime(df_0050_all['date'])
actual_hist_start = hist_returns_matrix.index.min()
actual_hist_end = hist_returns_matrix.index.max()

mask_0050_hist = (df_0050_all['date'] >= actual_hist_start) & (df_0050_all['date'] <= actual_hist_end)
df_0050_hist = df_0050_all[mask_0050_hist].sort_values('date').set_index('date')
df_0050_hist['daily_return'] = df_0050_hist['close'].pct_change(fill_method=None).fillna(0)

# =====================================================================
# 2. 計算所有 14 個投資組合在歷史基期下的每日報酬率
# =====================================================================
print("⏳ 步驟 2：正在矩陣內積計算 14 個投資組合的歷史軌跡...")

common_stocks = list(set(hist_returns_matrix.columns).intersection(set(weights_df.index)))
hist_returns_matrix = hist_returns_matrix[common_stocks]
weights_df = weights_df.loc[common_stocks]

# 直接利用矩陣內積算出 12 個模型的日報酬率
hist_portfolio_returns = hist_returns_matrix.dot(weights_df)

# 加入基準指標
hist_portfolio_returns['Benchmark (Equal Weight)'] = hist_returns_matrix.mean(axis=1)
hist_portfolio_returns = hist_portfolio_returns.join(df_0050_hist['daily_return'].rename('Benchmark (0050)'), how='left')
hist_portfolio_returns['Benchmark (0050)'] = hist_portfolio_returns['Benchmark (0050)'].fillna(0)

# 簡化欄位名稱，避免圖表標籤過長
rename_dict = {}
for col in hist_portfolio_returns.columns:
    new_name = col.replace("01_Answer_Actual_", "Answer_")\
                  .replace("02_Sample_", "Sample_")\
                  .replace("03_EWMA_", "EWMA_")\
                  .replace("04_LedoitWolf_Identity_", "LW_Id_")\
                  .replace("05_LedoitWolf_ConstCorr_", "LW_CC_")\
                  .replace("06_LedoitWolf_Factor_", "LW_Fac_")
    rename_dict[col] = new_name
hist_portfolio_returns = hist_portfolio_returns.rename(columns=rename_dict)

# =====================================================================
# 3. 蒙地卡羅模擬 (10萬次平行宇宙預測)
# =====================================================================
N_SIMULATIONS = 100000 
HORIZON = 63               
CONFIDENCE_LEVEL = 0.95    

np.random.seed(42)         
stress_summary = {}

print(f"⏳ 步驟 3：正在運行 100,000 次蒙地卡羅隨機漫步壓力測試 (14 個組合)...")

for col in hist_portfolio_returns.columns:
    mu = hist_portfolio_returns[col].mean()
    sigma = hist_portfolio_returns[col].std()
    
    sim_daily_returns = np.random.normal(mu, sigma, (HORIZON, N_SIMULATIONS))
    terminal_returns = np.prod(1 + sim_daily_returns, axis=0) - 1
    
    median_outcome = np.median(terminal_returns)
    var_95 = np.percentile(terminal_returns, (1 - CONFIDENCE_LEVEL) * 100)
    cvar_95 = terminal_returns[terminal_returns <= var_95].mean()
    
    stress_summary[col] = {
        'Median Return': median_outcome,
        'Value at Risk (95% VaR)': var_95,
        'Stress Expected Loss (95% CVaR)': cvar_95
    }

# 轉換為 DataFrame
df_stress_test = pd.DataFrame(stress_summary)

# 💡 為了讓圖表更好看，我們依照「極端股災預期損失 (CVaR)」由高到低（抗跌到最慘）排序
df_stress_test = df_stress_test.sort_values(by='Stress Expected Loss (95% CVaR)', axis=1, ascending=False)

print("\n" + "="*90)
print(" 🌋 全模型蒙地卡羅壓力測試報告 (依抗跌能力 CVaR 排序)")
print("="*90)
display(df_stress_test.style.format("{:.2%}").background_gradient(cmap='RdYlGn', axis=1))
print("="*90)

# =====================================================================
# 4. Plotly 風險圖表視覺化
# =====================================================================
print("⏳ 步驟 4：正在生成 Plotly 歷史校準尾端風險圖表...")
fig = go.Figure()

# 🔷 中位數預期回報
fig.add_trace(go.Bar(
    x=df_stress_test.columns, y=df_stress_test.loc['Median Return'],
    name='Median Return (Normal State)', marker_color='#1f77b4',
    hovertemplate="<b>%{x}</b><br>Median Return: %{y:.2%}<extra></extra>"
))

# 🔶 95% 在險價值 (VaR)
fig.add_trace(go.Bar(
    x=df_stress_test.columns, y=df_stress_test.loc['Value at Risk (95% VaR)'],
    name='95% VaR (Risk Threshold)', marker_color='#ff7f0e',
    hovertemplate="<b>%{x}</b><br>95% VaR: %{y:.2%}<extra></extra>"
))

# 🟥 95% 條件在險價值 (CVaR 極端股災預期損失)
fig.add_trace(go.Bar(
    x=df_stress_test.columns, y=df_stress_test.loc['Stress Expected Loss (95% CVaR)'],
    name='95% CVaR (Extreme Stress Loss)', marker_color='#d62728',
    hovertemplate="<b>%{x}</b><br>95% CVaR: %{y:.2%}<extra></extra>"
))

fig.update_layout(
    title=dict(
        text="<b>Monte Carlo Stress Testing: 14 Portfolios Comparison</b><br><sup>依抗跌能力 (95% CVaR) 排序：越靠左邊代表股災時損失越小</sup>", 
        font=dict(size=20), x=0.5
    ),
    xaxis_title="Portfolios (Sorted by Risk Resilience)",
    yaxis_title="Simulated Forward Return (%)",
    yaxis=dict(tickformat=".0%", zeroline=True, zerolinecolor='black', zerolinewidth=2),
    barmode='group', 
    template="plotly_white", 
    width=1400, 
    height=750,
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5)
)

fig.show()
print("🎉 真正歷史校準之 14 模型壓力測試圖表已繪製完成！")

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

print("⏳ 步驟 1：正在從 all_data_dic 擷取 " + date_dict['Strategy_Start_Date'] + " 至 " + date_dict['Strategy_End_Date'] + " 的真實測試期資料 (Ex-Post)...")

# =====================================================================
# 1. 擷取真實測試期報酬率矩陣 (用來檢驗體制轉換與真實壓力承受度)
# =====================================================================
start_date = pd.to_datetime(date_dict['Strategy_Start_Date'])
end_date = pd.to_datetime(date_dict['Strategy_End_Date'])

def clean_ticker(x):
    s = str(x).strip()
    if s.endswith('.0'): 
        s = s[:-2]
    return s

# 取得 12 個投資組合的權重
weights_df = df_all_portfolios_merged.drop(columns=['標的名稱'], errors='ignore').fillna(0)
weights_df.index = [clean_ticker(i) for i in weights_df.index]

test_returns_list = []
for stock_id, df in all_data_dic.items():
    s_id = clean_ticker(stock_id)
    if s_id in weights_df.index:
        temp_df = df.copy()
        temp_df['date'] = pd.to_datetime(temp_df['date'])
        temp_df = temp_df.sort_values('date').reset_index(drop=True)
        
        if 'daily_return' not in temp_df.columns:
            temp_df['close'] = pd.to_numeric(temp_df['close'], errors='coerce')
            temp_df['daily_return'] = temp_df['close'].pct_change(fill_method=None)
            
        # 🔴 關鍵修改：篩選未來的真實測試期間
        mask = (temp_df['date'] >= start_date) & (temp_df['date'] <= end_date)
        subset = temp_df[mask][['date', 'daily_return']].copy()
        subset['stock_id'] = s_id
        
        if not subset.empty:
            test_returns_list.append(subset)

df_test_all = pd.concat(test_returns_list, ignore_index=True)
test_returns_matrix = df_test_all.pivot(index='date', columns='stock_id', values='daily_return')
test_returns_matrix = test_returns_matrix.replace([np.inf, -np.inf], np.nan).fillna(0.0)

# 處理 0050 真實測試期資料
df_0050_all = adj_price_0050_df.copy()
df_0050_all['date'] = pd.to_datetime(df_0050_all['date'])
actual_test_start = test_returns_matrix.index.min()
actual_test_end = test_returns_matrix.index.max()

mask_0050_test = (df_0050_all['date'] >= actual_test_start) & (df_0050_all['date'] <= actual_test_end)
df_0050_test = df_0050_all[mask_0050_test].sort_values('date').set_index('date')
df_0050_test['daily_return'] = df_0050_test['close'].pct_change(fill_method=None).fillna(0)

# =====================================================================
# 2. 計算所有 14 個投資組合在真實測試期下的每日報酬率
# =====================================================================
print("⏳ 步驟 2：正在利用矩陣內積計算 14 個投資組合的事後真實軌跡...")

common_stocks = list(set(test_returns_matrix.columns).intersection(set(weights_df.index)))
test_returns_matrix = test_returns_matrix[common_stocks]
weights_df = weights_df.loc[common_stocks]

# 直接利用矩陣內積算出 12 個模型的日報酬率
test_portfolio_returns = test_returns_matrix.dot(weights_df)

# 加入基準指標 (Equal Weight 與 0050)
test_portfolio_returns['Benchmark (Equal Weight)'] = test_returns_matrix.mean(axis=1)
test_portfolio_returns = test_portfolio_returns.join(df_0050_test['daily_return'].rename('Benchmark (0050)'), how='left')
test_portfolio_returns['Benchmark (0050)'] = test_portfolio_returns['Benchmark (0050)'].fillna(0)

# 簡化欄位名稱，避免繪圖時 X 軸標籤過長擠成一團
rename_dict = {}
for col in test_portfolio_returns.columns:
    new_name = col.replace("01_Answer_Actual_", "Answer_")\
                  .replace("02_Sample_", "Sample_")\
                  .replace("03_EWMA_", "EWMA_")\
                  .replace("04_LedoitWolf_Identity_", "LW_Id_")\
                  .replace("05_LedoitWolf_ConstCorr_", "LW_CC_")\
                  .replace("06_LedoitWolf_Factor_", "LW_Fac_")
    rename_dict[col] = new_name
test_portfolio_returns = test_portfolio_returns.rename(columns=rename_dict)

# =====================================================================
# 3. 蒙地卡羅模擬 (10萬次平行宇宙預測，參數來自真實未來)
# =====================================================================
N_SIMULATIONS = 100000 
HORIZON = 63               
CONFIDENCE_LEVEL = 0.95    

np.random.seed(42)         
stress_summary = {}

print(f"⏳ 步驟 3：正在運行 100,000 次蒙地卡羅隨機漫步壓力測試 (14 個組合並行計算中)...")

for col in test_portfolio_returns.columns:
    # 🔴 這裡萃取的 mu 和 sigma，是這三個月真實發生的數據
    mu = test_portfolio_returns[col].mean()
    sigma = test_portfolio_returns[col].std()
    
    sim_daily_returns = np.random.normal(mu, sigma, (HORIZON, N_SIMULATIONS))
    terminal_returns = np.prod(1 + sim_daily_returns, axis=0) - 1
    
    median_outcome = np.median(terminal_returns)
    var_95 = np.percentile(terminal_returns, (1 - CONFIDENCE_LEVEL) * 100)
    cvar_95 = terminal_returns[terminal_returns <= var_95].mean()
    
    stress_summary[col] = {
        'Median Return': median_outcome,
        'Value at Risk (95% VaR)': var_95,
        'Stress Expected Loss (95% CVaR)': cvar_95
    }

# 轉換為 DataFrame
df_stress_test_ex_post = pd.DataFrame(stress_summary)

# 依照「極端股災預期損失 (CVaR)」由高到低（即最抗跌排到最慘烈）進行排序
df_stress_test_ex_post = df_stress_test_ex_post.sort_values(by='Stress Expected Loss (95% CVaR)', axis=1, ascending=False)

print("\n" + "="*90)
print(f" 🌋 14 模型事後蒙地卡羅壓力測試報告 (Ex-Post: {date_dict['Strategy_Start_Date']}~{date_dict['Strategy_End_Date']} 真實參數)")
print("="*90)
display(df_stress_test_ex_post.style.format("{:.2%}").background_gradient(cmap='RdYlGn', axis=1))
print("="*90)

# =====================================================================
# 4. Plotly 風險圖表視覺化
# =====================================================================
print("⏳ 步驟 4：正在生成 Plotly 事後真實校準尾端風險圖表...")
fig = go.Figure()

# 🔷 中位數預期回報
fig.add_trace(go.Bar(
    x=df_stress_test_ex_post.columns, y=df_stress_test_ex_post.loc['Median Return'],
    name='Median Return (Normal State)', marker_color='#1f77b4',
    hovertemplate="<b>%{x}</b><br>Median Return: %{y:.2%}<extra></extra>"
))

# 🔶 95% 在險價值 (VaR)
fig.add_trace(go.Bar(
    x=df_stress_test_ex_post.columns, y=df_stress_test_ex_post.loc['Value at Risk (95% VaR)'],
    name='95% VaR (Risk Threshold)', marker_color='#ff7f0e',
    hovertemplate="<b>%{x}</b><br>95% VaR: %{y:.2%}<extra></extra>"
))

# 🟥 95% 條件在險價值 (CVaR 極端股災預期損失)
fig.add_trace(go.Bar(
    x=df_stress_test_ex_post.columns, y=df_stress_test_ex_post.loc['Stress Expected Loss (95% CVaR)'],
    name='95% CVaR (Extreme Stress Loss)', marker_color='#d62728',
    hovertemplate="<b>%{x}</b><br>95% CVaR: %{y:.2%}<extra></extra>"
))

fig.update_layout(
    title=dict(
        text="<b>Ex-Post Monte Carlo Tail Risk Analysis (Data: Mar-Jun 2025)</b><br><sup>檢驗體制轉換：以真實樣本外波動度測試防禦力 (依 95% CVaR 排序)</sup>", 
        font=dict(size=20), x=0.5
    ),
    xaxis_title="Portfolios (Sorted by Realized Resilience)",
    yaxis_title="Simulated Forward Return (%)",
    yaxis=dict(tickformat=".0%", zeroline=True, zerolinecolor='black', zerolinewidth=2),
    xaxis=dict(tickangle=-45), 
    barmode='group', 
    template="plotly_white", 
    width=1400, 
    height=750,
    legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5)
)

fig.show()
print("🎉 14 模型事後壓力測試圖表已繪製完成！")

# 3. Using Classical Annealing to Solve Optimized Portfolio

In [ ]:
all_data_dic['2330']

In [ ]:
df_all_portfolios_merged

In [ ]:
import time  # ⏱️ 新增時間模組
import numpy as np
import pandas as pd
from scipy.optimize import dual_annealing
from IPython.display import display

# ==========================================
# 0. 輔助函數與資料對齊 
# ==========================================
def clean_ticker(x):
    s = str(x).strip()
    if s.endswith('.0'): 
        s = s[:-2]
    return s

# 假設 df_mu_clean 與 cov_matrix 系列已經存在於您的環境中
# df_mu_clean = ... 
# cov_matrix_answer = ...

# ==========================================
# 1. 定義模擬退火的目標函數 (Objective Functions)
# ==========================================
def objective_gmvp(w_raw, cov_matrix):
    if np.sum(w_raw) == 0: 
        return 1e9
    
    w = w_raw / np.sum(w_raw)
    port_var = np.dot(w.T, np.dot(cov_matrix, w))
    return np.sqrt(port_var)

def objective_msrp(w_raw, mu_vec, cov_matrix):
    if np.sum(w_raw) == 0: 
        return 1e9
        
    w = w_raw / np.sum(w_raw)
    port_ret = np.dot(w.T, mu_vec)
    port_var = np.dot(w.T, np.dot(cov_matrix, w))
    port_vol = np.sqrt(port_var)
    
    if port_vol == 0: 
        return 1e9
        
    sharpe = port_ret / port_vol
    return -sharpe

# ==========================================
# 2. 核心模擬退火執行函數 (加入計時機制)
# ==========================================
def optimize_with_sa(return_col, cov_matrix, model_name, is_actual=False):
    print(f"\n{'='*60}")
    print(f"🔥 正在使用模擬退火執行: {model_name}...")
    
    cov_df = cov_matrix.copy()
    cov_df.columns = [clean_ticker(c) for c in cov_df.columns]
    cov_df.index = [clean_ticker(i) for i in cov_df.index]
    
    mu_series = df_mu_clean[return_col]
    
    common_stocks = list(set(cov_df.columns).intersection(set(mu_series.index)))
    common_stocks.sort() 
    
    mu = mu_series.loc[common_stocks].values
    cov = cov_df.loc[common_stocks, common_stocks].values
    num_assets = len(common_stocks)
    
    bounds = [(0.0, 1.0) for _ in range(num_assets)]
    sa_kwargs = {"maxiter": 500, "initial_temp": 50000}
    
    # --- ⏱️ 1. 執行 GMVP 優化並計時 ---
    print(f"   -> 計算 {model_name} GMVP 中...")
    start_gmvp = time.perf_counter()  # 記錄開始時間
    res_gmvp = dual_annealing(
        objective_gmvp, 
        bounds=bounds, 
        args=(cov,), 
        **sa_kwargs
    )
    end_gmvp = time.perf_counter()    # 記錄結束時間
    gmvp_time = end_gmvp - start_gmvp # 計算耗時
    w_gmvp = res_gmvp.x / np.sum(res_gmvp.x)
    
    # --- ⏱️ 2. 執行 MSRP 優化並計時 ---
    print(f"   -> 計算 {model_name} MSRP 中...")
    start_msrp = time.perf_counter()  # 記錄開始時間
    res_msrp = dual_annealing(
        objective_msrp, 
        bounds=bounds, 
        args=(mu, cov), 
        **sa_kwargs
    )
    end_msrp = time.perf_counter()    # 記錄結束時間
    msrp_time = end_msrp - start_msrp # 計算耗時
    w_msrp = res_msrp.x / np.sum(res_msrp.x)
    
    df_gmvp = pd.Series(w_gmvp, index=common_stocks, name=f'{model_name}_GMVP')
    df_msrp = pd.Series(w_msrp, index=common_stocks, name=f'{model_name}_MSRP')
    
    weights_df = pd.concat([df_gmvp, df_msrp], axis=1)
    
    # 建立時間紀錄字典
    time_dict = {
        "Model_Name": model_name,
        "GMVP_Time_Sec": gmvp_time,
        "MSRP_Time_Sec": msrp_time
    }
    
    return weights_df, time_dict

# ==========================================
# 3. 定義 6 種測試組合並執行
# ==========================================
configurations = [
    {"name": "01_Answer_Actual", "return_col": "Actual_Return", "cov": cov_matrix_answer, "is_actual": True},
    {"name": "02_Sample",        "return_col": "Predicted_Return", "cov": cov_matrix_sample, "is_actual": False},
    {"name": "03_EWMA",          "return_col": "Predicted_Return", "cov": cov_matrix_ewma,   "is_actual": False},
    {"name": "04_LedoitWolf_Identity", "return_col": "Predicted_Return", "cov": cov_matrix_lw,     "is_actual": False},
    {"name": "05_LedoitWolf_ConstCorr","return_col": "Predicted_Return", "cov": cov_matrix_lw_cc, "is_actual": False},
    {"name": "06_LedoitWolf_Factor",   "return_col": "Predicted_Return", "cov": cov_matrix_lw_sf, "is_actual": False}
]

all_sa_weights = []
sa_time_records = []  # ⏱️ 收集模擬退火時間紀錄的清單

for config in configurations:
    w_df, t_dict = optimize_with_sa(
        return_col=config["return_col"],
        cov_matrix=config["cov"],
        model_name=config["name"],
        is_actual=config["is_actual"]
    )
    all_sa_weights.append(w_df)
    sa_time_records.append(t_dict) # 儲存時間

# ==========================================
# 4. 結果統整與展示
# ==========================================
# 4a. 權重總表
df_sa_all_portfolios = pd.concat(all_sa_weights, axis=1)
df_sa_all_portfolios.index.name = 'Stock_ID'

df_sa_all_portfolios[df_sa_all_portfolios < 0.001] = 0.0
active_mask = (df_sa_all_portfolios > 0).any(axis=1)
df_active_sa_all = df_sa_all_portfolios[active_mask].fillna(0)

print("\n" + "="*80)
print(" 🏆 模擬退火 (SA) 全模型投資組合權重總表 (Active Assets Only) ")
print("="*80)
display(df_active_sa_all.style.format("{:.2%}").background_gradient(cmap='Oranges', axis=1))

# 4b. ⏱️ 建立耗時統計 DataFrame: df_time_report_sa
df_time_report_sa = pd.DataFrame(sa_time_records)
df_time_report_sa = df_time_report_sa.set_index("Model_Name")
# 計算總耗時
df_time_report_sa['Total_Optimization_Time_Sec'] = df_time_report_sa.sum(axis=1, skipna=True)

print("\n" + "="*80)
print(" ⏱️ 模擬退火 (SA) 各模型最佳化求解耗時總表 (秒) ")
print("="*80)
display(df_time_report_sa.style.format("{:.4f}秒").background_gradient(cmap='Reds', axis=0))
print("="*80)
print("✅ 模擬退火計算完畢，結果已分別儲存至 `df_sa_all_portfolios` 與 `df_time_report_sa`")

In [ ]:
df_active_sa_all

In [ ]:
df_all_portfolios_merged

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# ==========================================
# 1. 排除非數值欄位 
# ==========================================
# 篩選出真正要比較的數值欄位 (排除 '標的名稱' 等字串欄位)
cols_to_compare = [c for c in df_all_portfolios_merged.columns if c != '標的名稱']

# ==========================================
# 2. 資料對齊 (Alignment)
# ==========================================
# 將 SA 的 DataFrame 重新索引，對齊 50 檔股票的 Index。沒有在 SA 出現的股票，權重補 0
df_sa_aligned = df_active_sa_all.reindex(df_all_portfolios_merged.index).fillna(0)

# 確保兩邊都只取相同的數值欄位進行比較
df_sa_aligned = df_sa_aligned[cols_to_compare]
df_mvo = df_all_portfolios_merged[cols_to_compare].fillna(0)

# ==========================================
# 3. 計算權重差異 (SA 權重 - MVO 傳統權重)
# ==========================================
# 直接計算完整 50 檔股票的差異，不再進行 active_mask 過濾
df_diff = df_sa_aligned - df_mvo

# ==========================================
# 4. 計算誤差統計指標
# ==========================================
mad = df_diff.abs().mean()
max_diff = df_diff.abs().max()

df_summary = pd.DataFrame({
    '平均絕對誤差 (MAD)': mad,
    '最大絕對誤差 (Max Diff)': max_diff
})

# ==========================================
# 5. 輸出視覺化結果
# ==========================================
print("\n" + "="*80)
print(" 📊 模型誤差統計表 (衡量 SA 與 MVO 的整體差距)")
print("="*80)
display(df_summary.style.format("{:.2%}").background_gradient(cmap='Reds'))

print("\n" + "="*80)
print(" 🔍 完整 50 檔個股權重差異表 (SA 權重減去 MVO 權重)")
print("="*80)
print("💡 正數(藍): SA 權重較高 | 負數(紅): MVO 權重較高 | 接近0(白): 兩者一致")
print("="*80)

# 使用 RdBu (紅-白-藍) 漸層色碼，並將中心點對稱設定
# 這裡將 df_diff_active 改為直接對 df_diff 取最大值
max_val = df_diff.abs().max().max()
if pd.isna(max_val) or max_val == 0:
    max_val = 1.0 # 避免全是0的時候報錯

# 直接顯示完整的 df_diff
display(df_diff.style.format("{:.2%}")
        .background_gradient(cmap='RdBu', vmin=-max_val, vmax=max_val))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

def plot_weight_comparison_mpl(target_model, df_mvo, df_sa):
    """
    Plots a comparison of weights between MVO and SA algorithms using Matplotlib.
    Includes subplots showing the absolute differences.
    """
    print(f"📊 Generating weight comparison plot for: {target_model}")
    
    # Define target columns
    col_gmvp = f"{target_model}_GMVP"
    col_msrp = f"{target_model}_MSRP"
    
    if col_gmvp not in df_mvo.columns or col_gmvp not in df_sa.columns:
        print(f"⚠️ Warning: Columns for '{target_model}' not found in the DataFrames.")
        return

    # --- 1. Data Preparation ---
    # Filter GMVP (Keep if either algorithm allocated > 0.1%)
    df_g = pd.DataFrame({'MVO': df_mvo[col_gmvp], 'SA': df_sa[col_gmvp]})
    df_g = df_g[(df_g['MVO'] > 0.001) | (df_g['SA'] > 0.001)].copy()
    df_g['Difference'] = df_g['SA'] - df_g['MVO']
    
    # Filter MSRP (Keep if either algorithm allocated > 0.1%)
    df_m = pd.DataFrame({'MVO': df_mvo[col_msrp], 'SA': df_sa[col_msrp]})
    df_m = df_m[(df_m['MVO'] > 0.001) | (df_m['SA'] > 0.001)].copy()
    df_m['Difference'] = df_m['SA'] - df_m['MVO']

    # --- 2. Create Figure & Subplots (2x2 Grid) ---
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(15, 10))
    fig.suptitle(f"Algorithm Weight Allocation Comparison:\n{target_model}", 
                 fontsize=18, fontweight='bold', y=0.98)

    # Helper function: Plot grouped bar charts (MVO vs SA)
    def plot_grouped_bars(ax, df_data, title):
        x = np.arange(len(df_data))
        width = 0.35
        
        ax.bar(x - width/2, df_data['MVO'], width, label='MVO', color='steelblue')
        ax.bar(x + width/2, df_data['SA'], width, label='SA', color='darkorange')
        
        ax.set_title(title, fontweight='bold', fontsize=14)
        ax.set_ylabel("Allocated Weight")
        ax.set_xticks(x)
        ax.set_xticklabels(df_data.index, rotation=45, ha="right")
        ax.yaxis.set_major_formatter(PercentFormatter(1.0)) # Format Y as %
        ax.legend()
        ax.grid(axis='y', linestyle='--', alpha=0.6)

    # Helper function: Plot difference bar charts (SA - MVO)
    def plot_difference_bars(ax, df_data, title):
        x = np.arange(len(df_data))
        # Green for positive difference, Crimson (Red) for negative difference
        colors = ['forestgreen' if val > 0 else 'crimson' for val in df_data['Difference']]
        
        ax.bar(x, df_data['Difference'], color=colors)
        ax.axhline(0, color='black', linewidth=1) # Baseline at 0
        
        ax.set_title(title, fontweight='bold', fontsize=14)
        ax.set_ylabel("Difference (SA - MVO)")
        ax.set_xticks(x)
        ax.set_xticklabels(df_data.index, rotation=45, ha="right")
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.grid(axis='y', linestyle='--', alpha=0.6)

    # --- 3. Render Subplots ---
    # Column 1: GMVP
    plot_grouped_bars(axes[0, 0], df_g, "GMVP: MVO vs SA")
    plot_difference_bars(axes[1, 0], df_g, "GMVP Difference")
    
    # Column 2: MSRP
    plot_grouped_bars(axes[0, 1], df_m, "MSRP: MVO vs SA")
    plot_difference_bars(axes[1, 1], df_m, "MSRP Difference")

    # Layout adjustment
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# ==========================================
# 執行繪圖測試
# ==========================================
# 呼叫函數，傳入您想查看的模型名稱
plot_weight_comparison_mpl('01_Answer_Actual', df_mvo, df_sa_aligned)

In [ ]:
plot_weight_comparison_mpl('03_EWMA', df_mvo, df_sa_aligned)

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# ==========================================
# 1. Define Ground Truth (Answer/Actual)
# ==========================================
gt_gmvp = df_mvo['01_Answer_Actual_GMVP'].fillna(0)
gt_msrp = df_mvo['01_Answer_Actual_MSRP'].fillna(0)

# ==========================================
# 2. Define Models to Test
# ==========================================
models_to_test = [
    "02_Sample",
    "03_EWMA",
    "04_LedoitWolf_Identity",
    "05_LedoitWolf_ConstCorr",
    "06_LedoitWolf_Factor"
]

results_gmvp = []
results_msrp = []

# ==========================================
# 3. Calculate MAE (Mean Absolute Error)
# ==========================================
for model in models_to_test:
    col_gmvp = f"{model}_GMVP"
    col_msrp = f"{model}_MSRP"
    
    # --- Calculate GMVP Error ---
    if col_gmvp in df_mvo.columns and col_gmvp in df_sa_aligned.columns:
        mvo_gmvp_weights = df_mvo[col_gmvp].fillna(0)
        sa_gmvp_weights = df_sa_aligned[col_gmvp].fillna(0)
        
        err_mvo_gmvp = (mvo_gmvp_weights - gt_gmvp).abs().mean()
        err_sa_gmvp = (sa_gmvp_weights - gt_gmvp).abs().mean()
        
        results_gmvp.append({
            "Model": model,
            "MVO_Error": err_mvo_gmvp,  # Changed to English
            "SA_Error": err_sa_gmvp     # Changed to English
        })

    # --- Calculate MSRP Error ---
    if col_msrp in df_mvo.columns and col_msrp in df_sa_aligned.columns:
        mvo_msrp_weights = df_mvo[col_msrp].fillna(0)
        sa_msrp_weights = df_sa_aligned[col_msrp].fillna(0)
        
        err_mvo_msrp = (mvo_msrp_weights - gt_msrp).abs().mean()
        err_sa_msrp = (sa_msrp_weights - gt_msrp).abs().mean()
        
        results_msrp.append({
            "Model": model,
            "MVO_Error": err_mvo_msrp,  # Changed to English
            "SA_Error": err_sa_msrp     # Changed to English
        })

# Convert to DataFrames and set Index
df_res_gmvp = pd.DataFrame(results_gmvp).set_index("Model")
df_res_msrp = pd.DataFrame(results_msrp).set_index("Model")

# ==========================================
# 4. Find the Best Overall Winner
# ==========================================
def find_winner(df_result):
    min_val = df_result.min().min()
    best_version = df_result.min().idxmin()
    best_model = df_result[best_version].idxmin()
    return best_model, best_version, min_val

winner_g_model, winner_g_ver, min_err_g = find_winner(df_res_gmvp)
winner_m_model, winner_m_ver, min_err_m = find_winner(df_res_msrp)

# ==========================================
# 5. Define Styling Logic for Dark Mode
# ==========================================
def highlight_row_min(row):
    """
    Highlights the lower error within the same row using Yellow Bold Text.
    """
    is_min = row == row.min()
    return ['color: #FFD700; font-weight: bold;' if v else '' for v in is_min]

# Deep Moss Green background for the global best model
champion_style = 'background-color: #1e5631; color: #ffffff;'

# ==========================================
# 6. Output Results
# ==========================================
print("\n" + "="*80)
print(" 🎯 Accuracy Competition: Which model matches [01_Answer_Actual] closest?")
print(" 💡 Reading Guide:")
print("    - Yellow Bold Text: The better performing algorithm (lower error) within THAT model.")
print("    - Dark Green Background: The global champion across ALL models.")
print("="*80)

print(f"\n🏆 [GMVP Overall Winner]: {winner_g_model} ({winner_g_ver})")
print(f"   -> Average Weight Deviation: {min_err_g:.4%}")
display(df_res_gmvp.style.format("{:.4%}")
        .apply(highlight_row_min, axis=1)                  # Row-wise: Highlight lower error in Yellow
        .highlight_min(axis=None, props=champion_style))   # Global: Highlight best cell in Green

print(f"\n🏆 [MSRP Overall Winner]: {winner_m_model} ({winner_m_ver})")
print(f"   -> Average Weight Deviation: {min_err_m:.4%}")
display(df_res_msrp.style.format("{:.4%}")
        .apply(highlight_row_min, axis=1)                  # Row-wise: Highlight lower error in Yellow
        .highlight_min(axis=None, props=champion_style))   # Global: Highlight best cell in Green

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

print("⏳ 步驟 1：正在提取測試期間的收盤價並計算每日報酬率...")

start_date = pd.to_datetime(date_dict['Strategy_Start_Date'])
end_date = pd.to_datetime(date_dict['Strategy_End_Date'])

# =====================================================================
# 1. 提取所有權重表中的個股，並計算日報酬率
# =====================================================================
# ⚠️ 修改點：這裡改為使用 df_sa_aligned (模擬退火版的權重)
weights_df = df_sa_aligned.drop(columns=['標的名稱'], errors='ignore').fillna(0)
stock_ids = weights_df.index.astype(str).str.strip().tolist()

returns_list = []
for sid in stock_ids:
    if sid in all_data_dic:
        df = all_data_dic[sid].copy()
        df['date'] = pd.to_datetime(df['date'])
        # 篩選時間區間
        mask = (df['date'] >= start_date) & (df['date'] <= end_date)
        df_sub = df[mask].sort_values('date').set_index('date')
        
        # 計算日報酬率 (若無交易日則填 0)
        ret = df_sub['close'].pct_change().fillna(0)
        ret.name = sid
        returns_list.append(ret)

# 合併所有個股的日報酬率 (若有某天某檔股票沒數據，補 0)
df_stock_returns = pd.concat(returns_list, axis=1).fillna(0)

# =====================================================================
# 2. 提取 0050 大盤的日報酬率
# =====================================================================
df_0050 = adj_price_0050_df.copy()
df_0050['date'] = pd.to_datetime(df_0050['date'])
mask_0050 = (df_0050['date'] >= start_date) & (df_0050['date'] <= end_date)
df_0050_sub = df_0050[mask_0050].sort_values('date').set_index('date')

ret_0050 = df_0050_sub['close'].pct_change().fillna(0)
ret_0050.name = 'Benchmark (0050)'

# =====================================================================
# 3. 計算投資組合 (Portfolios) 的日報酬率與累積報酬率
# =====================================================================
print("⏳ 步驟 2：正在計算 SA (模擬退火) 投資組合的績效指標...")

# 確保權重與報酬率矩陣的股票順序完全對齊
common_stocks = list(set(df_stock_returns.columns).intersection(set(weights_df.index)))
df_stock_returns = df_stock_returns[common_stocks]
weights_df = weights_df.loc[common_stocks]

# (A) 模型投組日報酬率 (矩陣內積: 每日報酬率 x 個股權重)
df_port_returns = df_stock_returns.dot(weights_df)

# (B) 等權重投組日報酬率 (Equal Weight)
ew_weight = np.ones(len(common_stocks)) / len(common_stocks)
df_port_returns['Equal Weight'] = df_stock_returns.dot(ew_weight)

# (C) 將 0050 併入比較表
df_all_daily_returns = df_port_returns.join(ret_0050, how='left').fillna(0)

# 計算所有投組的「累積報酬率 (Cumulative Return)」
df_cum_returns = (1 + df_all_daily_returns).cumprod() - 1

# =====================================================================
# 4. 計算核心績效指標 (Metrics)
# =====================================================================
def calc_metrics(returns):
    cum_ret = (1 + returns).prod() - 1
    daily_std = returns.std()
    ann_std = daily_std * np.sqrt(252) # 年化標準差
    
    # Sharpe Ratio (假設無風險利率 = 0)
    sharpe = (returns.mean() / daily_std) * np.sqrt(252) if daily_std != 0 else 0
    
    # Max Drawdown
    cum_roll_max = (1 + returns).cumprod().cummax()
    drawdown = ((1 + returns).cumprod() / cum_roll_max) - 1
    mdd = drawdown.min()
    
    # Calmar Ratio = Cumulative Return / abs(Max Drawdown)
    calmar = cum_ret / abs(mdd) if mdd != 0 else np.nan
    
    return pd.Series({
        'Cumulative Return': cum_ret,
        'Daily Std': daily_std,
        'Annualized Std': ann_std,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': mdd,
        'Calmar Ratio': calmar
    })

# 套用計算並轉置，方便閱讀
df_metrics_SA = df_all_daily_returns.apply(calc_metrics).T

print("\n" + "="*90)
print(f" 📊 SA (模擬退火) 投資組合回測績效總表 ({start_date.date()} - {end_date.date()})")
print("="*90)
# 將報表美化，轉為百分比及小數格式
format_dict = {
    'Cumulative Return': '{:.2%}', 'Daily Std': '{:.2%}', 'Annualized Std': '{:.2%}', 
    'Sharpe Ratio': '{:.2f}', 'Max Drawdown': '{:.2%}', 'Calmar Ratio': '{:.2f}'
}
display(df_metrics_SA.style.format(format_dict).background_gradient(subset=['Cumulative Return', 'Sharpe Ratio', 'Calmar Ratio'], cmap='Oranges'))
print("="*90)

# =====================================================================
# 5. Plotly 視覺化 (3 張走勢圖)
# =====================================================================
print("⏳ 步驟 3：正在繪製 Plotly 累積報酬走勢圖...")

# 定義群組
gmvp_cols = [c for c in weights_df.columns if 'GMVP' in c]
msrp_cols = [c for c in weights_df.columns if 'MSRP' in c]
baselines = ['Equal Weight', 'Benchmark (0050)']

def plot_cumulative_returns(cols_to_plot, title):
    fig = go.Figure()
    
    for col in cols_to_plot:
        # 特別強調 Answer 正解與 Baseline，其他模型用較細的線條
        if col == 'Benchmark (0050)':
            line_style = dict(color='crimson', width=3, dash='dashdot')
        elif col == 'Equal Weight':
            line_style = dict(color='gray', width=2, dash='dot')
        elif 'Answer_Actual' in col:
            line_style = dict(color='black', width=3)
        else:
            line_style = dict(width=1.5) # 其他預測模型
            
        fig.add_trace(go.Scatter(
            x=df_cum_returns.index, 
            y=df_cum_returns[col],
            mode='lines', 
            name=col,
            line=line_style,
            hovertemplate="<b>%{data.name}</b><br>Date: %{x}<br>Cum Return: %{y:.2%}<extra></extra>"
        ))

    fig.update_layout(
        title=dict(text=f"<b>{title}</b>", font=dict(size=20), x=0.5),
        xaxis_title="Date",
        yaxis_title="Cumulative Return",
        yaxis=dict(tickformat=".1%"),
        template="plotly_white",
        width=1300, 
        height=650,
        hovermode="x unified",
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.8)")
    )
    fig.show()

# 繪製三張圖 (⚠️ 修改點：標題加上 SA)
plot_cumulative_returns(gmvp_cols + baselines, "1. Out-of-Sample Performance: SA GMVP Models vs Benchmarks")
plot_cumulative_returns(msrp_cols + baselines, "2. Out-of-Sample Performance: SA MSRP Models vs Benchmarks")
plot_cumulative_returns(df_cum_returns.columns, "3. Out-of-Sample Performance: ALL SA Portfolios vs Benchmarks")

print("🎉 所有績效計算與繪圖已完成！")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

print("⏳ 正在計算 MVO 與 SA 的累積報酬率...")

# =====================================================================
# 1. 計算雙方版本的每日報酬與累積報酬
# =====================================================================
# 確保我們使用的是先前的 df_stock_returns, df_mvo, df_sa_aligned
# 計算 MVO 投組日報酬率
df_ret_mvo = df_stock_returns.dot(df_mvo)
# 計算 SA 投組日報酬率
df_ret_sa = df_stock_returns.dot(df_sa_aligned)

# 計算等權重 (Equal Weight) 作為 Benchmark
ew_weight = np.ones(len(df_stock_returns.columns)) / len(df_stock_returns.columns)
ret_ew = df_stock_returns.dot(ew_weight)
ret_ew.name = 'Equal Weight'

# 將所有報酬率合併，方便計算累積報酬
df_all_daily = pd.concat([df_ret_mvo.add_suffix(' (MVO)'), 
                          df_ret_sa.add_suffix(' (SA)'), 
                          ret_ew, 
                          ret_0050], axis=1).fillna(0)

# 計算累積報酬率
df_cum_all = (1 + df_all_daily).cumprod() - 1

# =====================================================================
# 2. 定義要繪製的模型清單
# =====================================================================
models = [
    "01_Answer_Actual",
    "02_Sample",
    "03_EWMA",
    "04_LedoitWolf_Identity",
    "05_LedoitWolf_ConstCorr",
    "06_LedoitWolf_Factor"
]

print("📊 開始繪製比較圖表...")

# =====================================================================
# 3. 迴圈繪製比較圖
# =====================================================================
for model in models:
    # 建立 1x2 的子圖 (左邊 GMVP, 右邊 MSRP)
    fig = make_subplots(
        rows=1, cols=2, 
        subplot_titles=(f"<b>GMVP 走勢比較</b>", f"<b>MSRP 走勢比較</b>"),
        horizontal_spacing=0.08
    )
    
    col_gmvp = f"{model}_GMVP"
    col_msrp = f"{model}_MSRP"
    
    # 共同的 Benchmark 軌跡設定函數
    def add_benchmarks(col_num):
        # 加入 0050
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all['Benchmark (0050)'],
            mode='lines', name='0050', line=dict(color='crimson', width=2, dash='dashdot'),
            showlegend=(col_num==1) # 只在左圖顯示圖例，避免重複
        ), row=1, col=col_num)
        # 加入 Equal Weight
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all['Equal Weight'],
            mode='lines', name='等權重 (EW)', line=dict(color='gray', width=2, dash='dot'),
            showlegend=(col_num==1)
        ), row=1, col=col_num)

    # --- 左子圖：繪製 GMVP ---
    if f"{col_gmvp} (MVO)" in df_cum_all.columns:
        # MVO 版本 (藍色實線)
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all[f"{col_gmvp} (MVO)"],
            mode='lines', name='解析解 (MVO)', line=dict(color='SteelBlue', width=2.5),
            legendgroup="MVO", showlegend=True
        ), row=1, col=1)
        
        # SA 版本 (橘色虛線，加上虛線更容易與實線重疊時辨識)
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all[f"{col_gmvp} (SA)"],
            mode='lines', name='模擬退火 (SA)', line=dict(color='DarkOrange', width=2.5, dash='dash'),
            legendgroup="SA", showlegend=True
        ), row=1, col=1)
        
        add_benchmarks(1)

    # --- 右子圖：繪製 MSRP ---
    if f"{col_msrp} (MVO)" in df_cum_all.columns:
        # MVO 版本 (藍色實線)
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all[f"{col_msrp} (MVO)"],
            mode='lines', name='解析解 (MVO)', line=dict(color='SteelBlue', width=2.5),
            legendgroup="MVO", showlegend=False # 避免圖例重複
        ), row=1, col=2)
        
        # SA 版本 (橘色虛線)
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all[f"{col_msrp} (SA)"],
            mode='lines', name='模擬退火 (SA)', line=dict(color='DarkOrange', width=2.5, dash='dash'),
            legendgroup="SA", showlegend=False
        ), row=1, col=2)
        
        add_benchmarks(2)

    # --- 圖表版面設定 ---
    fig.update_layout(
        title=dict(text=f"<b>MVO vs SA 績效比較：{model}</b>", font=dict(size=22), x=0.5),
        template="plotly_white",
        width=1400, height=600,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
    )
    
    # 將 Y 軸設為百分比格式
    fig.update_yaxes(tickformat=".1%", title_text="累積報酬率", row=1, col=1)
    fig.update_yaxes(tickformat=".1%", row=1, col=2)

    fig.show()

print("🎉 繪圖完成！")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 1. 設定變數範圍
x = np.linspace(-5, 5, 100)
p = np.linspace(-5, 5, 100)
X, P = np.meshgrid(x, p)

# 2. 定義哈密頓函數 H(p, x) = p^2/2m + 1/2*kx^2
# 假設 m=1, k=1
H = (P**2 / 2) + (X**2 / 2)

# 3. 繪製 3D 圖形
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# 繪製曲面
surf = ax.plot_surface(X, P, H, cmap='viridis', edgecolor='none', alpha=0.8)

# 添加標籤
ax.set_title('Harmonic Oscillator Hamiltonian $H(p, x)$')
ax.set_xlabel('Position (x)')
ax.set_ylabel('Momentum (p)')
ax.set_zlabel('Energy (H)')

# 顯示色階條
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)

plt.show()

# 4. Using D-Wave Quantum Annealing to Solve QUBO Portfolio Optimization

Goal: 雖然量子退火在投組最佳化上要超多標的 很多限制式時才能彰顯它的價值
但這裡只是在試用 Dwave 並看要花多少時間找到unconstrained的最佳解 有哪些流程 找到解的精度

## 4.1. Calculate Optimized Weights

In [ ]:
import time
import numpy as np
import pandas as pd
import neal
from dimod import ConstrainedQuadraticModel, Integer, cqm_to_bqm
from IPython.display import display

In [ ]:
import time
import numpy as np
import pandas as pd
import neal
from dimod import ConstrainedQuadraticModel, Integer, cqm_to_bqm
from IPython.display import display

# ==========================================
# 0. 輔助函數設定 (沿用)
# ==========================================
def clean_ticker(x):
    s = str(x).strip()
    if s.endswith('.0'): 
        s = s[:-2]
    return s

# ==========================================
# 1. 支援動態精確度的量子退火執行函數
# ==========================================
def optimize_with_qa_dynamic(return_col, cov_matrix, model_name, precision, hyperparams):
    """
    動態執行特定精確度的量子退火模擬器
    """
    cov_df = cov_matrix.copy()
    cov_df.columns = [clean_ticker(c) for c in cov_df.columns]
    cov_df.index = [clean_ticker(i) for i in cov_df.index]
    
    mu_series = df_mu_clean[return_col]
    common_stocks = list(set(cov_df.columns).intersection(set(mu_series.index)))
    common_stocks.sort() 
    
    mu = mu_series.loc[common_stocks].values
    cov = cov_df.loc[common_stocks, common_stocks].values
    num_assets = len(common_stocks)
    
    # --- 構建 Constrained Quadratic Model (CQM) ---
    cqm = ConstrainedQuadraticModel()
    x = {i: Integer(f'x_{i}', lower_bound=0, upper_bound=precision) for i in range(num_assets)}
    cqm.add_constraint(sum(x[i] for i in range(num_assets)) == precision, label='budget')
    
    risk_expr = 0
    for i in range(num_assets):
        for j in range(num_assets):
            risk_expr += cov[i, j] * x[i] * x[j]
            
    ret_expr = sum(mu[i] * x[i] for i in range(num_assets))

    sampler = neal.SimulatedAnnealingSampler()

    # --- ⏱️ 1. 計算 GMVP 並精確計時 ---
    cqm.set_objective(risk_expr)
    bqm_gmvp, invert_gmvp = cqm_to_bqm(cqm, lagrange_multiplier=hyperparams["lagrange_multiplier"])
    
    start_gmvp = time.perf_counter()
    res_gmvp = sampler.sample(
        bqm_gmvp, 
        num_reads=hyperparams["num_reads"], 
        num_sweeps=hyperparams["num_sweeps"],
        seed=42
    )
    end_gmvp = time.perf_counter()
    gmvp_time = end_gmvp - start_gmvp
    
    best_gmvp = None
    for sample in res_gmvp.samples():
        decoded = invert_gmvp(sample)
        if sum(decoded[f'x_{i}'] for i in range(num_assets)) == precision:
            best_gmvp = decoded
            break
            
    w_gmvp = np.array([best_gmvp[f'x_{i}'] / precision for i in range(num_assets)]) if best_gmvp else np.zeros(num_assets)

    # --- ⏱️ 2. 計算 MSRP 並精確計時 ---
    lambda_risk = 0.5
    cqm.set_objective(lambda_risk * risk_expr - ret_expr)
    bqm_msrp, invert_msrp = cqm_to_bqm(cqm, lagrange_multiplier=hyperparams["lagrange_multiplier"])
    
    start_msrp = time.perf_counter()
    res_msrp = sampler.sample(
        bqm_msrp, 
        num_reads=hyperparams["num_reads"], 
        num_sweeps=hyperparams["num_sweeps"],
        seed=42
    )
    end_msrp = time.perf_counter()
    msrp_time = end_msrp - start_msrp
    
    best_msrp = None
    for sample in res_msrp.samples():
        decoded = invert_msrp(sample)
        if sum(decoded[f'x_{i}'] for i in range(num_assets)) == precision:
            best_msrp = decoded
            break
            
    w_msrp = np.array([best_msrp[f'x_{i}'] / precision for i in range(num_assets)]) if best_msrp else np.zeros(num_assets)

    # 整理權重
    df_gmvp = pd.Series(w_gmvp, index=common_stocks, name=f'{model_name}_GMVP')
    df_msrp = pd.Series(w_msrp, index=common_stocks, name=f'{model_name}_MSRP')
    weights_df = pd.concat([df_gmvp, df_msrp], axis=1)
    
    return weights_df, gmvp_time + msrp_time

# ==========================================
# 2. 定義 2 種精確度的實驗配置 (已移除 1000)
# ==========================================
precision_experiments = {
    20: {
        "precision": 20,
        "hyperparams": {"lagrange_multiplier": 5.0, "num_reads": 100, "num_sweeps": 2000}
    },
    100: {
        "precision": 100,
        "hyperparams": {"lagrange_multiplier": 15.0, "num_reads": 300, "num_sweeps": 8000}
    }
}

# 6種測試共變異數組合
configurations = [
    {"name": "01_Answer_Actual", "return_col": "Actual_Return", "cov": cov_matrix_answer},
    {"name": "02_Sample",        "return_col": "Predicted_Return", "cov": cov_matrix_sample},
    {"name": "03_EWMA",          "return_col": "Predicted_Return", "cov": cov_matrix_ewma},
    {"name": "04_LedoitWolf_Identity", "return_col": "Predicted_Return", "cov": cov_matrix_lw},
    {"name": "05_LedoitWolf_ConstCorr", "return_col": "Predicted_Return", "cov": cov_matrix_lw_cc},
    {"name": "06_LedoitWolf_Factor", "return_col": "Predicted_Return", "cov": cov_matrix_lw_sf}
]

# 用來儲存時間統計的字典
time_stats = {"Model_Name": [config["name"] for config in configurations]}

# ==========================================
# 3. 核心批次迴圈執行
# ==========================================
for p_value, exp in precision_experiments.items():
    print(f"\n📢 正在啟動規模測試：Precision = {p_value} (每份資金單位: {1/p_value:.2%})")
    print(f"⚙️ 超參數配置: {exp['hyperparams']}")
    
    current_p_weights = []
    current_p_times = []
    
    for config in configurations:
        w_df, total_time = optimize_with_qa_dynamic(
            return_col=config["return_col"],
            cov_matrix=config["cov"],
            model_name=config["name"],
            precision=exp["precision"],
            hyperparams=exp["hyperparams"]
        )
        current_p_weights.append(w_df)
        current_p_times.append(total_time)
        print(f"   ✅ {config['name']} 計算完成，耗時: {total_time:.4f} 秒")
        
    # 將所有模型的權重合併成一張表
    df_p_all = pd.concat(current_p_weights, axis=1)
    df_p_all.index.name = 'Stock_ID'
    
    # 動態指派至您要求的指定變數名稱
    if p_value == 20:
        df_qa20_all_portfolios = df_p_all.copy()
    elif p_value == 100:
        df_qa100_all_portfolios = df_p_all.copy()
        
    # 紀錄時間
    time_stats[f"Precision_{p_value}_Time_Sec"] = current_p_times

# ==========================================
# 4. 統計結果統整與展示
# ==========================================
df_time_comparison_qa = pd.DataFrame(time_stats).set_index("Model_Name")

print("\n" + "="*90)
print(" ⏱️ 量子退火模擬器：不同精確度 (Precision 20 vs 100) 總求解耗時對比表 ")
print("="*90)
display(df_time_comparison_qa.style.format("{:.4f}秒").background_gradient(cmap='Reds', axis=None))
print("="*90)
print("✅ 權重結果已分別儲存至以下 DataFrame：")
print("   - `df_qa20_all_portfolios`   (5.0% 顆粒度)")
print("   - `df_qa100_all_portfolios`  (1.0% 顆粒度)")

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

# ==========================================
# 1. 排除非數值欄位 
# ==========================================
# 篩選出真正要比較的數值欄位 (排除 '標的名稱' 等字串欄位)
cols_to_compare = [c for c in df_all_portfolios_merged.columns if c != '標的名稱']

# ==========================================
# 2. 資料對齊 (Alignment)
# ==========================================
# 確保 MVO 基準對齊並只取數值欄位
df_mvo = df_all_portfolios_merged[cols_to_compare].fillna(0)

# 將 SA、QA20、QA100 的 DataFrame 重新索引，對齊 50 檔股票的 Index。沒有出現的股票補 0
df_sa_aligned    = df_active_sa_all.reindex(df_all_portfolios_merged.index).fillna(0)[cols_to_compare]
df_qa20_aligned  = df_qa20_all_portfolios.reindex(df_all_portfolios_merged.index).fillna(0)[cols_to_compare]  # 🌟 新增 QA20
df_qa100_aligned = df_qa100_all_portfolios.reindex(df_all_portfolios_merged.index).fillna(0)[cols_to_compare] # 🌟 新增 QA100

# ==========================================
# 3. 計算權重差異 (啟發式權重 - MVO 傳統權重)
# ==========================================
# 分別計算 SA, QA20, QA100 相對於傳統 MVO 的差異矩陣
df_diff_sa    = df_sa_aligned - df_mvo
df_diff_qa20  = df_qa20_aligned - df_mvo  # 🌟 新增 QA20 差異
df_diff_qa100 = df_qa100_aligned - df_mvo # 🌟 新增 QA100 差異

# ==========================================
# 4. 計算誤差統計指標 (擴充為四種維度)
# ==========================================
# 統整三者相對於 MVO 的平均絕對誤差 (MAD) 與最大誤差 (Max Diff)
df_summary = pd.DataFrame({
    'SA 平均絕對誤差 (MAD)': df_diff_sa.abs().mean(),
    'SA 最大絕對誤差 (Max Diff)': df_diff_sa.abs().max(),
    'QA20 平均絕對誤差 (MAD)': df_diff_qa20.abs().mean(),    # 🌟 新增 QA20 統計
    'QA20 最大絕對誤差 (Max Diff)': df_diff_qa20.abs().max(), # 🌟 新增 QA20 統計
    'QA100 平均絕對誤差 (MAD)': df_diff_qa100.abs().mean(),  # 🌟 新增 QA100 統計
    'QA100 最大絕對誤差 (Max Diff)': df_diff_qa100.abs().max()# 🌟 新增 QA100 統計
})

# ==========================================
# 5. 輸出視覺化結果
# ==========================================
print("\n" + "="*90)
print(" 📊 模型誤差統計表 (衡量 SA / QA20 / QA100 與 MVO 的整體差距)")
print("="*90)
# 顏色越深代表與傳統 MVO 的解析解權重差異越大
display(df_summary.style.background_gradient(cmap='Reds', axis=0).format("{:.2%}"))

# --- 1. 計算 SA 熱力圖極值與繪圖 ---
max_val_sa = df_diff_sa.abs().max().max()
if pd.isna(max_val_sa) or max_val_sa == 0: max_val_sa = 1.0

print("\n" + "="*80)
print(" 🔍 完整 50 檔個股權重差異表 (SA 權重 - MVO 權重)")
print("="*80)
print("💡 正數(藍): SA 權重較高 | 負數(紅): MVO 權重較高 | 接近0(白): 兩者一致")
display(df_diff_sa.style.format("{:.2%}")
        .background_gradient(cmap='RdBu', vmin=-max_val_sa, vmax=max_val_sa))

# --- 2. 計算 QA20 熱力圖極值與繪圖 ---
max_val_qa20 = df_diff_qa20.abs().max().max()
if pd.isna(max_val_qa20) or max_val_qa20 == 0: max_val_qa20 = 1.0

print("\n" + "="*80)
print(" 🔍 完整 50 檔個股權重差異表 (QA20 權重 - MVO 權重)")
print("="*80)
print("💡 正數(藍): QA20 權重較高 | 負數(紅): MVO 權重較高 | 接近0(白): 兩者一致")
display(df_diff_qa20.style.format("{:.2%}")
        .background_gradient(cmap='RdBu', vmin=-max_val_qa20, vmax=max_val_qa20))

# --- 3. 計算 QA100 熱力圖極值與繪圖 ---
max_val_qa100 = df_diff_qa100.abs().max().max()
if pd.isna(max_val_qa100) or max_val_qa100 == 0: max_val_qa100 = 1.0

print("\n" + "="*80)
print(" 🔍 完整 50 檔個股權重差異表 (QA100 權重 - MVO 權重)")
print("="*80)
print("💡 正數(藍): QA100 權重較高 | 負數(紅): MVO 權重較高 | 接近0(白): 兩者一致")
display(df_diff_qa100.style.format("{:.2%}")
        .background_gradient(cmap='RdBu', vmin=-max_val_qa100, vmax=max_val_qa100))

print("\n" + "="*80)
print("🎉 誤差比對完成！您可以特別對比 QA20 與 QA100 的熱力圖顏色飽和度，通常 QA100 的色彩會明顯變淡（更接近白色），代表與 MVO 的差距正在縮小。")

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

print("⏳ 正在整合 MVO, SA, QA20, QA100 的求解時間指標...")

# =====================================================================
# 1. 資料清洗與抽取 (從各自的 Time Report 中抓取總優化時間)
# =====================================================================
# 建立一個基礎 DataFrame，以 6 種模型配置為 Index
df_time_final = pd.DataFrame(index=df_time_report.index)

# (A) 提取傳統解析解 MVO 的時間 (加總 GMVP 與 MSRP)
df_time_final['MVO'] = df_time_report['Total_Optimization_Time_Sec']

# (B) 提取傳統模擬退火 SA 的時間 (加總 GMVP 與 MSRP)
df_time_final['SA'] = df_time_report_sa['Total_Optimization_Time_Sec']

# (C) 提取量子退火模擬器 Precision = 20 的總時間
df_time_final['QA20'] = df_time_comparison_qa['Precision_20_Time_Sec']

# (D) 提取量子退火模擬器 Precision = 100 的總時間
df_time_final['QA100'] = df_time_comparison_qa['Precision_100_Time_Sec']

# =====================================================================
# 2. 輸出橫向對比總表 (嚴格按照 MVO -> SA -> QA20 -> QA100 順序)
# =====================================================================
print("\n" + "="*90)
print(" 🏆 投資組合最佳化：四大演算法求解耗時橫向對比總表 ")
print(" 💡 閱讀指南：顏色越深 (紅) 代表該演算法運算負荷越重、花費時間越長。")
print("="*90)

# 將欄位依要求順序排列
df_time_final = df_time_final[['MVO', 'SA', 'QA20', 'QA100']]

# 呼叫 Style 美化，顯示到小數點後 4 位
display(df_time_final.style.format("{:.4f}秒").background_gradient(cmap='Reds', axis=None))
print("="*90)

# =====================================================================
# 3. 繪製專屬的耗時對比直條圖 (單獨一張圖)
# =====================================================================
print("\n⏳ 正在繪製演算法運算時間對比直條圖...")

fig = go.Figure()

# 定義演算法與其對應的色彩 (維持視覺色系連貫)
colors = {
    'MVO': 'steelblue', 
    'SA': 'darkorange', 
    'QA20': 'mediumpurple', 
    'QA100': 'rebeccapurple'
}

for algo in ['MVO', 'SA', 'QA20', 'QA100']:
    fig.add_trace(go.Bar(
        x=df_time_final.index,
        y=df_time_final[algo],
        name=algo,
        marker_color=colors[algo],
        # 在柱狀圖上方直接秀出秒數，方便對比
        text=df_time_final[algo].apply(lambda x: f"{x:.3f}s"),
        textposition='outside'
    ))

fig.update_layout(
    title=dict(text="<b>MVO vs SA vs QA20 vs QA100: 演算法求解耗時比較 (秒)</b>", font=dict(size=22), x=0.5),
    xaxis_title="Covariance Model Config",
    yaxis_title="Total Optimization Time (Seconds)",
    template="plotly_white",
    width=1400, height=600,
    barmode='group', # 並排顯示
    legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="right", x=1)
)

fig.show()
print("🎉 時間指標比較與獨立圖表繪製已完成！")

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

# ==========================================
# 1. 排除非數值欄位 
# ==========================================
cols_to_compare = [c for c in df_all_portfolios_merged.columns if c != '標的名稱']

# ==========================================
# 2. 資料對齊 (Alignment)
# ==========================================
# 將 QA20 與 QA100 的 DataFrame 重新索引，對齊 50 檔股票的 Index。沒有出現的股票補 0
df_qa20_aligned  = df_qa20_all_portfolios.reindex(df_all_portfolios_merged.index).fillna(0)[cols_to_compare]
df_qa100_aligned = df_qa100_all_portfolios.reindex(df_all_portfolios_merged.index).fillna(0)[cols_to_compare]

# ==========================================
# 3. 計算權重差異 (QA20 權重 - QA100 權重)
# ==========================================
# 這裡的核心改變：直接讓兩個量子退火模型互相對減，觀察精確度提升帶來的資金流動
df_diff_qa_scale = df_qa20_aligned - df_qa100_aligned

# ==========================================
# 4. 計算誤差統計指標 (衡量兩者間的微調幅度)
# ==========================================
mad_scale = df_diff_qa_scale.abs().mean()
max_diff_scale = df_diff_qa_scale.abs().max()

df_summary_scale = pd.DataFrame({
    'QA20 vs QA100 平均絕對差異 (MAD)': mad_scale,
    'QA20 vs QA100 最大絕對差異 (Max Diff)': max_diff_scale
})

# ==========================================
# 5. 輸出視覺化結果
# ==========================================
print("\n" + "="*90)
print(" 📊 量子退火尺度規模誤差統計表 (衡量 Precision 20 升級至 100 的權重變動)")
print("="*90)
# 顏色越深代表精確度切細後，該組合的權重被微調（重組）的幅度越大
display(df_summary_scale.style.background_gradient(cmap='Purples', axis=0).format("{:.2%}"))

# --- 計算熱力圖極值並繪圖 ---
max_val_scale = df_diff_qa_scale.abs().max().max()
if pd.isna(max_val_scale) or max_val_scale == 0: 
    max_val_scale = 1.0

print("\n" + "="*80)
print(" 🔍 完整 50 檔個股權重差異表 (QA20 權重 減去 QA100 權重)")
print("="*80)
print("💡 正數(藍): 代表在 5% 粗糙顆粒下被『多分配』的權重 | 負數(紅): 代表精確度提高到 1% 後『補回來』的權重")
print("💡 接近0(白): 代表顆粒度變細並不影響該資產的配置，模型看法高度一致")
print("="*80)

# 使用對稱的 RdBu (紅-白-藍) 漸層色碼，完美視覺化資金在個股間移動的軌跡
display(df_diff_qa_scale.style.format("{:.2%}")
        .background_gradient(cmap='RdBu', vmin=-max_val_scale, vmax=max_val_scale))

print("\n" + "="*80)
print("💡 觀察指南：")
print("   - 藍色區塊（正數）：表示這些股票在 `Precision=20` 時，因為每次只能加減 5%，被強迫『向上無條件進位』或過度集中。")
print("   - 紅色區塊（負數）：表示當精確度開放到 1% (`Precision=100`) 後，退火模擬器終於有能力把資金精準地『補加』到這些潛力資產中。")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

def plot_weight_comparison_mpl(target_model, df_mvo, df_sa, df_qa20, df_qa100):
    """
    Plots a comparison of weights between MVO, SA, QA20, and QA100 algorithms using Matplotlib.
    Displays GMVP and MSRP allocations side-by-side with 4-grouped bar charts.
    """
    print(f"📊 Generating 4-algorithm weight comparison plot for: {target_model}")
    
    # 定義要抓取的欄位名稱
    col_gmvp = f"{target_model}_GMVP"
    col_msrp = f"{target_model}_MSRP"
    
    # 防呆檢查：確認欄位存在於四個資料表中
    dfs = [df_mvo, df_sa, df_qa20, df_qa100]
    names = ['MVO', 'SA', 'QA20', 'QA100']
    for df, name in zip(dfs, names):
        if col_gmvp not in df.columns or col_msrp not in df.columns:
            print(f"⚠️ Warning: Columns for '{target_model}' not found in {name} DataFrame.")
            return

    # --- 1. 資料準備與過濾 ---
    # 過濾 GMVP (只要有任何一個演算法分配超過 0.1% 就保留該股票)
    df_g = pd.DataFrame({
        'MVO': df_mvo[col_gmvp], 
        'SA': df_sa[col_gmvp], 
        'QA20': df_qa20[col_gmvp], 
        'QA100': df_qa100[col_gmvp]
    })
    df_g = df_g[(df_g['MVO'] > 0.001) | (df_g['SA'] > 0.001) | (df_g['QA20'] > 0.001) | (df_g['QA100'] > 0.001)].copy()
    
    # 過濾 MSRP (只要有任何一個演算法分配超過 0.1% 就保留該股票)
    df_m = pd.DataFrame({
        'MVO': df_mvo[col_msrp], 
        'SA': df_sa[col_msrp], 
        'QA20': df_qa20[col_msrp], 
        'QA100': df_qa100[col_msrp]
    })
    df_m = df_m[(df_m['MVO'] > 0.001) | (df_m['SA'] > 0.001) | (df_m['QA20'] > 0.001) | (df_m['QA100'] > 0.001)].copy()

    # --- 2. 建立圖表與子圖 (1x2 網格) ---
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(18, 6))
    fig.suptitle(f"Algorithm Weight Allocation Comparison:\n{target_model} (MVO vs SA vs QA20 vs QA100)", 
                 fontsize=18, fontweight='bold', y=1.05)

    # 繪製群組長條圖的輔助函數 (四根柱子)
    def plot_quad_bars(ax, df_data, title):
        x = np.arange(len(df_data))
        width = 0.2  # 調細柱子寬度，完美容納四根柱子
        
        # 依序繪製 MVO, SA, QA20, QA100
        ax.bar(x - 1.5*width, df_data['MVO'], width, label='MVO', color='steelblue')
        ax.bar(x - 0.5*width, df_data['SA'], width, label='SA', color='darkorange')
        ax.bar(x + 0.5*width, df_data['QA20'], width, label='QA20', color='mediumpurple') # 淺紫
        ax.bar(x + 1.5*width, df_data['QA100'], width, label='QA100', color='rebeccapurple') # 深紫
        
        ax.set_title(title, fontweight='bold', fontsize=14)
        ax.set_ylabel("Allocated Weight")
        ax.set_xticks(x)
        ax.set_xticklabels(df_data.index, rotation=45, ha="right")
        ax.yaxis.set_major_formatter(PercentFormatter(1.0)) # 將 Y 軸格式化為百分比
        ax.legend()
        ax.grid(axis='y', linestyle='--', alpha=0.6)

    # --- 3. 渲染子圖 ---
    # 第一張圖: GMVP 比較
    plot_quad_bars(axes[0], df_g, "GMVP: MVO vs SA vs QA20 vs QA100")
    
    # 第二張圖: MSRP 比較
    plot_quad_bars(axes[1], df_m, "MSRP: MVO vs SA vs QA20 vs QA100")

    # 版面微調，避免標籤被切掉
    plt.tight_layout()
    plt.show()

# ==========================================
# 執行繪圖測試
# ==========================================
# 呼叫函數，將 MVO, SA 以及對齊後的 QA20、QA100 齊齊傳入
plot_weight_comparison_mpl('01_Answer_Actual', df_mvo, df_sa_aligned, df_qa20_aligned, df_qa100_aligned)

In [ ]:
plot_weight_comparison_mpl('03_EWMA', df_mvo, df_sa_aligned, df_qa20_aligned, df_qa100_aligned)

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# ==========================================
# 1. Define Ground Truth (Answer/Actual)
# ==========================================
gt_gmvp = df_mvo['01_Answer_Actual_GMVP'].fillna(0)
gt_msrp = df_mvo['01_Answer_Actual_MSRP'].fillna(0)

# ==========================================
# 2. Define Models to Test
# ==========================================
models_to_test = [
    "02_Sample",
    "03_EWMA",
    "04_LedoitWolf_Identity",
    "05_LedoitWolf_ConstCorr",
    "06_LedoitWolf_Factor"
]

results_gmvp = []
results_msrp = []

# ==========================================
# 3. Calculate MAE (Mean Absolute Error)
# ==========================================
for model in models_to_test:
    col_gmvp = f"{model}_GMVP"
    col_msrp = f"{model}_MSRP"
    
    # --- Calculate GMVP Error ---
    # 🌟 檢查條件同步加入 QA20 與 QA100 的對齊表
    if (col_gmvp in df_mvo.columns and 
        col_gmvp in df_sa_aligned.columns and 
        col_gmvp in df_qa20_aligned.columns and 
        col_gmvp in df_qa100_aligned.columns):
        
        mvo_gmvp_weights = df_mvo[col_gmvp].fillna(0)
        sa_gmvp_weights = df_sa_aligned[col_gmvp].fillna(0)
        qa20_gmvp_weights = df_qa20_aligned[col_gmvp].fillna(0)   # 🌟 導入 QA20 權重
        qa100_gmvp_weights = df_qa100_aligned[col_gmvp].fillna(0) # 🌟 導入 QA100 權重
        
        err_mvo_gmvp = (mvo_gmvp_weights - gt_gmvp).abs().mean()
        err_sa_gmvp = (sa_gmvp_weights - gt_gmvp).abs().mean()
        err_qa20_gmvp = (qa20_gmvp_weights - gt_gmvp).abs().mean()   # 🌟 計算 QA20 誤差
        err_qa100_gmvp = (qa100_gmvp_weights - gt_gmvp).abs().mean() # 🌟 計算 QA100 誤差
        
        results_gmvp.append({
            "Model": model,
            "MVO_Error": err_mvo_gmvp,  
            "SA_Error": err_sa_gmvp,    
            "QA20_Error": err_qa20_gmvp,   # 🌟 新增至結果字典
            "QA100_Error": err_qa100_gmvp  # 🌟 新增至結果字典
        })

    # --- Calculate MSRP Error ---
    if (col_msrp in df_mvo.columns and 
        col_msrp in df_sa_aligned.columns and 
        col_msrp in df_qa20_aligned.columns and 
        col_msrp in df_qa100_aligned.columns):
        
        mvo_msrp_weights = df_mvo[col_msrp].fillna(0)
        sa_msrp_weights = df_sa_aligned[col_msrp].fillna(0)
        qa20_msrp_weights = df_qa20_aligned[col_msrp].fillna(0)   # 🌟 導入 QA20 權重
        qa100_msrp_weights = df_qa100_aligned[col_msrp].fillna(0) # 🌟 導入 QA100 權重
        
        err_mvo_msrp = (mvo_msrp_weights - gt_msrp).abs().mean()
        err_sa_msrp = (sa_msrp_weights - gt_msrp).abs().mean()
        err_qa20_msrp = (qa20_msrp_weights - gt_msrp).abs().mean()   # 🌟 計算 QA20 誤差
        err_qa100_msrp = (qa100_msrp_weights - gt_msrp).abs().mean() # 🌟 計算 QA100 誤差
        
        results_msrp.append({
            "Model": model,
            "MVO_Error": err_mvo_msrp,  
            "SA_Error": err_sa_msrp,    
            "QA20_Error": err_qa20_msrp,   # 🌟 新增至結果字典
            "QA100_Error": err_qa100_msrp  # 🌟 新增至結果字典
        })

# Convert to DataFrames and set Index
df_res_gmvp = pd.DataFrame(results_gmvp).set_index("Model")
df_res_msrp = pd.DataFrame(results_msrp).set_index("Model")

# ==========================================
# 4. Find the Best Overall Winner
# ==========================================
# 此函數會自動偵測 4 個欄位中的最小值
def find_winner(df_result):
    min_val = df_result.min().min()
    best_version = df_result.min().idxmin()
    best_model = df_result[best_version].idxmin()
    return best_model, best_version, min_val

winner_g_model, winner_g_ver, min_err_g = find_winner(df_res_gmvp)
winner_m_model, winner_m_ver, min_err_m = find_winner(df_res_msrp)

# ==========================================
# 5. Define Styling Logic for Dark Mode
# ==========================================
def highlight_row_min(row):
    """
    Highlights the lower error within the same row using Yellow Bold Text.
    Works automatically for 4 columns.
    """
    is_min = row == row.min()
    return ['color: #FFD700; font-weight: bold;' if v else '' for v in is_min]

# Deep Moss Green background for the global best model
champion_style = 'background-color: #1e5631; color: #ffffff;'

# ==========================================
# 6. Output Results
# ==========================================
print("\n" + "="*80)
print(" 🎯 Accuracy Competition: Which model matches [01_Answer_Actual] closest?")
print(" 💡 Reading Guide:")
print("    - Yellow Bold Text: The best performing algorithm (lowest error) within THAT covariance model.")
print("    - Dark Green Background: The global champion across ALL models and algorithms.")
print("="*80)

print(f"\n🏆 [GMVP Overall Winner]: {winner_g_model} ({winner_g_ver})")
print(f"   -> Average Weight Deviation: {min_err_g:.4%}")
display(df_res_gmvp.style.format("{:.4%}")
        .apply(highlight_row_min, axis=1)                  # Row-wise: Highlight lowest error in Yellow
        .highlight_min(axis=None, props=champion_style))   # Global: Highlight best cell in Green

print(f"\n🏆 [MSRP Overall Winner]: {winner_m_model} ({winner_m_ver})")
print(f"   -> Average Weight Deviation: {min_err_m:.4%}")
display(df_res_msrp.style.format("{:.4%}")
        .apply(highlight_row_min, axis=1)                  # Row-wise: Highlight lowest error in Yellow
        .highlight_min(axis=None, props=champion_style))   # Global: Highlight best cell in Green

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

print("⏳ 步驟 1：正在提取測試期間的收盤價並計算每日報酬率...")

start_date = pd.to_datetime(date_dict['Strategy_Start_Date'])
end_date = pd.to_datetime(date_dict['Strategy_End_Date'])

# =====================================================================
# 1. 提取所有權重表中的個股，並計算日報酬率 (共用)
# =====================================================================
# 以 df_qa20_all_portfolios 作為基準取得股票清單
weights_base = df_qa20_all_portfolios.drop(columns=['標的名稱'], errors='ignore').fillna(0)
stock_ids = weights_base.index.astype(str).str.strip().tolist()

returns_list = []
for sid in stock_ids:
    if sid in all_data_dic:
        df = all_data_dic[sid].copy()
        df['date'] = pd.to_datetime(df['date'])
        mask = (df['date'] >= start_date) & (df['date'] <= end_date)
        df_sub = df[mask].sort_values('date').set_index('date')
        
        ret = df_sub['close'].pct_change().fillna(0)
        ret.name = sid
        returns_list.append(ret)

df_stock_returns = pd.concat(returns_list, axis=1).fillna(0)

# =====================================================================
# 2. 提取 0050 大盤的日報酬率 (共用)
# =====================================================================
df_0050 = adj_price_0050_df.copy()
df_0050['date'] = pd.to_datetime(df_0050['date'])
mask_0050 = (df_0050['date'] >= start_date) & (df_0050['date'] <= end_date)
df_0050_sub = df_0050[mask_0050].sort_values('date').set_index('date')

ret_0050 = df_0050_sub['close'].pct_change().fillna(0)
ret_0050.name = 'Benchmark (0050)'

# =====================================================================
# 3. 定義核心績效指標計算函數
# =====================================================================
def calc_metrics(returns):
    cum_ret = (1 + returns).prod() - 1
    daily_std = returns.std()
    ann_std = daily_std * np.sqrt(252)
    sharpe = (returns.mean() / daily_std) * np.sqrt(252) if daily_std != 0 else 0
    
    cum_roll_max = (1 + returns).cumprod().cummax()
    drawdown = ((1 + returns).cumprod() / cum_roll_max) - 1
    mdd = drawdown.min()
    calmar = cum_ret / abs(mdd) if mdd != 0 else np.nan
    
    return pd.Series({
        'Cumulative Return': cum_ret,
        'Daily Std': daily_std,
        'Annualized Std': ann_std,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': mdd,
        'Calmar Ratio': calmar
    })

# 美化格式設定
format_dict = {
    'Cumulative Return': '{:.2%}', 'Daily Std': '{:.2%}', 'Annualized Std': '{:.2%}', 
    'Sharpe Ratio': '{:.2f}', 'Max Drawdown': '{:.2%}', 'Calmar Ratio': '{:.2f}'
}

# =====================================================================
# 4. Plotly 走勢圖繪製函數
# =====================================================================
def plot_cumulative_returns(df_cum, cols_to_plot, title):
    fig = go.Figure()
    for col in cols_to_plot:
        if col == 'Benchmark (0050)':
            line_style = dict(color='crimson', width=3, dash='dashdot')
        elif col == 'Equal Weight':
            line_style = dict(color='gray', width=2, dash='dot')
        elif 'Answer_Actual' in col:
            line_style = dict(color='black', width=3)
        else:
            line_style = dict(width=1.5)
            
        fig.add_trace(go.Scatter(
            x=df_cum.index, y=df_cum[col],
            mode='lines', name=col, line=line_style,
            hovertemplate="<b>%{data.name}</b><br>Date: %{x}<br>Cum Return: %{y:.2%}<extra></extra>"
        ))
    fig.update_layout(
        title=dict(text=f"<b>{title}</b>", font=dict(size=20), x=0.5),
        xaxis_title="Date", yaxis_title="Cumulative Return",
        yaxis=dict(tickformat=".1%"), template="plotly_white",
        width=1300, height=650, hovermode="x unified",
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.8)")
    )
    fig.show()

# =====================================================================
# 5. 核心計算流程：輪流計算 QA20 與 QA100
# =====================================================================
qa_experiments = {
    "QA20": df_qa20_all_portfolios,
    "QA100": df_qa100_all_portfolios
}

# 建立字典存放各自的累積報酬率資料表，供日後或外部呼叫
df_cum_returns_dict = {}

for label, df_raw_weights in qa_experiments.items():
    print(f"\n⏳ 步驟 2：正在計算 {label} 投資組合的績效指標與走勢...")
    
    # 資料對齊與清理
    weights_df = df_raw_weights.drop(columns=['標的名稱'], errors='ignore').fillna(0)
    common_stocks = list(set(df_stock_returns.columns).intersection(set(weights_df.index)))
    common_stocks.sort()
    
    sub_returns = df_stock_returns[common_stocks]
    sub_weights = weights_df.loc[common_stocks]
    
    # 計算模型日報酬率
    df_port_returns = sub_returns.dot(sub_weights)
    
    # 計算等權重日報酬率
    ew_weight = np.ones(len(common_stocks)) / len(common_stocks)
    df_port_returns['Equal Weight'] = sub_returns.dot(ew_weight)
    
    # 合併大盤大盤大盤大盤
    df_all_daily_returns = df_port_returns.join(ret_0050, how='left').fillna(0)
    
    # 計算累積報酬率並存入字典
    df_cum_returns = (1 + df_all_daily_returns).cumprod() - 1
    df_cum_returns_dict[label] = df_cum_returns
    
    # 計算績效指標表
    df_metrics = df_all_daily_returns.apply(calc_metrics).T
    
    # 動態指派給對應的指定變數名稱
    if label == "QA20":
        df_metrics_QA20 = df_metrics.copy()
    else:
        df_metrics_QA100 = df_metrics.copy()
        
    # 展示回測總表
    print("="*90)
    print(f" 📊 {label} (量子退火) 投資組合回測績效總表 ({start_date.date()} - {end_date.date()})")
    print("="*90)
    display(df_metrics.style.format(format_dict).background_gradient(
        subset=['Cumulative Return', 'Sharpe Ratio', 'Calmar Ratio'], cmap='Purples'
    ))
    print("="*90)
    
    # 提取圖表群組
    gmvp_cols = [c for c in sub_weights.columns if 'GMVP' in c]
    msrp_cols = [c for c in sub_weights.columns if 'MSRP' in c]
    baselines = ['Equal Weight', 'Benchmark (0050)']
    
    # 繪製 Plotly 圖表
    plot_cumulative_returns(df_cum_returns, gmvp_cols + baselines, f"1. Out-of-Sample Performance: {label} GMVP Models vs Benchmarks")
    plot_cumulative_returns(df_cum_returns, msrp_cols + baselines, f"2. Out-of-Sample Performance: {label} MSRP Models vs Benchmarks")
    plot_cumulative_returns(df_cum_returns, df_cum_returns.columns, f"3. Out-of-Sample Performance: ALL {label} Portfolios vs Benchmarks")

print("\n🎉 QA20 與 QA100 的所有績效計算、獨立儲存及走勢繪圖已全數完成！")
print("💡 提示：QA20 績效表已存至 `df_metrics_QA20`，QA100 績效表已存至 `df_metrics_QA100`。")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

print("⏳ 正在計算 MVO, SA, QA20 與 QA100 的累積報酬率...")

# =====================================================================
# 1. 計算各版本的每日報酬與累積報酬
# =====================================================================
# 計算 MVO 投組日報酬率
df_ret_mvo = df_stock_returns.dot(df_mvo)
# 計算 SA 投組日報酬率
df_ret_sa = df_stock_returns.dot(df_sa_aligned)
# 🌟 修改點：分別計算 QA20 與 QA100 的投組日報酬率
df_ret_qa20 = df_stock_returns.dot(df_qa20_aligned) 
df_ret_qa100 = df_stock_returns.dot(df_qa100_aligned) 

# 計算等權重 (Equal Weight) 作為 Benchmark
ew_weight = np.ones(len(df_stock_returns.columns)) / len(df_stock_returns.columns)
ret_ew = df_stock_returns.dot(ew_weight)
ret_ew.name = 'Equal Weight'

# 將所有報酬率合併，方便計算累積報酬 (🌟 加入 QA20 與 QA100)
df_all_daily = pd.concat([df_ret_mvo.add_suffix(' (MVO)'), 
                          df_ret_sa.add_suffix(' (SA)'), 
                          df_ret_qa20.add_suffix(' (QA20)'),   # 🌟 導入 QA20
                          df_ret_qa100.add_suffix(' (QA100)'), # 🌟 導入 QA100
                          ret_ew, 
                          ret_0050], axis=1).fillna(0)

# 計算累積報酬率
df_cum_all = (1 + df_all_daily).cumprod() - 1

# =====================================================================
# 2. 定義要繪製的模型清單
# =====================================================================
models = [
    "01_Answer_Actual",
    "02_Sample",
    "03_EWMA",
    "04_LedoitWolf_Identity",
    "05_LedoitWolf_ConstCorr",
    "06_LedoitWolf_Factor"
]

print("📊 開始繪製比較圖表...")

# =====================================================================
# 3. 迴圈繪製比較圖
# =====================================================================
for model in models:
    # 建立 1x2 的子圖 (左邊 GMVP, 右邊 MSRP)
    fig = make_subplots(
        rows=1, cols=2, 
        subplot_titles=(f"<b>GMVP 走勢比較</b>", f"<b>MSRP 走勢比較</b>"),
        horizontal_spacing=0.08
    )
    
    col_gmvp = f"{model}_GMVP"
    col_msrp = f"{model}_MSRP"
    
    # 共同的 Benchmark 軌跡設定函數
    def add_benchmarks(col_num):
        # 加入 0050
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all['Benchmark (0050)'],
            mode='lines', name='0050', line=dict(color='crimson', width=2, dash='dashdot'),
            showlegend=(col_num==1) # 只在左圖顯示圖例，避免重複
        ), row=1, col=col_num)
        # 加入 Equal Weight
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all['Equal Weight'],
            mode='lines', name='等權重 (EW)', line=dict(color='gray', width=2, dash='dot'),
            showlegend=(col_num==1)
        ), row=1, col=col_num)

    # --- 左子圖：繪製 GMVP ---
    if f"{col_gmvp} (MVO)" in df_cum_all.columns:
        # MVO 版本 (藍色實線)
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all[f"{col_gmvp} (MVO)"],
            mode='lines', name='解析解 (MVO)', line=dict(color='SteelBlue', width=2.5),
            legendgroup="MVO", showlegend=True
        ), row=1, col=1)
        
        # SA 版本 (橘色虛線)
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all[f"{col_gmvp} (SA)"],
            mode='lines', name='模擬退火 (SA)', line=dict(color='DarkOrange', width=2.5, dash='dash'),
            legendgroup="SA", showlegend=True
        ), row=1, col=1)
        
        # 🌟 QA20 版本 (淺紫色點線)
        if f"{col_gmvp} (QA20)" in df_cum_all.columns:
            fig.add_trace(go.Scatter(
                x=df_cum_all.index, y=df_cum_all[f"{col_gmvp} (QA20)"],
                mode='lines', name='量子退火 (QA20)', line=dict(color='MediumPurple', width=2.5, dash='dot'),
                legendgroup="QA20", showlegend=True
            ), row=1, col=1)
            
        # 🌟 QA100 版本 (深紫色點劃線)
        if f"{col_gmvp} (QA100)" in df_cum_all.columns:
            fig.add_trace(go.Scatter(
                x=df_cum_all.index, y=df_cum_all[f"{col_gmvp} (QA100)"],
                mode='lines', name='量子退火 (QA100)', line=dict(color='RebeccaPurple', width=2.5, dash='dashdot'),
                legendgroup="QA100", showlegend=True
            ), row=1, col=1)
        
        add_benchmarks(1)

    # --- 右子圖：繪製 MSRP ---
    if f"{col_msrp} (MVO)" in df_cum_all.columns:
        # MVO 版本 (藍色實線)
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all[f"{col_msrp} (MVO)"],
            mode='lines', name='解析解 (MVO)', line=dict(color='SteelBlue', width=2.5),
            legendgroup="MVO", showlegend=False # 避免圖例重複
        ), row=1, col=2)
        
        # SA 版本 (橘色虛線)
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all[f"{col_msrp} (SA)"],
            mode='lines', name='模擬退火 (SA)', line=dict(color='DarkOrange', width=2.5, dash='dash'),
            legendgroup="SA", showlegend=False
        ), row=1, col=2)
        
        # 🌟 QA20 版本 (淺紫色點線)
        if f"{col_msrp} (QA20)" in df_cum_all.columns:
            fig.add_trace(go.Scatter(
                x=df_cum_all.index, y=df_cum_all[f"{col_msrp} (QA20)"],
                mode='lines', name='量子退火 (QA20)', line=dict(color='MediumPurple', width=2.5, dash='dot'),
                legendgroup="QA20", showlegend=False
            ), row=1, col=2)
            
        # 🌟 QA100 版本 (深紫色點劃線)
        if f"{col_msrp} (QA100)" in df_cum_all.columns:
            fig.add_trace(go.Scatter(
                x=df_cum_all.index, y=df_cum_all[f"{col_msrp} (QA100)"],
                mode='lines', name='量子退火 (QA100)', line=dict(color='RebeccaPurple', width=2.5, dash='dashdot'),
                legendgroup="QA100", showlegend=False
            ), row=1, col=2)
        
        add_benchmarks(2)

    # --- 圖表版面設定 ---
    # 修改標題，同時放入 QA20 與 QA100
    fig.update_layout(
        title=dict(text=f"<b>MVO vs SA vs QA20 vs QA100 績效比較：{model}</b>", font=dict(size=22), x=0.5),
        template="plotly_white",
        width=1400, height=600,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
    )
    
    # 將 Y 軸設為百分比格式
    fig.update_yaxes(tickformat=".1%", title_text="累積報酬率", row=1, col=1)
    fig.update_yaxes(tickformat=".1%", row=1, col=2)

    fig.show()

print("🎉 所有比較繪圖已完成！")

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

print("⏳ 正在整合 MVO, SA, QA20, QA100 的核心績效指標...")

# =====================================================================
# 1. 萃取核心指標並建立指定順序的 MultiIndex DataFrame
# =====================================================================
key_metrics = ['Calmar Ratio', 'Cumulative Return', 'Max Drawdown']
algos_order = ['MVO', 'SA', 'QA20', 'QA100'] # 🌟 擴充並嚴格固定順序

# 從各自的 metrics 表中萃取資料
df_m = df_metrics_MVO[key_metrics].copy()
df_s = df_metrics_SA[key_metrics].copy()
df_q20 = df_metrics_QA20[key_metrics].copy()   # 🌟 導入 QA20 績效
df_q100 = df_metrics_QA100[key_metrics].copy() # 🌟 導入 QA100 績效

# 建立多層索引 (MultiIndex) 報表
df_comparison = pd.concat({'MVO': df_m, 'SA': df_s, 'QA20': df_q20, 'QA100': df_q100}, axis=1)

# 交換層級並依指定順序強制重新索引，確保順序固定為 MVO -> SA -> QA20 -> QA100
df_comparison = df_comparison.swaplevel(0, 1, axis=1)
target_columns = pd.MultiIndex.from_product([key_metrics, algos_order], names=['Metric', 'Algo'])
df_comparison = df_comparison.reindex(columns=target_columns)

# ==========================================
# 2. 顯示合併報表 (4演算法精密邊框修正版)
# ==========================================
print("\n" + "="*120)
print(" 🏆 四大演算法績效橫向對比總表 (MVO vs SA vs QA20 vs QA100)")
print(" 💡 閱讀指南：")
print("    - 累積報酬率 (獲利) 最佳者標示為 金黃色")
print("    - 最大回撤 (防禦)   最佳者標示為 亮綠色")
print("    - 卡瑪比率 (綜合)   最佳者標示為 亮藍色")
print("="*120)

# 定義 4 個演算法的數值顯示格式
format_dict = {}
for algo in algos_order:
    format_dict[('Cumulative Return', algo)] = '{:.2%}'
    format_dict[('Max Drawdown', algo)] = '{:.2%}'
    format_dict[('Calmar Ratio', algo)] = '{:.2f}'

# --- 三種不同顏色的高光函數 ---
def highlight_best_return(s):
    """累積報酬最佳：金黃色 (Gold)"""
    is_max = s == s.max()
    return ['color: #FFD700; font-weight: bold;' if v else '' for v in is_max]

def highlight_best_mdd(s):
    """最大回撤最佳 (負最少)：亮綠色 (SpringGreen)"""
    is_max = s == s.max() 
    return ['color: #00FA9A; font-weight: bold;' if v else '' for v in is_max]

def highlight_best_calmar(s):
    """卡瑪比率最佳：青藍色 (Cyan)"""
    is_max = s == s.max()
    return ['color: #00FFFF; font-weight: bold;' if v else '' for v in is_max]

# --- 基礎格式化 ---
styled_df = df_comparison.style.format(format_dict)

# --- 套用多色高光 ---
styled_df = styled_df.apply(highlight_best_return, subset=[('Cumulative Return', a) for a in algos_order], axis=1)
styled_df = styled_df.apply(highlight_best_mdd, subset=[('Max Drawdown', a) for a in algos_order], axis=1)
styled_df = styled_df.apply(highlight_best_calmar, subset=[('Calmar Ratio', a) for a in algos_order], axis=1)

# --- 🌟 4欄位架構專屬的「貫穿型」邊框修復 🌟 ---
# 每個指標有 4 欄。Calmar 佔 0,1,2,3。
# 因此第一道分隔線在第 4 欄 (col4) 左側；第二道分隔線在第 8 欄 (col8) 左側。
custom_table_styles = [
    {'selector': '', 'props': [('border-collapse', 'collapse')]},
    
    # 🛠️ 修正 Calmar Ratio 與 Cumulative Return 之間的分隔線 (第 4 欄開頭)
    {'selector': 'th.col4', 'props': [('border-left', '3px solid #AAAAAA !important')]},
    {'selector': 'td.col4', 'props': [('border-left', '3px solid #AAAAAA !important')]},
    
    # 🛠️ 修正 Cumulative Return 與 Max Drawdown 之間的分隔線 (第 8 欄開頭)
    {'selector': 'th.col8', 'props': [('border-left', '3px solid #AAAAAA !important')]},
    {'selector': 'td.col8', 'props': [('border-left', '3px solid #AAAAAA !important')]}
]

# 套用全表 CSS 樣式並渲染
styled_df = styled_df.set_table_styles(custom_table_styles, overwrite=False)
display(styled_df)

# =====================================================================
# 3. Plotly 視覺化比較 (三張直條圖)
# =====================================================================
print("\n⏳ 正在繪製績效對比直條圖...")

models = df_metrics_MVO.index.tolist()
models_to_plot = [m for m in models if 'Weight' not in m and 'Benchmark' not in m]

fig = make_subplots(
    rows=3, cols=1, 
    subplot_titles=(
        "<b>1. 累積報酬率比較 (Cumulative Return)</b>", 
        "<b>2. 最大回撤比較 (Max Drawdown)</b>",
        "<b>3. 卡瑪比率比較 (Calmar Ratio)</b>"
    ),
    vertical_spacing=0.1
)

# 🌟 更新直條圖顏色配置，加入 QA20 與 QA100
colors = {
    'MVO': 'steelblue', 
    'SA': 'darkorange', 
    'QA20': 'mediumpurple',   # 淺紫
    'QA100': 'rebeccapurple'  # 深紫
}

metrics_config = [
    ('Cumulative Return', 1, ".2%"),
    ('Max Drawdown', 2, ".2%"),
    ('Calmar Ratio', 3, ".2f")
]

for metric, row, tickformat in metrics_config:
    for algo in algos_order: # 🌟 確保每一組直條圖並排的順序與表格完全一致
        color = colors[algo]
        
        # 根據演算法抓取對應的 metrics 資料表
        if algo == 'MVO': df_temp = df_metrics_MVO
        elif algo == 'SA': df_temp = df_metrics_SA
        elif algo == 'QA20': df_temp = df_metrics_QA20
        else: df_temp = df_metrics_QA100
        
        fig.add_trace(go.Bar(
            x=models_to_plot, 
            y=df_temp.loc[models_to_plot, metric],
            name=f'{algo}', 
            marker_color=color,
            legendgroup=algo,
            showlegend=(row == 1), # 避免重複產生多組圖例
            text=df_temp.loc[models_to_plot, metric].apply(lambda x: f"{x:{tickformat}}"),
            textposition='outside'
        ), row=row, col=1)
        
    fig.update_yaxes(tickformat=tickformat, row=row, col=1)

fig.update_layout(
    title=dict(text="<b>MVO vs SA vs QA20 vs QA100: 核心績效視覺化比較</b>", font=dict(size=22), x=0.5),
    template="plotly_white",
    width=1400, height=1200, # 微調高度以容納四根柱子並排
    barmode='group',
    legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="right", x=1)
)

fig.show()
print("🎉 四維度演算法績效橫向對比總表與圖表全部渲染完成！")

In [ ]:
import pandas as pd
from IPython.display import display

print("⏳ 正在統計四大演算法 (MVO, SA, QA20, QA100) 的勝出次數...")

# 設定嚴格的演算法顯示與統計順序
algos_order = ['MVO', 'SA', 'QA20', 'QA100']

# =====================================================================
# 1. 排除基準與等權重列，只專注於預測模型與對照組
# =====================================================================
excluded_rows = ['Equal Weight', 'Benchmark (0050)']
df_filtered_models = df_comparison.drop(index=excluded_rows, errors='ignore')

# =====================================================================
# 2. 自動找出每個指標、每個模型下的贏家
# =====================================================================
# 建立一個空清單來儲存所有贏家
all_winners = []

# 取得所有不重複的指標名稱 ('Calmar Ratio', 'Cumulative Return', 'Max Drawdown')
metrics_list = df_filtered_models.columns.levels[0]

for metric in metrics_list:
    # 提取特定指標下，MVO, SA, QA20, QA100 四個演算法的子資料表
    df_metric_sub = df_filtered_models[metric]
    
    # 找出每一列（每個模型）中數值最大的演算法名稱
    # 因為三個指標在我們對齊後的資料結構中都是越大越好，所以直接用 idxmax
    row_winners = df_metric_sub.idxmax(axis=1)
    
    # 將贏家結果收集起來
    for model, winner in row_winners.items():
        all_winners.append({
            'Metric': metric,
            'Model': model,
            'Winner_Algo': winner
        })

# 轉換為 DataFrame 方便檢視與統計
df_winners_detail = pd.DataFrame(all_winners)

# =====================================================================
# 3. 計算總勝出次數與各指標勝出分布 (鎖定 4 個演算法)
# =====================================================================
# 計算總體勝出次數
total_wins_series = df_winners_detail['Winner_Algo'].value_counts()

# 確保 MVO, SA, QA20, QA100 四個演算法都有出現在排行榜字典中（若勝出 0 次則補 0）
total_wins = {algo: total_wins_series.get(algo, 0) for algo in algos_order}

# 建立交叉表 (指標 x 演算法)，看看各自在什麼領域勝出
df_cross_stats = pd.crosstab(df_winners_detail['Metric'], df_winners_detail['Winner_Algo'])

# 確保交叉表的欄位完整（若完全沒勝出則自動補零欄位）
for algo in algos_order:
    if algo not in df_cross_stats.columns:
        df_cross_stats[algo] = 0

# 強制按照指定的 MVO -> SA -> QA20 -> QA100 順序排列欄位
df_cross_stats = df_cross_stats[algos_order]

# =====================================================================
# 4. 輸出統計報告
# =====================================================================
print("\n" + "="*70)
print(" 🏆 四大演算法勝出次數統計報告 (MVO vs SA vs QA20 vs QA100) ")
print(" 💡 備註：此統計已自動排除 Equal Weight 與 Benchmark (0050)")
print("="*70)

print("\n📊 1. 總勝出次數排行榜 (已依 MVO ➔ SA ➔ QA20 ➔ QA100 順序對齊)：")
for algo in algos_order:
    print(f"   - 【{algo}】 在所有戰場中一共勝出 {total_wins[algo]} 次")

print("\n🔍 2. 各核心指標下的勝出次數分布：")
# 顏色越紅代表在該指標下拿到的冠軍次數越多
display(df_cross_stats.style.background_gradient(cmap='YlOrRd', axis=None))

print("\n📜 3. 詳細各模型贏家明細表：")
display(df_winners_detail.set_index(['Metric', 'Model']))
print("="*70)
print("🎉 四大演算法的所有勝出次數統計已完美整合完畢！")

In [ ]:
import pandas as pd
from IPython.display import display

print("⏳ 正在分拆統計 GMVP 與 MSRP 在四大演算法下的勝出次數...")

# 設定嚴格的演算法顯示與統計順序
algos_order = ['MVO', 'SA', 'QA20', 'QA100']

# =====================================================================
# 1. 排除基準與等權重列，只專注於預測模型與對照組
# =====================================================================
excluded_rows = ['Equal Weight', 'Benchmark (0050)']
df_filtered_models = df_comparison.drop(index=excluded_rows, errors='ignore')

# =====================================================================
# 2. 自動找出每個指標、每個模型下的贏家
# =====================================================================
all_winners = []
metrics_list = df_filtered_models.columns.levels[0]

for metric in metrics_list:
    df_metric_sub = df_filtered_models[metric]
    row_winners = df_metric_sub.idxmax(axis=1)
    
    for model, winner in row_winners.items():
        # 🌟 修改點：動態判定這個模型屬於 GMVP 還是 MSRP 戰場
        portfolio_type = "GMVP" if model.endswith("_GMVP") else "MSRP"
        
        all_winners.append({
            'Portfolio_Type': portfolio_type, # 戰場分類
            'Metric': metric,
            'Model': model,
            'Winner_Algo': winner
        })

# 轉換為總明細 DataFrame
df_winners_detail = pd.DataFrame(all_winners)

# =====================================================================
# 3. 定義一個自動化統計與輸出的輔助函數
# =====================================================================
def print_battlefield_report(p_type, title_icon, table_cmap):
    # 篩選特定戰場 (GMVP 或 MSRP)
    df_sub_battle = df_winners_detail[df_winners_detail['Portfolio_Type'] == p_type]
    
    # 計算該戰場的總體勝出次數
    wins_series = df_sub_battle['Winner_Algo'].value_counts()
    total_wins = {algo: wins_series.get(algo, 0) for algo in algos_order}
    
    # 建立該戰場的交叉表 (指標 x 演算法)
    df_cross = pd.merge(df_sub_battle, pd.crosstab(df_sub_battle['Metric'], df_sub_battle['Winner_Algo']), left_on='Metric', right_index=True)
    df_cross = pd.crosstab(df_sub_battle['Metric'], df_sub_battle['Winner_Algo'])
    
    # 補零防呆與順序排列
    for algo in algos_order:
        if algo not in df_cross.columns:
            df_cross[algo] = 0
    df_cross = df_cross[algos_order]
    
    # 輸出視覺化報告
    print(f"\n{title_icon} 【{p_type} 獨立戰場勝出統計】")
    print("-" * 50)
    print("📊 1. 總勝出次數排行榜：")
    for algo in algos_order:
        print(f"   - 【{algo}】 共勝出 {total_wins[algo]} 次")
        
    print("\n🔍 2. 各指標下的勝出次數分佈：")
    display(df_cross.style.background_gradient(cmap=table_cmap, axis=None))


# =====================================================================
# 4. 輸出最終分拆報告
# =====================================================================
print("\n" + "="*80)
print(" 🏆 四大演算法分拆戰場勝出次數統計報告 (GMVP 獨立戰場 vs MSRP 獨立戰場) ")
print(" 💡 說明：此報告已將純風險優化 (GMVP) 與 夏普比率優化 (MSRP) 的數據完全剝離統計。")
print("="*80)

# 1. 輸出 GMVP 戰場統計 (使用藍綠色系，象徵穩健與風險控制)
print_battlefield_report(p_type="GMVP", title_icon="🛡️", table_cmap="GnBu")

print("\n" + "="*80)

# 2. 輸出 MSRP 戰場統計 (使用橘紅色系，象徵追求超額報酬)
print_battlefield_report(p_type="MSRP", title_icon="🎯", table_cmap="OrRd")

print("\n" + "="*80)
print("📜 3. 完整跨戰場贏家總明細表：")
display(df_winners_detail.set_index(['Portfolio_Type', 'Metric', 'Model']))
print("="*80)
print("🎉 GMVP 與 MSRP 的分拆勝出次數統計已完美執行完畢！")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 設定隨機種子以確保圖形可再現
np.random.seed(42)

# 1. 定義右開拋物線參數 (X: 風險, Y: 預期報酬率)
# X = A * (Y - Y0)**2 + X0
Y0 = 0.06   # GMVP 的預期報酬率 (頂點 Y 軸)
X0 = 0.08   # GMVP 的風險/標準差 (頂點 X 軸，最左端點)
A = 12      # 拋物線曲率參數

# 僅生成大於等於 Y0 的報酬率數據 (即拋物線的上半部)
returns_efficient = np.linspace(Y0, 0.18, 500)
risks_efficient = A * (returns_efficient - Y0)**2 + X0

# GMVP 點座標
gmvp_risk = X0
gmvp_return = Y0

# 無風險利率
rf_rate = 0.02

# 2. 透過數值計算尋找 MSRP (最高夏普比率投資組合) 切點
# 夏普比率公式: (R_p - R_f) / Std_p
sharpe_ratios = (returns_efficient - rf_rate) / risks_efficient
msrp_idx = np.argmax(sharpe_ratios)
msrp_risk = risks_efficient[msrp_idx]
msrp_return = returns_efficient[msrp_idx]

# 3. 建立圖表 (避免使用 .figure() 以符合規範)
fig, ax = plt.subplots(figsize=(10, 6))

# 畫出效率前緣 (僅上半部，使用藍色實線)
ax.plot(risks_efficient, returns_efficient, 'b-', linewidth=2.5, label='Efficient Frontier')

# 畫出資本市場線 (CML) / 切線
cml_x = np.linspace(0, 0.25, 100)
cml_y = rf_rate + ((msrp_return - rf_rate) / msrp_risk) * cml_x
ax.plot(cml_x, cml_y, 'g--', linewidth=1.5, label='Capital Market Line (CML)')

# 標示無風險利率 (Rf)
ax.scatter(0, rf_rate, color='black', zorder=5)
ax.text(0.005, rf_rate + 0.003, f'Rf (Risk-Free Rate = {rf_rate*100:.0f}%)', fontsize=10, fontweight='bold')

# 標示並註解 GMVP (拋物線的最左側端點)
ax.scatter(gmvp_risk, gmvp_return, color='red', s=100, zorder=5, label='GMVP')
ax.annotate(f'GMVP\n(Risk: {gmvp_risk*100:.1f}%, Ret: {gmvp_return*100:.1f}%)',
             xy=(gmvp_risk, gmvp_return),
             xytext=(gmvp_risk + 0.01, gmvp_return - 0.015),
             arrowprops=dict(facecolor='black', shrink=0.08, width=1, headwidth=6))

# 標示並註解 MSRP (CML 與拋物線的切點)
ax.scatter(msrp_risk, msrp_return, color='orange', s=100, zorder=5, label='MSRP')
ax.annotate(f'MSRP (Tangency Portfolio)\n(Risk: {msrp_risk*100:.1f}%, Ret: {msrp_return*100:.1f}%)',
             xy=(msrp_risk, msrp_return),
             xytext=(msrp_risk + 0.01, msrp_return + 0.01),
             arrowprops=dict(facecolor='black', shrink=0.08, width=1, headwidth=6))

# 圖表格式美化 (全英文標籤)
ax.set_title('Portfolio Optimization: GMVP vs MSRP', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Risk / Volatility (Standard Deviation)', fontsize=12)
ax.set_ylabel('Expected Return', fontsize=12)
ax.set_xlim(0, 0.25)
ax.set_ylim(0, 0.20)
ax.grid(True, linestyle=':', alpha=0.6)
ax.legend(loc='upper left', fontsize=10)

# 儲存圖表
plt.savefig('efficient_frontier_parabola.png', dpi=300, bbox_inches='tight')

In [ ]:
# ==========================================
# 0. 輔助函數與資料對齊 (沿用設定)
# ==========================================
def clean_ticker(x):
    s = str(x).strip()
    if s.endswith('.0'): 
        s = s[:-2]
    return s

# ==========================================
# 1. 核心量子退火執行函數 (neal Simulator)
# ==========================================
def optimize_with_qa_simulator(return_col, cov_matrix, model_name, is_actual=False):
    print(f"\n{'='*60}")
    print(f"⚛️ 正在使用量子退火模擬器執行: {model_name}...")
    
    # 複製並清理共變異數矩陣
    cov_df = cov_matrix.copy()
    cov_df.columns = [clean_ticker(c) for c in cov_df.columns]
    cov_df.index = [clean_ticker(i) for i in cov_df.index]
    
    mu_series = df_mu_clean[return_col]
    
    common_stocks = list(set(cov_df.columns).intersection(set(mu_series.index)))
    common_stocks.sort() 
    
    mu = mu_series.loc[common_stocks].values
    cov = cov_df.loc[common_stocks, common_stocks].values
    num_assets = len(common_stocks)
    
    # ---------------------------------------------------------
    # ⚙️ 離散化設定 (Discretization)
    # 將總權重 100% 切分為 20 份 (每份代表 5%)
    # 注意：數字設越大越精準，但二元變數(Qubits)數量會暴增導致運算極慢
    # ---------------------------------------------------------
    precision = 20  
    
    # --- 構建 Constrained Quadratic Model (CQM) ---
    cqm = ConstrainedQuadraticModel()
    
    # 宣告整數變數，每個資產可分配 0 ~ precision 份
    x = {i: Integer(f'x_{i}', lower_bound=0, upper_bound=precision) for i in range(num_assets)}
    
    # 限制式：所有資產分配到的總份數必須等於 precision (即總和 100%)
    cqm.add_constraint(sum(x[i] for i in range(num_assets)) == precision, label='budget')
    
    # 定義二次項：投資組合變異數 (Risk)
    risk_expr = 0
    for i in range(num_assets):
        for j in range(num_assets):
            risk_expr += cov[i, j] * x[i] * x[j]
            
    # 定義一次項：投資組合預期報酬 (Return)
    ret_expr = sum(mu[i] * x[i] for i in range(num_assets))

    # =========================================================
    # 🎛️ 退火模擬器超參數設定區 (Hyperparameters)
    # =========================================================
    qa_hyperparams = {
        "lagrange_multiplier": 5.0, # 懲罰項權重：強迫模型遵守總權重100%的限制式
        "num_reads": 100,           # 退火取樣次數 (重複執行次數，取能量最低者)
        "num_sweeps": 2000,         # 單次退火的降溫步數 (越大越容易找到全域最佳解)
        "seed": 42                  # 固定隨機種子以確保結果可重現
    }

    sampler = neal.SimulatedAnnealingSampler()

    # --- ⏱️ 1. 計算 GMVP (最小化風險) ---
    print(f"   -> 轉換並計算 {model_name} GMVP 中...")
    cqm.set_objective(risk_expr)
    bqm_gmvp, invert_gmvp = cqm_to_bqm(cqm, lagrange_multiplier=qa_hyperparams["lagrange_multiplier"])
    
    start_gmvp = time.perf_counter() # 記錄高精度開始時間
    res_gmvp = sampler.sample(
        bqm_gmvp, 
        num_reads=qa_hyperparams["num_reads"], 
        num_sweeps=qa_hyperparams["num_sweeps"],
        seed=qa_hyperparams["seed"]
    )
    end_gmvp = time.perf_counter()   # 記錄結束時間
    gmvp_time = end_gmvp - start_gmvp
    
    # 從取樣結果中，過濾出符合「總和限制式」且能量最低的最佳解
    best_gmvp = None
    for sample in res_gmvp.samples():
        decoded = invert_gmvp(sample)
        if sum(decoded[f'x_{i}'] for i in range(num_assets)) == precision:
            best_gmvp = decoded
            break
            
    w_gmvp = np.array([best_gmvp[f'x_{i}'] / precision for i in range(num_assets)]) if best_gmvp else np.zeros(num_assets)

    # --- ⏱️ 2. 計算 MSRP (最大化夏普 -> 轉換為風險趨避模型) ---
    # 目標：最小化 (lambda_risk * 風險 - 預期報酬)
    print(f"   -> 轉換並計算 {model_name} MSRP 中...")
    lambda_risk = 0.5 # 風險趨避係數 (可依照需求調整)
    cqm.set_objective(lambda_risk * risk_expr - ret_expr)
    
    bqm_msrp, invert_msrp = cqm_to_bqm(cqm, lagrange_multiplier=qa_hyperparams["lagrange_multiplier"])
    
    start_msrp = time.perf_counter()
    res_msrp = sampler.sample(
        bqm_msrp, 
        num_reads=qa_hyperparams["num_reads"], 
        num_sweeps=qa_hyperparams["num_sweeps"],
        seed=qa_hyperparams["seed"]
    )
    end_msrp = time.perf_counter()
    msrp_time = end_msrp - start_msrp
    
    best_msrp = None
    for sample in res_msrp.samples():
        decoded = invert_msrp(sample)
        if sum(decoded[f'x_{i}'] for i in range(num_assets)) == precision:
            best_msrp = decoded
            break
            
    w_msrp = np.array([best_msrp[f'x_{i}'] / precision for i in range(num_assets)]) if best_msrp else np.zeros(num_assets)

    # 整理權重結果
    df_gmvp = pd.Series(w_gmvp, index=common_stocks, name=f'{model_name}_GMVP')
    df_msrp = pd.Series(w_msrp, index=common_stocks, name=f'{model_name}_MSRP')
    weights_df = pd.concat([df_gmvp, df_msrp], axis=1)
    
    # 建立時間紀錄字典
    time_dict = {
        "Model_Name": model_name,
        "GMVP_Time_Sec": gmvp_time,
        "MSRP_Time_Sec": msrp_time
    }
    
    return weights_df, time_dict

# ==========================================
# 2. 定義測試組合並執行
# ==========================================
configurations = [
    {"name": "01_Answer_Actual", "return_col": "Actual_Return", "cov": cov_matrix_answer, "is_actual": True},
    {"name": "02_Sample",        "return_col": "Predicted_Return", "cov": cov_matrix_sample, "is_actual": False},
    {"name": "03_EWMA",          "return_col": "Predicted_Return", "cov": cov_matrix_ewma,   "is_actual": False},
    {"name": "04_LedoitWolf_Identity", "return_col": "Predicted_Return", "cov": cov_matrix_lw,     "is_actual": False},
    {"name": "05_LedoitWolf_ConstCorr","return_col": "Predicted_Return", "cov": cov_matrix_lw_cc, "is_actual": False},
    {"name": "06_LedoitWolf_Factor",   "return_col": "Predicted_Return", "cov": cov_matrix_lw_sf, "is_actual": False}
]

all_qa_weights = []
qa_time_records = []

for config in configurations:
    w_df, t_dict = optimize_with_qa_simulator(
        return_col=config["return_col"],
        cov_matrix=config["cov"],
        model_name=config["name"],
        is_actual=config["is_actual"]
    )
    all_qa_weights.append(w_df)
    qa_time_records.append(t_dict)

# ==========================================
# 3. 結果統整與展示
# ==========================================
# 3a. 權重總表
df_qa_all_portfolios = pd.concat(all_qa_weights, axis=1)
df_qa_all_portfolios.index.name = 'Stock_ID'

# 過濾並清理
active_mask = (df_qa_all_portfolios > 0).any(axis=1)
df_active_qa_all = df_qa_all_portfolios[active_mask].fillna(0)

print("\n" + "="*80)
print(" 🏆 量子退火模擬器 (QA/neal) 權重總表 (Active Assets Only) ")
print("="*80)
display(df_active_qa_all.style.format("{:.2%}").background_gradient(cmap='Purples', axis=1))

# 3b. ⏱️ 建立耗時統計 DataFrame: df_time_report_qa
df_time_report_qa = pd.DataFrame(qa_time_records)
df_time_report_qa = df_time_report_qa.set_index("Model_Name")
df_time_report_qa['Total_Optimization_Time_Sec'] = df_time_report_qa.sum(axis=1, skipna=True)

print("\n" + "="*80)
print(" ⏱️ 量子退火模擬器 (QA/neal) 求解耗時總表 (秒) ")
print("="*80)
display(df_time_report_qa.style.format("{:.4f}秒").background_gradient(cmap='Reds', axis=0))
print("="*80)
print("✅ 量子退火模擬計算完畢，結果已分別儲存至 `df_qa_all_portfolios` 與 `df_time_report_qa`")

In [ ]:
# ==========================================
# 1. 排除非數值欄位 
# ==========================================
# 篩選出真正要比較的數值欄位 (排除 '標的名稱' 等字串欄位)
cols_to_compare = [c for c in df_all_portfolios_merged.columns if c != '標的名稱']

# ==========================================
# 2. 資料對齊 (Alignment)
# ==========================================
# 確保 MVO 基準對齊並只取數值欄位
df_mvo = df_all_portfolios_merged[cols_to_compare].fillna(0)

# 將 SA 與 QA 的 DataFrame 重新索引，對齊 50 檔股票的 Index。沒有出現的股票補 0
df_sa_aligned = df_active_sa_all.reindex(df_all_portfolios_merged.index).fillna(0)[cols_to_compare]
df_qa_aligned = df_active_qa_all.reindex(df_all_portfolios_merged.index).fillna(0)[cols_to_compare]

# ==========================================
# 3. 計算權重差異 (啟發式權重 - MVO 傳統權重)
# ==========================================
# 分別計算 SA 與 QA 的差異矩陣
df_diff_sa = df_sa_aligned - df_mvo
df_diff_qa = df_qa_aligned - df_mvo

# ==========================================
# 4. 計算誤差統計指標
# ==========================================
# 統整兩者的平均絕對誤差(MAD)與最大誤差(Max Diff)
df_summary = pd.DataFrame({
    'SA 平均絕對誤差 (MAD)': df_diff_sa.abs().mean(),
    'SA 最大絕對誤差 (Max Diff)': df_diff_sa.abs().max(),
    'QA 平均絕對誤差 (MAD)': df_diff_qa.abs().mean(),
    'QA 最大絕對誤差 (Max Diff)': df_diff_qa.abs().max()
})

# ==========================================
# 5. 輸出視覺化結果
# ==========================================
print("\n" + "="*80)
print(" 📊 模型誤差統計表 (衡量 SA / QA 與 MVO 的整體差距)")
print("="*80)
# 顏色越深代表與傳統 MVO 的差異越大
display(df_summary.style.format("{:.2%}").background_gradient(cmap='Reds', axis=0))

# --- 計算 SA 熱力圖極值 ---
max_val_sa = df_diff_sa.abs().max().max()
if pd.isna(max_val_sa) or max_val_sa == 0: max_val_sa = 1.0

print("\n" + "="*80)
print(" 🔍 完整 50 檔個股權重差異表 (SA 權重 - MVO 權重)")
print("="*80)
print("💡 正數(藍): SA 權重較高 | 負數(紅): MVO 權重較高 | 接近0(白): 兩者一致")
display(df_diff_sa.style.format("{:.2%}")
        .background_gradient(cmap='RdBu', vmin=-max_val_sa, vmax=max_val_sa))

# --- 計算 QA 熱力圖極值 ---
max_val_qa = df_diff_qa.abs().max().max()
if pd.isna(max_val_qa) or max_val_qa == 0: max_val_qa = 1.0

print("\n" + "="*80)
print(" 🔍 完整 50 檔個股權重差異表 (QA 權重 - MVO 權重)")
print("="*80)
print("💡 正數(藍): QA 權重較高 | 負數(紅): MVO 權重較高 | 接近0(白): 兩者一致")
display(df_diff_qa.style.format("{:.2%}")
        .background_gradient(cmap='RdBu', vmin=-max_val_qa, vmax=max_val_qa))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

def plot_weight_comparison_mpl(target_model, df_mvo, df_qa):
    """
    Plots a comparison of weights between MVO and QA (Quantum Annealing) algorithms.
    Includes subplots showing the absolute differences.
    """
    print(f"📊 Generating QA weight comparison plot for: {target_model}")
    
    # Define target columns
    col_gmvp = f"{target_model}_GMVP"
    col_msrp = f"{target_model}_MSRP"
    
    if col_gmvp not in df_mvo.columns or col_gmvp not in df_qa.columns:
        print(f"⚠️ Warning: Columns for '{target_model}' not found in the DataFrames.")
        return

    # --- 1. Data Preparation ---
    # Filter GMVP (Keep if either algorithm allocated > 0.1%)
    df_g = pd.DataFrame({'MVO': df_mvo[col_gmvp], 'QA': df_qa[col_gmvp]})
    df_g = df_g[(df_g['MVO'] > 0.001) | (df_g['QA'] > 0.001)].copy()
    df_g['Difference'] = df_g['QA'] - df_g['MVO']
    
    # Filter MSRP (Keep if either algorithm allocated > 0.1%)
    df_m = pd.DataFrame({'MVO': df_mvo[col_msrp], 'QA': df_qa[col_msrp]})
    df_m = df_m[(df_m['MVO'] > 0.001) | (df_m['QA'] > 0.001)].copy()
    df_m['Difference'] = df_m['QA'] - df_m['MVO']

    # --- 2. Create Figure & Subplots (2x2 Grid) ---
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(15, 10))
    fig.suptitle(f"Algorithm Weight Allocation Comparison:\n{target_model} (MVO vs QA)", 
                 fontsize=18, fontweight='bold', y=0.98)

    # Helper function: Plot grouped bar charts (MVO vs QA)
    def plot_grouped_bars(ax, df_data, title):
        x = np.arange(len(df_data))
        width = 0.35
        
        # MVO 使用鋼鐵藍，QA 使用紫色以做區分
        ax.bar(x - width/2, df_data['MVO'], width, label='MVO', color='steelblue')
        ax.bar(x + width/2, df_data['QA'], width, label='QA', color='rebeccapurple')
        
        ax.set_title(title, fontweight='bold', fontsize=14)
        ax.set_ylabel("Allocated Weight")
        ax.set_xticks(x)
        ax.set_xticklabels(df_data.index, rotation=45, ha="right")
        ax.yaxis.set_major_formatter(PercentFormatter(1.0)) # Format Y as %
        ax.legend()
        ax.grid(axis='y', linestyle='--', alpha=0.6)

    # Helper function: Plot difference bar charts (QA - MVO)
    def plot_difference_bars(ax, df_data, title):
        x = np.arange(len(df_data))
        # 綠色代表 QA 權重較高，紅色代表 MVO 權重較高
        colors = ['forestgreen' if val > 0 else 'crimson' for val in df_data['Difference']]
        
        ax.bar(x, df_data['Difference'], color=colors)
        ax.axhline(0, color='black', linewidth=1) # Baseline at 0
        
        ax.set_title(title, fontweight='bold', fontsize=14)
        ax.set_ylabel("Difference (QA - MVO)")
        ax.set_xticks(x)
        ax.set_xticklabels(df_data.index, rotation=45, ha="right")
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.grid(axis='y', linestyle='--', alpha=0.6)

    # --- 3. Render Subplots ---
    # Column 1: GMVP
    plot_grouped_bars(axes[0, 0], df_g, "GMVP: MVO vs QA")
    plot_difference_bars(axes[1, 0], df_g, "GMVP Difference (QA - MVO)")
    
    # Column 2: MSRP
    plot_grouped_bars(axes[0, 1], df_m, "MSRP: MVO vs QA")
    plot_difference_bars(axes[1, 1], df_m, "MSRP Difference (QA - MVO)")

    # Layout adjustment
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# ==========================================
# 執行繪圖測試
# ==========================================
# 呼叫函數，傳入 df_qa_aligned 進行繪圖
plot_weight_comparison_mpl('01_Answer_Actual', df_mvo, df_qa_aligned)

In [ ]:
plot_weight_comparison_mpl('03_EWMA', df_mvo, df_qa_aligned)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

def plot_weight_comparison_mpl(target_model, df_mvo, df_sa, df_qa):
    """
    Plots a comparison of weights between MVO, SA, and QA algorithms.
    Displays GMVP and MSRP allocations side-by-side.
    """
    print(f"📊 Generating combined weight comparison plot for: {target_model}")
    
    # 定義要抓取的欄位名稱
    col_gmvp = f"{target_model}_GMVP"
    col_msrp = f"{target_model}_MSRP"
    
    # 防呆檢查：確認欄位存在於三個資料表中
    for df, name in zip([df_mvo, df_sa, df_qa], ['MVO', 'SA', 'QA']):
        if col_gmvp not in df.columns or col_msrp not in df.columns:
            print(f"⚠️ Warning: Columns for '{target_model}' not found in {name} DataFrame.")
            return

    # --- 1. 資料準備與過濾 ---
    # 過濾 GMVP (只要有任何一個演算法分配超過 0.1% 就保留該股票)
    df_g = pd.DataFrame({'MVO': df_mvo[col_gmvp], 'SA': df_sa[col_gmvp], 'QA': df_qa[col_gmvp]})
    df_g = df_g[(df_g['MVO'] > 0.001) | (df_g['SA'] > 0.001) | (df_g['QA'] > 0.001)].copy()
    
    # 過濾 MSRP (只要有任何一個演算法分配超過 0.1% 就保留該股票)
    df_m = pd.DataFrame({'MVO': df_mvo[col_msrp], 'SA': df_sa[col_msrp], 'QA': df_qa[col_msrp]})
    df_m = df_m[(df_m['MVO'] > 0.001) | (df_m['SA'] > 0.001) | (df_m['QA'] > 0.001)].copy()

    # --- 2. 建立圖表與子圖 (1x2 網格) ---
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 6))
    fig.suptitle(f"Algorithm Weight Allocation Comparison:\n{target_model} (MVO vs SA vs QA)", 
                 fontsize=18, fontweight='bold', y=1.05)

    # 繪製群組長條圖的輔助函數 (三根柱子)
    def plot_triple_bars(ax, df_data, title):
        x = np.arange(len(df_data))
        width = 0.25  # 調整寬度以容納三根柱子
        
        # 依序繪製 MVO(左), SA(中), QA(右)
        ax.bar(x - width, df_data['MVO'], width, label='MVO', color='steelblue')
        ax.bar(x, df_data['SA'], width, label='SA', color='darkorange')
        ax.bar(x + width, df_data['QA'], width, label='QA', color='rebeccapurple')
        
        ax.set_title(title, fontweight='bold', fontsize=14)
        ax.set_ylabel("Allocated Weight")
        ax.set_xticks(x)
        ax.set_xticklabels(df_data.index, rotation=45, ha="right")
        ax.yaxis.set_major_formatter(PercentFormatter(1.0)) # 將 Y 軸格式化為百分比
        ax.legend()
        ax.grid(axis='y', linestyle='--', alpha=0.6)

    # --- 3. 渲染子圖 ---
    # 第一張圖: GMVP 比較
    plot_triple_bars(axes[0], df_g, "GMVP: MVO vs SA vs QA")
    
    # 第二張圖: MSRP 比較
    plot_triple_bars(axes[1], df_m, "MSRP: MVO vs SA vs QA")

    # 版面微調，避免標籤被切掉
    plt.tight_layout()
    plt.show()

# ==========================================
# 執行繪圖測試
# ==========================================
# 呼叫函數，將 MVO, SA, QA 齊齊傳入 (記得確保變數名稱符合你目前的環境)
plot_weight_comparison_mpl('01_Answer_Actual', df_mvo, df_sa_aligned, df_qa_aligned)

In [ ]:
plot_weight_comparison_mpl('03_EWMA', df_mvo, df_sa_aligned, df_qa_aligned)

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# ==========================================
# 1. Define Ground Truth (Answer/Actual)
# ==========================================
# 這是事後看來的「完美解答」
gt_gmvp = df_mvo['01_Answer_Actual_GMVP'].fillna(0)
gt_msrp = df_mvo['01_Answer_Actual_MSRP'].fillna(0)

# ==========================================
# 2. Define Models to Test
# ==========================================
models_to_test = [
    "02_Sample",
    "03_EWMA",
    "04_LedoitWolf_Identity",
    "05_LedoitWolf_ConstCorr",
    "06_LedoitWolf_Factor"
]

results_gmvp = []
results_msrp = []

# ==========================================
# 3. Calculate MAE (Mean Absolute Error)
# ==========================================
for model in models_to_test:
    col_gmvp = f"{model}_GMVP"
    col_msrp = f"{model}_MSRP"
    
    # --- Calculate GMVP Error ---
    if col_gmvp in df_mvo.columns and col_gmvp in df_sa_aligned.columns and col_gmvp in df_qa_aligned.columns:
        mvo_gmvp_weights = df_mvo[col_gmvp].fillna(0)
        sa_gmvp_weights = df_sa_aligned[col_gmvp].fillna(0)
        qa_gmvp_weights = df_qa_aligned[col_gmvp].fillna(0) # 🌟 新增 QA 權重
        
        err_mvo_gmvp = (mvo_gmvp_weights - gt_gmvp).abs().mean()
        err_sa_gmvp = (sa_gmvp_weights - gt_gmvp).abs().mean()
        err_qa_gmvp = (qa_gmvp_weights - gt_gmvp).abs().mean() # 🌟 計算 QA 誤差
        
        results_gmvp.append({
            "Model": model,
            "MVO_Error": err_mvo_gmvp,  
            "SA_Error": err_sa_gmvp,    
            "QA_Error": err_qa_gmvp     # 🌟 新增至結果字典
        })

    # --- Calculate MSRP Error ---
    if col_msrp in df_mvo.columns and col_msrp in df_sa_aligned.columns and col_msrp in df_qa_aligned.columns:
        mvo_msrp_weights = df_mvo[col_msrp].fillna(0)
        sa_msrp_weights = df_sa_aligned[col_msrp].fillna(0)
        qa_msrp_weights = df_qa_aligned[col_msrp].fillna(0) # 🌟 新增 QA 權重
        
        err_mvo_msrp = (mvo_msrp_weights - gt_msrp).abs().mean()
        err_sa_msrp = (sa_msrp_weights - gt_msrp).abs().mean()
        err_qa_msrp = (qa_msrp_weights - gt_msrp).abs().mean() # 🌟 計算 QA 誤差
        
        results_msrp.append({
            "Model": model,
            "MVO_Error": err_mvo_msrp,  
            "SA_Error": err_sa_msrp,    
            "QA_Error": err_qa_msrp     # 🌟 新增至結果字典
        })

# Convert to DataFrames and set Index
df_res_gmvp = pd.DataFrame(results_gmvp).set_index("Model")
df_res_msrp = pd.DataFrame(results_msrp).set_index("Model")

# ==========================================
# 4. Find the Best Overall Winner
# ==========================================
# 這個函數不需要改，它會自動找尋整張表中最小的值
def find_winner(df_result):
    min_val = df_result.min().min()
    best_version = df_result.min().idxmin()
    best_model = df_result[best_version].idxmin()
    return best_model, best_version, min_val

winner_g_model, winner_g_ver, min_err_g = find_winner(df_res_gmvp)
winner_m_model, winner_m_ver, min_err_m = find_winner(df_res_msrp)

# ==========================================
# 5. Define Styling Logic for Dark Mode
# ==========================================
def highlight_row_min(row):
    """
    Highlights the lower error within the same row using Yellow Bold Text.
    Works automatically for 3 columns.
    """
    is_min = row == row.min()
    return ['color: #FFD700; font-weight: bold;' if v else '' for v in is_min]

# Deep Moss Green background for the global best model
champion_style = 'background-color: #1e5631; color: #ffffff;'

# ==========================================
# 6. Output Results
# ==========================================
print("\n" + "="*80)
print(" 🎯 Accuracy Competition: Which model matches [01_Answer_Actual] closest?")
print(" 💡 Reading Guide:")
print("    - Yellow Bold Text: The best performing algorithm (lowest error) within THAT covariance model.")
print("    - Dark Green Background: The global champion across ALL models and algorithms.")
print("="*80)

print(f"\n🏆 [GMVP Overall Winner]: {winner_g_model} ({winner_g_ver})")
print(f"   -> Average Weight Deviation: {min_err_g:.4%}")
display(df_res_gmvp.style.format("{:.4%}")
        .apply(highlight_row_min, axis=1)                  # Row-wise: Highlight lowest error in Yellow
        .highlight_min(axis=None, props=champion_style))   # Global: Highlight best cell in Green

print(f"\n🏆 [MSRP Overall Winner]: {winner_m_model} ({winner_m_ver})")
print(f"   -> Average Weight Deviation: {min_err_m:.4%}")
display(df_res_msrp.style.format("{:.4%}")
        .apply(highlight_row_min, axis=1)                  # Row-wise: Highlight lowest error in Yellow
        .highlight_min(axis=None, props=champion_style))   # Global: Highlight best cell in Green

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

print("⏳ 步驟 1：正在提取測試期間的收盤價並計算每日報酬率...")

start_date = pd.to_datetime(date_dict['Strategy_Start_Date'])
end_date = pd.to_datetime(date_dict['Strategy_End_Date'])

# =====================================================================
# 1. 提取所有權重表中的個股，並計算日報酬率
# =====================================================================
# 🌟 修改點：這裡改為使用 df_qa_aligned (量子退火版的權重)
weights_df = df_qa_aligned.drop(columns=['標的名稱'], errors='ignore').fillna(0)
stock_ids = weights_df.index.astype(str).str.strip().tolist()

returns_list = []
for sid in stock_ids:
    if sid in all_data_dic:
        df = all_data_dic[sid].copy()
        df['date'] = pd.to_datetime(df['date'])
        # 篩選時間區間
        mask = (df['date'] >= start_date) & (df['date'] <= end_date)
        df_sub = df[mask].sort_values('date').set_index('date')
        
        # 計算日報酬率 (若無交易日則填 0)
        ret = df_sub['close'].pct_change().fillna(0)
        ret.name = sid
        returns_list.append(ret)

# 合併所有個股的日報酬率 (若有某天某檔股票沒數據，補 0)
df_stock_returns = pd.concat(returns_list, axis=1).fillna(0)

# =====================================================================
# 2. 提取 0050 大盤的日報酬率
# =====================================================================
df_0050 = adj_price_0050_df.copy()
df_0050['date'] = pd.to_datetime(df_0050['date'])
mask_0050 = (df_0050['date'] >= start_date) & (df_0050['date'] <= end_date)
df_0050_sub = df_0050[mask_0050].sort_values('date').set_index('date')

ret_0050 = df_0050_sub['close'].pct_change().fillna(0)
ret_0050.name = 'Benchmark (0050)'

# =====================================================================
# 3. 計算投資組合 (Portfolios) 的日報酬率與累積報酬率
# =====================================================================
print("⏳ 步驟 2：正在計算 QA (量子退火) 投資組合的績效指標...")

# 確保權重與報酬率矩陣的股票順序完全對齊
common_stocks = list(set(df_stock_returns.columns).intersection(set(weights_df.index)))
df_stock_returns = df_stock_returns[common_stocks]
weights_df = weights_df.loc[common_stocks]

# (A) 模型投組日報酬率 (矩陣內積: 每日報酬率 x 個股權重)
df_port_returns = df_stock_returns.dot(weights_df)

# (B) 等權重投組日報酬率 (Equal Weight)
ew_weight = np.ones(len(common_stocks)) / len(common_stocks)
df_port_returns['Equal Weight'] = df_stock_returns.dot(ew_weight)

# (C) 將 0050 併入比較表
df_all_daily_returns = df_port_returns.join(ret_0050, how='left').fillna(0)

# 計算所有投組的「累積報酬率 (Cumulative Return)」
df_cum_returns = (1 + df_all_daily_returns).cumprod() - 1

# =====================================================================
# 4. 計算核心績效指標 (Metrics)
# =====================================================================
def calc_metrics(returns):
    cum_ret = (1 + returns).prod() - 1
    daily_std = returns.std()
    ann_std = daily_std * np.sqrt(252) # 年化標準差
    
    # Sharpe Ratio (假設無風險利率 = 0)
    sharpe = (returns.mean() / daily_std) * np.sqrt(252) if daily_std != 0 else 0
    
    # Max Drawdown
    cum_roll_max = (1 + returns).cumprod().cummax()
    drawdown = ((1 + returns).cumprod() / cum_roll_max) - 1
    mdd = drawdown.min()
    
    # Calmar Ratio = Cumulative Return / abs(Max Drawdown)
    calmar = cum_ret / abs(mdd) if mdd != 0 else np.nan
    
    return pd.Series({
        'Cumulative Return': cum_ret,
        'Daily Std': daily_std,
        'Annualized Std': ann_std,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': mdd,
        'Calmar Ratio': calmar
    })

# 套用計算並轉置，方便閱讀
df_metrics_QA = df_all_daily_returns.apply(calc_metrics).T

print("\n" + "="*90)
print(f" 📊 QA (量子退火) 投資組合回測績效總表 ({start_date.date()} - {end_date.date()})")
print("="*90)
# 將報表美化，轉為百分比及小數格式
format_dict = {
    'Cumulative Return': '{:.2%}', 'Daily Std': '{:.2%}', 'Annualized Std': '{:.2%}', 
    'Sharpe Ratio': '{:.2f}', 'Max Drawdown': '{:.2%}', 'Calmar Ratio': '{:.2f}'
}
# 🌟 修改點：漸層色換為 Purples
display(df_metrics_QA.style.format(format_dict).background_gradient(subset=['Cumulative Return', 'Sharpe Ratio', 'Calmar Ratio'], cmap='Purples'))
print("="*90)

# =====================================================================
# 5. Plotly 視覺化 (3 張走勢圖)
# =====================================================================
print("⏳ 步驟 3：正在繪製 Plotly 累積報酬走勢圖...")

# 定義群組
gmvp_cols = [c for c in weights_df.columns if 'GMVP' in c]
msrp_cols = [c for c in weights_df.columns if 'MSRP' in c]
baselines = ['Equal Weight', 'Benchmark (0050)']

def plot_cumulative_returns(cols_to_plot, title):
    fig = go.Figure()
    
    for col in cols_to_plot:
        # 特別強調 Answer 正解與 Baseline，其他模型用較細的線條
        if col == 'Benchmark (0050)':
            line_style = dict(color='crimson', width=3, dash='dashdot')
        elif col == 'Equal Weight':
            line_style = dict(color='gray', width=2, dash='dot')
        elif 'Answer_Actual' in col:
            line_style = dict(color='black', width=3)
        else:
            line_style = dict(width=1.5) # 其他預測模型
            
        fig.add_trace(go.Scatter(
            x=df_cum_returns.index, 
            y=df_cum_returns[col],
            mode='lines', 
            name=col,
            line=line_style,
            hovertemplate="<b>%{data.name}</b><br>Date: %{x}<br>Cum Return: %{y:.2%}<extra></extra>"
        ))

    fig.update_layout(
        title=dict(text=f"<b>{title}</b>", font=dict(size=20), x=0.5),
        xaxis_title="Date",
        yaxis_title="Cumulative Return",
        yaxis=dict(tickformat=".1%"),
        template="plotly_white",
        width=1300, 
        height=650,
        hovermode="x unified",
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.8)")
    )
    fig.show()

# 繪製三張圖 (🌟 修改點：標題加上 QA)
plot_cumulative_returns(gmvp_cols + baselines, "1. Out-of-Sample Performance: QA GMVP Models vs Benchmarks")
plot_cumulative_returns(msrp_cols + baselines, "2. Out-of-Sample Performance: QA MSRP Models vs Benchmarks")
plot_cumulative_returns(df_cum_returns.columns, "3. Out-of-Sample Performance: ALL QA Portfolios vs Benchmarks")

print("🎉 所有績效計算與繪圖已完成！")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

print("⏳ 正在計算 MVO, SA 與 QA 的累積報酬率...")

# =====================================================================
# 1. 計算三方版本的每日報酬與累積報酬
# =====================================================================
# 確保我們使用的是先前的 df_stock_returns, df_mvo, df_sa_aligned, df_qa_aligned
# 計算 MVO 投組日報酬率
df_ret_mvo = df_stock_returns.dot(df_mvo)
# 計算 SA 投組日報酬率
df_ret_sa = df_stock_returns.dot(df_sa_aligned)
# 🌟 計算 QA 投組日報酬率
df_ret_qa = df_stock_returns.dot(df_qa_aligned) 

# 計算等權重 (Equal Weight) 作為 Benchmark
ew_weight = np.ones(len(df_stock_returns.columns)) / len(df_stock_returns.columns)
ret_ew = df_stock_returns.dot(ew_weight)
ret_ew.name = 'Equal Weight'

# 將所有報酬率合併，方便計算累積報酬 (🌟 加入 QA)
df_all_daily = pd.concat([df_ret_mvo.add_suffix(' (MVO)'), 
                          df_ret_sa.add_suffix(' (SA)'), 
                          df_ret_qa.add_suffix(' (QA)'), 
                          ret_ew, 
                          ret_0050], axis=1).fillna(0)

# 計算累積報酬率
df_cum_all = (1 + df_all_daily).cumprod() - 1

# =====================================================================
# 2. 定義要繪製的模型清單
# =====================================================================
models = [
    "01_Answer_Actual",
    "02_Sample",
    "03_EWMA",
    "04_LedoitWolf_Identity",
    "05_LedoitWolf_ConstCorr",
    "06_LedoitWolf_Factor"
]

print("📊 開始繪製比較圖表...")

# =====================================================================
# 3. 迴圈繪製比較圖
# =====================================================================
for model in models:
    # 建立 1x2 的子圖 (左邊 GMVP, 右邊 MSRP)
    fig = make_subplots(
        rows=1, cols=2, 
        subplot_titles=(f"<b>GMVP 走勢比較</b>", f"<b>MSRP 走勢比較</b>"),
        horizontal_spacing=0.08
    )
    
    col_gmvp = f"{model}_GMVP"
    col_msrp = f"{model}_MSRP"
    
    # 共同的 Benchmark 軌跡設定函數
    def add_benchmarks(col_num):
        # 加入 0050
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all['Benchmark (0050)'],
            mode='lines', name='0050', line=dict(color='crimson', width=2, dash='dashdot'),
            showlegend=(col_num==1) # 只在左圖顯示圖例，避免重複
        ), row=1, col=col_num)
        # 加入 Equal Weight
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all['Equal Weight'],
            mode='lines', name='等權重 (EW)', line=dict(color='gray', width=2, dash='dot'),
            showlegend=(col_num==1)
        ), row=1, col=col_num)

    # --- 左子圖：繪製 GMVP ---
    if f"{col_gmvp} (MVO)" in df_cum_all.columns:
        # MVO 版本 (藍色實線)
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all[f"{col_gmvp} (MVO)"],
            mode='lines', name='解析解 (MVO)', line=dict(color='SteelBlue', width=2.5),
            legendgroup="MVO", showlegend=True
        ), row=1, col=1)
        
        # SA 版本 (橘色虛線，加上虛線更容易與實線重疊時辨識)
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all[f"{col_gmvp} (SA)"],
            mode='lines', name='模擬退火 (SA)', line=dict(color='DarkOrange', width=2.5, dash='dash'),
            legendgroup="SA", showlegend=True
        ), row=1, col=1)
        
        # 🌟 QA 版本 (紫色點線)
        if f"{col_gmvp} (QA)" in df_cum_all.columns:
            fig.add_trace(go.Scatter(
                x=df_cum_all.index, y=df_cum_all[f"{col_gmvp} (QA)"],
                mode='lines', name='量子退火 (QA)', line=dict(color='RebeccaPurple', width=2.5, dash='dot'),
                legendgroup="QA", showlegend=True
            ), row=1, col=1)
        
        add_benchmarks(1)

    # --- 右子圖：繪製 MSRP ---
    if f"{col_msrp} (MVO)" in df_cum_all.columns:
        # MVO 版本 (藍色實線)
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all[f"{col_msrp} (MVO)"],
            mode='lines', name='解析解 (MVO)', line=dict(color='SteelBlue', width=2.5),
            legendgroup="MVO", showlegend=False # 避免圖例重複
        ), row=1, col=2)
        
        # SA 版本 (橘色虛線)
        fig.add_trace(go.Scatter(
            x=df_cum_all.index, y=df_cum_all[f"{col_msrp} (SA)"],
            mode='lines', name='模擬退火 (SA)', line=dict(color='DarkOrange', width=2.5, dash='dash'),
            legendgroup="SA", showlegend=False
        ), row=1, col=2)
        
        # 🌟 QA 版本 (紫色點線)
        if f"{col_msrp} (QA)" in df_cum_all.columns:
            fig.add_trace(go.Scatter(
                x=df_cum_all.index, y=df_cum_all[f"{col_msrp} (QA)"],
                mode='lines', name='量子退火 (QA)', line=dict(color='RebeccaPurple', width=2.5, dash='dot'),
                legendgroup="QA", showlegend=False
            ), row=1, col=2)
        
        add_benchmarks(2)

    # --- 圖表版面設定 ---
    # 🌟 修改標題，加入 QA
    fig.update_layout(
        title=dict(text=f"<b>MVO vs SA vs QA 績效比較：{model}</b>", font=dict(size=22), x=0.5),
        template="plotly_white",
        width=1400, height=600,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
    )
    
    # 將 Y 軸設為百分比格式
    fig.update_yaxes(tickformat=".1%", title_text="累積報酬率", row=1, col=1)
    fig.update_yaxes(tickformat=".1%", row=1, col=2)

    fig.show()

print("🎉 所有比較繪圖已完成！")

In [ ]:
df_metrics_MVO

In [ ]:
df_metrics_SA

In [ ]:
df_metrics_QA

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

print("⏳ 正在整合 MVO, SA, QA 的核心績效指標...")

# =====================================================================
# 1. 萃取核心指標並建立 MultiIndex DataFrame
# =====================================================================
key_metrics = ['Cumulative Return', 'Max Drawdown', 'Calmar Ratio']

# 從各自的 metrics 表中萃取資料
df_m = df_metrics_MVO[key_metrics].copy()
df_s = df_metrics_SA[key_metrics].copy()
df_q = df_metrics_QA[key_metrics].copy()

# 建立多層索引 (MultiIndex) 報表
df_comparison = pd.concat({'MVO': df_m, 'SA': df_s, 'QA': df_q}, axis=1)

# 交換層級並排序，讓同一個指標的三個演算法排在一起比較
df_comparison = df_comparison.swaplevel(0, 1, axis=1).sort_index(axis=1)
# ==========================================
# 2. 顯示合併報表 (修復分隔線斷裂版)
# ==========================================
print("\n" + "="*110)
print(" 🏆 三大演算法績效橫向對比總表 (MVO vs SA vs QA)")
print(" 💡 閱讀指南：")
print("    - 累積報酬率 (獲利) 最佳者標示為 金黃色")
print("    - 最大回撤 (防禦)   最佳者標示為 亮綠色")
print("    - 卡瑪比率 (綜合)   最佳者標示為 亮藍色")
print("="*110)

# 定義數值顯示格式
format_dict = {}
for algo in ['MVO', 'SA', 'QA']:
    format_dict[('Cumulative Return', algo)] = '{:.2%}'
    format_dict[('Max Drawdown', algo)] = '{:.2%}'
    format_dict[('Calmar Ratio', algo)] = '{:.2f}'

# --- 三種不同顏色的高光函數 ---
def highlight_best_return(s):
    """累積報酬最佳：金黃色 (Gold)"""
    is_max = s == s.max()
    return ['color: #FFD700; font-weight: bold;' if v else '' for v in is_max]

def highlight_best_mdd(s):
    """最大回撤最佳 (負最少)：亮綠色 (SpringGreen)"""
    is_max = s == s.max() 
    return ['color: #00FA9A; font-weight: bold;' if v else '' for v in is_max]

def highlight_best_calmar(s):
    """卡瑪比率最佳：青藍色 (Cyan)"""
    is_max = s == s.max()
    return ['color: #00FFFF; font-weight: bold;' if v else '' for v in is_max]

# --- 基礎格式化 ---
styled_df = df_comparison.style.format(format_dict)

# --- 套用多色高光 ---
styled_df = styled_df.apply(highlight_best_return, subset=[('Cumulative Return', a) for a in ['MVO', 'SA', 'QA']], axis=1)
styled_df = styled_df.apply(highlight_best_mdd, subset=[('Max Drawdown', a) for a in ['MVO', 'SA', 'QA']], axis=1)
styled_df = styled_df.apply(highlight_best_calmar, subset=[('Calmar Ratio', a) for a in ['MVO', 'SA', 'QA']], axis=1)

# --- 🌟 全新設計的「貫穿型」邊框修復技術 🌟 ---
# 為了避免斷裂，我們不單獨給 td 加屬性，而是改用「全表 CSS 選擇器」統一定義
# 這樣能保證最上面的大標題、小標題、以及下方的每一列數據，線條寬度與顏色完全同步！

custom_table_styles = [
    # 1. 讓整張表的邊框模型融合，確保線條不會被儲存格間隙切斷
    {'selector': '', 'props': [('border-collapse', 'collapse')]},
    
    # 2. 🛠️ 修正 Calmar Ratio 與 Cumulative Return 之間的分隔線 (第 3 欄，從 0 開始算)
    {'selector': 'th.col3', 'props': [('border-left', '3px solid #AAAAAA !important')]},
    {'selector': 'td.col3', 'props': [('border-left', '3px solid #AAAAAA !important')]},
    
    # 3. 🛠️ 修正 Cumulative Return 與 Max Drawdown 之間的分隔線 (第 6 欄)
    {'selector': 'th.col6', 'props': [('border-left', '3px solid #AAAAAA !important')]},
    {'selector': 'td.col6', 'props': [('border-left', '3px solid #AAAAAA !important')]}
]

# 套用全新設計的 CSS 樣式
styled_df = styled_df.set_table_styles(custom_table_styles, overwrite=False)

# 顯示最終完美的 DataFrame
display(styled_df)

# =====================================================================
# 3. Plotly 視覺化比較 (三張直條圖)
# =====================================================================
print("\n⏳ 正在繪製績效對比直條圖...")

models = df_metrics_MVO.index.tolist()
models_to_plot = [m for m in models if 'Weight' not in m and 'Benchmark' not in m]

fig = make_subplots(
    rows=3, cols=1, 
    subplot_titles=(
        "<b>1. 累積報酬率比較 (Cumulative Return)</b>", 
        "<b>2. 最大回撤比較 (Max Drawdown)</b>",
        "<b>3. 卡瑪比率比較 (Calmar Ratio)</b>"
    ),
    vertical_spacing=0.1
)

colors = {'MVO': 'steelblue', 'SA': 'darkorange', 'QA': 'rebeccapurple'}

metrics_config = [
    ('Cumulative Return', 1, ".2%"),
    ('Max Drawdown', 2, ".2%"),
    ('Calmar Ratio', 3, ".2f")
]

for metric, row, tickformat in metrics_config:
    for algo, color in colors.items():
        if algo == 'MVO': df_temp = df_metrics_MVO
        elif algo == 'SA': df_temp = df_metrics_SA
        else: df_temp = df_metrics_QA
        
        fig.add_trace(go.Bar(
            x=models_to_plot, 
            y=df_temp.loc[models_to_plot, metric],
            name=f'{algo}', 
            marker_color=color,
            legendgroup=algo,
            showlegend=(row == 1), 
            text=df_temp.loc[models_to_plot, metric].apply(lambda x: f"{x:{tickformat}}"),
            textposition='outside'
        ), row=row, col=1)
        
    fig.update_yaxes(tickformat=tickformat, row=row, col=1)

fig.update_layout(
    title=dict(text="<b>MVO vs SA vs QA: 核心績效視覺化比較</b>", font=dict(size=22), x=0.5),
    template="plotly_white",
    width=1400, height=1100, 
    barmode='group',
    legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="right", x=1)
)

fig.show()

In [ ]:
import pandas as pd
from IPython.display import display

print("⏳ 正在統計三大演算法的勝出次數...")

# =====================================================================
# 1. 排除基準與等權重列，只專注於預測模型與對照組
# =====================================================================
excluded_rows = ['Equal Weight', 'Benchmark (0050)']
df_filtered_models = df_comparison.drop(index=excluded_rows, errors='ignore')

# =====================================================================
# 2. 自動找出每個指標、每個模型下的贏家
# =====================================================================
# 建立一個空清單來儲存所有贏家
all_winners = []

# 我們原本的欄位是 MultiIndex: (指標名稱, 演算法名稱)
# 取得所有不重複的指標名稱 ('Calmar Ratio', 'Cumulative Return', 'Max Drawdown')
metrics_list = df_filtered_models.columns.levels[0]

for metric in metrics_list:
    # 提取特定指標下，MVO, QA, SA 三個演算法的子資料表
    df_metric_sub = df_filtered_models[metric]
    
    # 找出每一列（每個模型）中數值最大的演算法名稱
    # 因為三個指標都是越大越好，所以直接用 idxmax
    row_winners = df_metric_sub.idxmax(axis=1)
    
    # 將贏家結果收集起來
    for model, winner in row_winners.items():
        all_winners.append({
            'Metric': metric,
            'Model': model,
            'Winner_Algo': winner
        })

# 轉換為 DataFrame 方便檢視與統計
df_winners_detail = pd.DataFrame(all_winners)

# =====================================================================
# 3. 計算總勝出次數與各指標勝出分布
# =====================================================================
# 計算總體勝出次數
total_wins = df_winners_detail['Winner_Algo'].value_counts()

# 確保 MVO, SA, QA 三個演算法都有出現在統計中（若勝出 0 次則補 0）
for algo in ['MVO', 'SA', 'QA']:
    if algo not in total_wins:
        total_wins[algo] = 0

# 建立交叉表 (指標 x 演算法)，看看各自在什麼領域勝出
df_cross_stats = pd.crosstab(df_winners_detail['Metric'], df_winners_detail['Winner_Algo'])

# 確保交叉表的欄位完整
for algo in ['MVO', 'SA', 'QA']:
    if algo not in df_cross_stats.columns:
        df_cross_stats[algo] = 0
df_cross_stats = df_cross_stats[['MVO', 'SA', 'QA']]

# =====================================================================
# 4. 輸出統計報告
# =====================================================================
print("\n" + "="*60)
print(" 🏆 三大演算法勝出次數統計報告 (不含 Benchmark) ")
print("="*60)

print("\n📊 1. 總勝出次數排行榜：")
for algo, wins in total_wins.items():
    print(f"   - 【{algo}】 共勝出 {wins} 次")

print("\n🔍 2. 各核心指標下的勝出次數分布：")
display(df_cross_stats.style.background_gradient(cmap='YlOrRd', axis=None))

print("\n📜 3. 詳細各模型贏家明細表：")
display(df_winners_detail.set_index(['Metric', 'Model']))
print("="*60)

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

print("⏳ 正在整合 MVO, SA, QA 的核心績效指標與勝出統計...")

# =====================================================================
# 1. 萃取核心指標並建立指定順序的 MultiIndex DataFrame
# =====================================================================
key_metrics = ['Calmar Ratio', 'Cumulative Return', 'Max Drawdown']
algos_order = ['MVO', 'SA', 'QA'] # 🌟 嚴格定義演算法順序

# 從各自的 metrics 表中萃取資料
df_m = df_metrics_MVO[key_metrics].copy()
df_s = df_metrics_SA[key_metrics].copy()
df_q = df_metrics_QA[key_metrics].copy()

# 建立多層索引 (MultiIndex) 報表
df_comparison = pd.concat({'MVO': df_m, 'SA': df_s, 'QA': df_q}, axis=1)

# 交換層級，使結構變為 (指標名稱, 演算法名稱)
df_comparison = df_comparison.swaplevel(0, 1, axis=1)

# 🌟 重大修改：強制指定外層(指標)與內層(演算法)的排列順序，確保完全符合 MVO -> SA -> QA
target_columns = pd.MultiIndex.from_product([key_metrics, algos_order], names=['Metric', 'Algo'])
df_comparison = df_comparison.reindex(columns=target_columns)

# =====================================================================
# 2. 顯示合併報表 (修復分隔線貫穿版)
# =====================================================================
print("\n" + "="*110)
print(" 🏆 三大演算法績效橫向對比總表 (MVO vs SA vs QA)")
print(" 💡 閱讀指南：")
print("    - 累積報酬率 (獲利) 最佳者標示為 金黃色")
print("    - 最大回撤 (防禦)   最佳者標示為 亮綠色")
print("    - 卡瑪比率 (綜合)   最佳者標示為 亮藍色")
print("="*110)

# 定義數值顯示格式
format_dict = {}
for algo in algos_order:
    format_dict[('Cumulative Return', algo)] = '{:.2%}'
    format_dict[('Max Drawdown', algo)] = '{:.2%}'
    format_dict[('Calmar Ratio', algo)] = '{:.2f}'

# --- 三種不同顏色的高光函數 ---
def highlight_best_return(s):
    """累積報酬最佳：金黃色 (Gold)"""
    is_max = s == s.max()
    return ['color: #FFD700; font-weight: bold;' if v else '' for v in is_max]

def highlight_best_mdd(s):
    """最大回撤最佳 (負最少)：亮綠色 (SpringGreen)"""
    is_max = s == s.max() 
    return ['color: #00FA9A; font-weight: bold;' if v else '' for v in is_max]

def highlight_best_calmar(s):
    """卡瑪比率最佳：青藍色 (Cyan)"""
    is_max = s == s.max()
    return ['color: #00FFFF; font-weight: bold;' if v else '' for v in is_max]

# --- 基礎格式化 ---
styled_df = df_comparison.style.format(format_dict)

# --- 套用多色高光 ---
styled_df = styled_df.apply(highlight_best_return, subset=[('Cumulative Return', a) for a in algos_order], axis=1)
styled_df = styled_df.apply(highlight_best_mdd, subset=[('Max Drawdown', a) for a in algos_order], axis=1)
styled_df = styled_df.apply(highlight_best_calmar, subset=[('Calmar Ratio', a) for a in algos_order], axis=1)

# --- 🌟 貫穿型邊框修復 (依據新順序重算 Column index) 🌟 ---
# 欄位排序為：Calmar(M,S,Q), Cumulative(M,S,Q), Max Drawdown(M,S,Q)
# 因此第 4 欄 (index 3) 是 Cumulative_MVO；第 7 欄 (index 6) 是 MaxDrawdown_MVO
custom_table_styles = [
    {'selector': '', 'props': [('border-collapse', 'collapse')]},
    
    # 🛠️ 修正 Calmar Ratio 與 Cumulative Return 之間的分隔線 (第 3 欄)
    {'selector': 'th.col3', 'props': [('border-left', '3px solid #AAAAAA !important')]},
    {'selector': 'td.col3', 'props': [('border-left', '3px solid #AAAAAA !important')]},
    
    # 🛠️ 修正 Cumulative Return 與 Max Drawdown 之間的分隔線 (第 6 欄)
    {'selector': 'th.col6', 'props': [('border-left', '3px solid #AAAAAA !important')]},
    {'selector': 'td.col6', 'props': [('border-left', '3px solid #AAAAAA !important')]}
]

styled_df = styled_df.set_table_styles(custom_table_styles, overwrite=False)
display(styled_df)

# =====================================================================
# 3. Plotly 視覺化比較 (三張直條圖)
# =====================================================================
print("\n⏳ 正在繪製績效對比直條圖...")

models = df_metrics_MVO.index.tolist()
models_to_plot = [m for m in models if 'Weight' not in m and 'Benchmark' not in m]

fig = make_subplots(
    rows=3, cols=1, 
    subplot_titles=(
        "<b>1. 累積報酬率比較 (Cumulative Return)</b>", 
        "<b>2. 最大回撤比較 (Max Drawdown)</b>",
        "<b>3. 卡瑪比率比較 (Calmar Ratio)</b>"
    ),
    vertical_spacing=0.1
)

# 🌟 圖例與直條圖色彩配置，順序調整為 MVO, SA, QA
colors = {'MVO': 'steelblue', 'SA': 'darkorange', 'QA': 'rebeccapurple'}

metrics_config = [
    ('Cumulative Return', 1, ".2%"),
    ('Max Drawdown', 2, ".2%"),
    ('Calmar Ratio', 3, ".2f")
]

for metric, row, tickformat in metrics_config:
    for algo in algos_order: # 🌟 確保直條圖並排順序也是 MVO -> SA -> QA
        color = colors[algo]
        if algo == 'MVO': df_temp = df_metrics_MVO
        elif algo == 'SA': df_temp = df_metrics_SA
        else: df_temp = df_metrics_QA
        
        fig.add_trace(go.Bar(
            x=models_to_plot, 
            y=df_temp.loc[models_to_plot, metric],
            name=f'{algo}', 
            marker_color=color,
            legendgroup=algo,
            showlegend=(row == 1), 
            text=df_temp.loc[models_to_plot, metric].apply(lambda x: f"{x:{tickformat}}"),
            textposition='outside'
        ), row=row, col=1)
        
    fig.update_yaxes(tickformat=tickformat, row=row, col=1)

fig.update_layout(
    title=dict(text="<b>MVO vs SA vs QA: 核心績效視覺化比較</b>", font=dict(size=22), x=0.5),
    template="plotly_white",
    width=1400, height=1100, 
    barmode='group',
    legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="right", x=1)
)
fig.show()

# =====================================================================
# 4. 統計三大演算法的勝出次數
# =====================================================================
print("\n⏳ 正在統計三大演算法的勝出次數...")

excluded_rows = ['Equal Weight', 'Benchmark (0050)']
df_filtered_models = df_comparison.drop(index=excluded_rows, errors='ignore')

all_winners = []
metrics_list = df_filtered_models.columns.levels[0]

for metric in metrics_list:
    df_metric_sub = df_filtered_models[metric]
    # 由於三個指標均是數值越大越優，直接使用 idxmax 找出贏家
    row_winners = df_metric_sub.idxmax(axis=1)
    
    for model, winner in row_winners.items():
        all_winners.append({
            'Metric': metric,
            'Model': model,
            'Winner_Algo': winner
        })

df_winners_detail = pd.DataFrame(all_winners)

# 計算總體勝出次數，並依 MVO -> SA -> QA 排序輸出
total_wins_series = df_winners_detail['Winner_Algo'].value_counts()
total_wins = {algo: total_wins_series.get(algo, 0) for algo in algos_order}

# 建立交叉表，並強制指定欄位順序為 MVO, SA, QA
df_cross_stats = pd.crosstab(df_winners_detail['Metric'], df_winners_detail['Winner_Algo'])
for algo in algos_order:
    if algo not in df_cross_stats.columns:
        df_cross_stats[algo] = 0
df_cross_stats = df_cross_stats[algos_order]

# =====================================================================
# 5. 輸出統計報告
# =====================================================================
print("\n" + "="*60)
print(" 🏆 三大演算法勝出次數統計報告 (不含 Benchmark) ")
print("="*60)

print("\n📊 1. 總勝出次數排行榜 (已依 MVO ➔ SA ➔ QA 排序)：")
for algo in algos_order:
    print(f"   - 【{algo}】 共勝出 {total_wins[algo]} 次")

print("\n🔍 2. 各核心指標下的勝出次數分布：")
display(df_cross_stats.style.background_gradient(cmap='YlOrRd', axis=None))

print("\n📜 3. 詳細各模型贏家明細表：")
# 調整明細表呈現，使其順序一目了然
display(df_winners_detail.set_index(['Metric', 'Model']))
print("="*60)
print("🎉 演算法綜合比對全流程執行完畢！")

# 5. Using IBM Quiskit to Solve QUBO Portfolio Optimization